In [ ]:
# Ejecuta esto en una celda
!/home/quant/venv/bin/pip install pyarrow fastparquet

## 🛠️ Procesamiento de Datos: Contrato Continuo NQ (1m)

Esta fase del código transforma los datos crudos (raw data) de futuros en un **Dataset Continuo** listo para trading algorítmico, resolviendo los problemas de zonas horarias y vencimientos de contratos.

### Descripción de la Funcionalidad
La función realiza un proceso de **ETL** (Extract, Transform, Load) especializado en futuros:

1.  **Normalización Temporal:** Convierte los timestamps de formato UTC (Zulu) a la hora local del mercado (**America/New_York**), asegurando que el Open de las 09:30 AM coincida siempre con la realidad del parqué.
2.  **Lógica de Rollover Dinámico:** Identifica todos los contratos disponibles (ej. NQZ5, NQH6) y selecciona para cada día aquel con el **mayor volumen transaccionado**. Esto garantiza que siempre operes el contrato "Front Month" (el más líquido).
3.  **Filtro Anti-Duplicidad:** Elimina las filas de contratos secundarios que aparecen durante las semanas de transición, evitando que el cálculo del ORB se ensucie con precios de contratos sin liquidez.
4.  **Limpieza Numérica:** Fuerza la conversión de precios a flotantes y elimina registros corruptos o incompletos.

---

### 📦 Estructura del Archivo de Salida (Parquet)

Al finalizar, el DataFrame se exporta comúnmente a formato `.parquet`. A diferencia del CSV, el Parquet conserva los tipos de datos y está optimizado para lecturas rápidas en Python.

**Especificaciones del archivo:**

| Columna | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| **timestamp** (Index) | datetime64[ns] | Fecha y hora local de NY (sin zona horaria). |
| **Open** | float64 | Precio de apertura de la vela de 1 minuto. |
| **High** | float64 | Precio máximo alcanzado en el minuto. |
| **Low** | float64 | Precio mínimo alcanzado en el minuto. |
| **Close** | float64 | Precio de cierre de la vela de 1 minuto. |
| **Volume** | float64 | Cantidad de contratos operados en ese minuto. |
| **Ticker** | object (string) | Identificador del contrato activo ese día (ej. `NQZ25`). |

> **Nota:** El uso de Parquet reduce el tamaño del archivo en disco en un ~70% comparado con el CSV original.


In [1]:
import pandas as pd

def load_continuous_nq_data_v2(path):
    print("🔍 Iniciando carga y limpieza de contratos múltiples...")
    
    # 1. Carga bruta del archivo
    df = pd.read_csv(
        path,
        header=None,
        skiprows=1,
        names=['Time', 'Col1', 'Col2', 'Col3', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker'],
        usecols=['Time', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker']
    )

    # 2. Conversión horaria inmediata (UTC -> NY)
    df['Time'] = pd.to_datetime(df['Time'], format='%Y-%m-%dT%H:%M:%S.%fZ')
    df.set_index('Time', inplace=True)
    df.index.name = 'timestamp' # Nombramos el índice explícitamente para evitar el KeyError
    df.index = df.index.tz_localize('UTC').tz_convert('America/New_York')
    df.index = df.index.tz_localize(None)
    
    # 3. IDENTIFICAR CONTRATO LÍDER POR DÍA
    df['Date'] = df.index.date
    vols = df.groupby(['Date', 'Ticker'])['Volume'].sum().reset_index()
    idx_max_vol = vols.groupby('Date')['Volume'].idxmax()
    front_contracts = vols.loc[idx_max_vol, ['Date', 'Ticker']]
    
    print(f"📈 Rollover: {len(front_contracts['Ticker'].unique())} contratos identificados.")

    # 4. FILTRADO ESTRICTO (Merge corregido)
    # Al hacer reset_index, la columna se llamará 'timestamp'
    df_reset = df.reset_index()
    df_clean = pd.merge(
        df_reset, 
        front_contracts, 
        on=['Date', 'Ticker'], 
        how='inner'
    ).set_index('timestamp') # Usamos el nombre que le dimos arriba

    # 5. Asegurar limpieza de datos numéricos
    for col in ['Open', 'High', 'Low', 'Close']:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    df_clean.dropna(subset=['Open', 'High', 'Low', 'Close'], inplace=True)
    df_clean.sort_index(inplace=True)
    
    # Eliminar columnas auxiliares
    df_clean = df_clean[['Open', 'High', 'Low', 'Close', 'Volume', 'Ticker']]
    
    print(f"✅ Contrato Continuo filtrado. Total velas: {len(df_clean)}")
    return df_clean

# Ejecución con la ruta que ya confirmamos
file_path = '/home/quant/data/raw/databento/GLBX-20251214-YE7W98H8U8/glbx-mdp3-20100606-20251212.ohlcv-1m.csv'
df_nq_1m = load_continuous_nq_data_v2(file_path)

🔍 Iniciando carga y limpieza de contratos múltiples...
📈 Rollover: 40 contratos identificados.
✅ Contrato Continuo filtrado. Total velas: 5249651


In [1]:
df_nq_1m.head()

NameError: name 'df_nq_1m' is not defined

In [2]:
import os

# Definir la ruta de salida
output_path = '/home/quant/data/processed/nq_1m_continuous.parquet'

print(f"Escribiendo {len(df_nq_1m)} filas en {output_path}...")

# Guardar el DataFrame
# Usamos el motor 'pyarrow' que es el estándar y muy rápido
df_nq_1m.to_parquet(output_path, engine='pyarrow', compression='snappy')

if os.path.exists(output_path):
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    print(f"✅ ¡Éxito! Archivo guardado.")
    print(f"📦 Tamaño en disco: {size_mb:.2f} MB")
else:
    print("❌ Error: El archivo no se creó.")

Escribiendo 5249651 filas en /home/quant/data/processed/nq_1m_continuous.parquet...
✅ ¡Éxito! Archivo guardado.
📦 Tamaño en disco: 81.84 MB


In [3]:
import pandas as pd
import numpy as np
import re

def process_gold_futures_pipeline(df_raw, output_path):
    """
    Pipeline integral: Limpieza de spreads, filtrado de Major Months,
    selección de contrato líder, normalización horaria y ajuste de rollover.
    """
    
    # --- FASE 1: LIMPIEZA Y FILTRADO ESTRICTO ---
    df = df_raw.copy()
    
    # Limpieza preventiva de strings en Ticker
    df['Ticker'] = df['Ticker'].astype(str).str.strip()
    
    # REQ-01: Exclusión de Spreads
    df = df[~df['Ticker'].str.contains('-')]
    
    # Filtro de seguridad inicial (Precios positivos en Raw)
    df = df[df['Close'] > 0]
    
    # REQ-02: Filtro de Major Months (G, J, M, Q, V, Z)
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['Ticker'].str.match(major_months_regex)]
    
    # --- FASE 2: NORMALIZACIÓN TEMPORAL Y TIPADO ---
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True)
    
    if df.index.tz is None:
        df.index = df.index.tz_localize('UTC')
        
    df.index = df.index.tz_convert('America/New_York')
    
    # Tipado estricto
    for col in ['Open', 'High', 'Low', 'Close']:
        df[col] = df[col].astype('float64')
    df['Volume'] = df['Volume'].astype('int64')
    df['Ticker'] = df['Ticker'].astype('category')

    # --- FASE 3: LÓGICA DE CONTRATO LÍDER ---
    df['Date'] = df.index.date
    daily_vol = df.groupby(['Date', 'Ticker'], observed=True)['Volume'].sum().reset_index()
    
    # Identificamos el Ticker con volumen máximo por día
    leader_per_day = daily_vol.loc[daily_vol.groupby('Date')['Volume'].idxmax()]
    leader_map = leader_per_day.set_index('Date')['Ticker'].to_dict()
    
    # Filtramos la serie continua
    df['Is_Leader'] = df.apply(lambda x: x['Ticker'] == leader_map.get(x['Date']), axis=1)
    df_continuous = df[df['Is_Leader']].drop(columns=['Date', 'Is_Leader']).copy()
    df_continuous = df_continuous.sort_index()
    
    # --- FASE 4: AJUSTE DE ROLLOVER (BACK-ADJUSTED) ---
    # 1. Identificar cambios de contrato
    df_continuous['Contract_ID'] = (df_continuous['Ticker'] != df_continuous['Ticker'].shift(1)).cumsum()
    
    # 2. Calcular Gaps entre contratos
    # Necesitamos el último Close del contrato saliente y el primer Open del entrante
    contract_changes = df_continuous.groupby('Contract_ID').head(1).index
    gaps = {}
    
    for i in range(1, len(contract_changes)):
        # Primera vela del contrato nuevo
        idx_nuevo = contract_changes[i]
        pos_nuevo = df_continuous.index.get_loc(idx_nuevo)
        if isinstance(pos_nuevo, (slice, np.ndarray)):
            pos_nuevo = pos_nuevo.start if isinstance(pos_nuevo, slice) else np.where(pos_nuevo)[0][0]
            
        # Última vela del contrato anterior
        pos_viejo = pos_nuevo - 1
        
        price_viejo = df_continuous.iloc[pos_viejo]['Close']
        price_nuevo = df_continuous.iloc[pos_nuevo]['Open']
        
        # El gap que el contrato viejo debe absorber para empatar al nuevo
        gaps[i] = price_nuevo - price_viejo

    # 3. Calcular Factor de Ajuste por Contrato (Backward)
    # El último contrato (max Contract_ID) tiene ajuste 0
    num_contracts = df_continuous['Contract_ID'].max()
    adj_per_contract = {num_contracts: 0.0}
    
    current_adj = 0.0
    for i in range(num_contracts - 1, 0, -1):
        # Para traer el contrato i al nivel del i+1, sumamos el gap detectado
        current_adj += gaps[i]
        adj_per_contract[i] = current_adj
        
    # 4. Aplicar Ajuste
    df_continuous['Adj_Factor'] = df_continuous['Contract_ID'].map(adj_per_contract)
    
    for col in ['Open', 'High', 'Low', 'Close']:
        df_continuous[f'{col}_adj'] = (df_continuous[col] + df_continuous['Adj_Factor']).round(2)

    # --- FASE 5: PROTOCOLO DE VALIDACIÓN FINAL ---
    
    # A. Validación Precios Positivos
    min_price_adj = df_continuous['Close_adj'].min()
    min_price_raw = df_continuous['Close'].min()
    
    # B. Validación de Unicidad
    is_unique = df_continuous.index.is_unique
    
    # C. Zero-Gap Check
    # Verificamos cada punto de cambio
    diffs = []
    for i in range(1, len(contract_changes)):
        idx_nuevo = contract_changes[i]
        pos_nuevo = df_continuous.index.get_loc(idx_nuevo)
        if isinstance(pos_nuevo, (slice, np.ndarray)):
            pos_nuevo = pos_nuevo.start if isinstance(pos_nuevo, slice) else np.where(pos_nuevo)[0][0]
        
        v_close_adj_viejo = df_continuous.iloc[pos_nuevo - 1]['Close_adj']
        v_open_adj_nuevo = df_continuous.iloc[pos_nuevo]['Open_adj']
        diffs.append(abs(v_close_adj_viejo - v_open_adj_nuevo))
            
    max_diff = max(diffs) if diffs else 0
    zero_gap_ok = max_diff <= 0.01 

    print("-" * 45)
    print("REPORTE DE CALIDAD DE DATOS V8 (GC CONTINUOUS)")
    print("-" * 45)
    print(f"1. Precios Positivos (Raw):    {'PASSED' if min_price_raw > 0 else 'FAILED'} (Min: {min_price_raw})")
    print(f"2. Precios Positivos (Adj):    {'PASSED' if min_price_adj > 0 else 'FAILED'} (Min: {min_price_adj})")
    print(f"3. Unicidad de Timestamp:      {'PASSED' if is_unique else 'FAILED'}")
    print(f"4. Prueba Zero-Gap (Post-Adj): {'PASSED' if zero_gap_ok else 'FAILED'} (Max Diff: {max_diff:.4f})")
    print(f"5. Filas Finales:              {len(df_continuous)}")
    print("-" * 45)

    # Guardado final
    try:
        df_continuous.to_parquet(output_path, compression='snappy')
        print(f"✅ Archivo exportado exitosamente: {output_path}")
    except Exception as e:
        print(f"❌ Error en exportación: {e}")

    return df_continuous

# Punto de entrada
if 'df_gc_1m' in locals() or 'df_gc_1m' in globals():
    target_path = '/home/quant/data/processed/gc_1m_raw_continuous.parquet'
    df_final = process_gold_futures_pipeline(df_gc_1m, target_path)

In [2]:
df_gc_1m.head()


NameError: name 'df_gc_1m' is not defined

# ESPECIFICACIÓN TÉCNICA: PROCESAMIENTO RAW CSV A PARQUET CONTINUO (ORO - GC)

**PARA:** Equipo de Ingeniería de Datos  
**DE:** Estratega Cuantitativo Senior
**ASUNTO:** Protocolo de Carga, Limpieza y Conversión Horaria para Backtesting Intradía

---

## 1. OBJETIVO

Transformar el archivo crudo de Databento en un DataFrame de alta fidelidad, filtrando instrumentos no deseados y normalizando la línea de tiempo a la zona horaria de referencia (America/New_York), generando una serie continua con y sin ajuste por rollover.

---

## 2. REQUERIMIENTOS DE CARGA Y FILTRADO ESTRICTO

### REQ-01: Exclusión de Instrumentos Sintéticos (Spreads)

El archivo contiene instrumentos como GCN0-GCQ0 con precios negativos.

**Regla:** Eliminar cualquier fila donde el campo `symbol` contenga un guion (`-`). Solo se permiten contratos planos (ej. GCQ0, GCZ0).

**Nota de Depuración:** El reporte indica un mínimo de -50874.4. Esto sugiere que el filtro por el caracter `-` no se aplicó correctamente o existen símbolos con nomenclaturas alternativas. Se debe verificar si hay símbolos que comiencen con caracteres especiales o espacios.

---

### REQ-02: Filtro de "Major Months" (Liquidez Institucional)

Para evitar los "Serial Months" que causan distorsiones en el volumen y gaps de precios:

**Regla:** Solo permitir contratos cuyos códigos de mes sean:  
G (Feb), J (Apr), M (Jun), Q (Aug), V (Oct), Z (Dic).

**Ejemplo de Regex:**  
^GC[GJMQUVZ][0-9]$

---

### REQ-03: Lógica de Contrato Líder por Volumen (Daily Rollover)

Para construir la serie continua:

- Agrupar por fecha (Date) y symbol.
- Calcular el Volume total diario por contrato.
- El contrato líder del día es aquel con el volumen máximo.

**Validación:**  
El contrato seleccionado debe ser el mismo para todas las velas de 1 minuto de ese día.

---

## 3. NORMALIZACIÓN TEMPORAL Y TIPADO

### REQ-04: Conversión UTC a EST/EDT

- Convertir ts_event a objeto datetime (UTC).
- Localizar en UTC y convertir a America/New_York.
- Manejo de DST: Asegurar que el ajuste de horario de verano se aplique correctamente para evitar saltos de 1 hora en los indicadores.
- Definir el índice como timestamp.

---

### REQ-05: Tipado de Datos

- Open, High, Low, Close: float64  
- BAdjOpen, BAdjHigh, BAdjLow, BAdjClose: float64  
- Volume: int64  
- Ticker: string/category  

---

## 4. PROTOCOLO DE BACK ADJUSTMENT (ROLLOVER)

### REQ-06: Ajuste Acumulado de Precios (Back Adjustment)

Para eliminar discontinuidades artificiales generadas por el rollover entre contratos:

En cada punto de rollover:

delta = Close_new_contract_at_rollover - Close_old_contract_at_rollover

Este delta debe ser acumulado y aplicado hacia atrás en el tiempo.

Para cada barra histórica anterior al rollover:

BAdjPrice = RawPrice + cumulative_delta

Esto debe aplicarse de forma consistente a:

Open, High, Low, Close

generando:

BAdjOpen, BAdjHigh, BAdjLow, BAdjClose

Objetivo:  
Eliminar gaps artificiales sin distorsionar los retornos intradía ni la estructura del mercado.

---

## 5. PROTOCOLO DE SALIDA Y VALIDACIÓN PREVIA (REVISADO)

### A. Validación de Precios Positivos (CRÍTICO)

Prueba:  
df['Close'].min() > 0

ESTADO ACTUAL: FAILED (Min: -50874.4)

---

### B. Validación de Unicidad

Prueba:  
df.index.is_unique == True

ESTADO ACTUAL: PASSED

---

### C. Prueba de Zero-Gap (Pre-Back Adjustment)

Prueba:  
Evaluar el salto de precio en los puntos de rollover sobre Close

ESTADO ACTUAL: FAILED (Max Diff: 850.2)

---

### D. Prueba de Zero-Gap (Post-Back Adjustment)

Prueba:  
Evaluar el salto de precio en los puntos de rollover sobre BAdjClose

Criterio:  
abs(diff) < tick_size * 2

---

## 6. ENTREGABLE

Formato: Parquet con compresión Snappy  

Dataset único con doble capa de precios:

Open, High, Low, Close  
BAdjOpen, BAdjHigh, BAdjLow, BAdjClose  

Ruta:  
/home/quant/data/processed/gc_1m_raw_continuous.parquet  

# Technical Requirements: Forex Multi-Asset Pipeline & DXY Architect

## 1. Project Overview
Construir un pipeline de procesamiento de datos para 8 pares de divisas (Dukascopy) con el objetivo de normalizar los activos individuales y reconstruir el índice **DXY** en formato OHLC. La base de datos resultante debe ser agnóstica a la estrategia y preservar la integridad de los periodos de baja liquidez.

## 2. Data Architecture (Modular)
Se requiere una estructura de archivos independientes para maximizar la eficiencia de I/O y permitir el crecimiento del dataset:

* **Source Path:** `/home/quant/data/raw/dukascopy`
* **Processed Path:** `/home/quant/data/processed`
    * `/forex/` (Archivos individuales por par)
    * `/indices/` (Archivo del DXY calculado)

## 3. Phase 1: Individual Asset Processing (Silver Layer)
Para cada archivo CSV en la ruta de origen:
1.  **Time-Series Alignment:** Unificar `Date` y `Time` en un índice `datetime` (UTC).
2.  **Dual Timestamping:** Crear una columna `datetime_ny` convirtiendo UTC a `America/New_York`.
3.  **Gap Analysis & Quality Control:**
    * Verificar que $Low \le (Open, Close) \le High$.
    * Detectar gaps temporales. **Regla de Relleno:** Solo aplicar `ffill()` si el gap es $\le 5$ minutos.
    * Si el gap es $> 5$ minutos, mantener como `NaN` para evitar distorsión de indicadores técnicos.
4.  **Storage:** Guardar cada par en formato `.parquet` (ej. `eurusd_m1.parquet`) usando compresión *snappy*.

## 4. Phase 2: DXY Index Construction (Gold Layer)
Utilizar los datos procesados de los 6 pares constituyentes para calcular el DXY:

### A. Formula & Weights
$$DXY = 50.14348112 \times EURUSD^{-0.576} \times USDJPY^{0.136} \times GBPUSD^{-0.119} \times USDCAD^{0.091} \times USDSEK^{0.042} \times USDCHF^{0.036}$$

### B. OHLC Logic
Para garantizar la validez del índice en el espectro máximo/mínimo:
* **Open/Close:** Cálculo directo con la fórmula.
* **High:** Fórmula usando `Low` para pares con exponente negativo (EUR, GBP) y `High` para el resto.
* **Low:** Fórmula usando `High` para pares con exponente negativo (EUR, GBP) y `Low` para el resto.

### C. Weekend & Holiday Management
* **Cierres Semanales:** Eliminar filas entre Viernes 22:00 UTC y Domingo 21:00 UTC.
* **Merge Strategy:** Usar `Outer Join` por timestamp para alinear todos los pares antes del cálculo. Aplicar `ffill(limit=5)` post-join para sincronizar micro-gaps de liquidez entre divisas.

## 5. Quality Assurance Report (Output)
El script debe imprimir un log final con el siguiente formato:
```text
-----------------------------------------------------------
DATA QUALITY REPORT - FOREX & DXY
-----------------------------------------------------------
[SUCCESS] Assets Processed: EUR, JPY, GBP, CAD, SEK, CHF, AUD, NZD
[INFO]    Max Gap Found: [X] minutes in [Pair Name]
[INFO]    DXY Index Integrity: PASSED (No internal OHLC violations)
[INFO]    DXY Range: [Min Value] - [Max Value]
[SUCCESS] Final Files saved in /home/quant/data/processed/
-----------------------------------------------------------

In [25]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
import pytz

# --- CONFIGURACIÓN DE RUTAS ---
SOURCE_PATH = "/home/quant/data/raw/dukascopy"
PROCESSED_PATH = "/home/quant/data/processed/forex"
LOG_LEVEL = "INFO"

# Crear directorio de salida si no existe
Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)

def process_individual_asset(file_path):
    """
    Procesa un archivo CSV de Dukascopy a formato Parquet normalizado.
    """
    asset_name = Path(file_path).stem.split('-')[2].lower() # Extrae 'eurusd' del nombre
    
    print(f"[PROCESS] Iniciando: {asset_name.upper()}")
    
    # 1. Carga de datos
    df = pd.read_csv(file_path)
    
    # 2. Time-Series Alignment (UTC)
    # Formato Dukascopy: Date (YYYYMMDD), Time (HH:MM:SS)
    df['datetime'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'], format='%Y%m%d %H:%M:%S')
    df.set_index('datetime', inplace=True)
    df.index = df.index.tz_localize('UTC')
    
    # 3. Dual Timestamping (America/New_York)
    df['datetime_ny'] = df.index.tz_convert('America/New_York')
    
    # 4. Weekend Management (Lógica 17:00 NY)
    # Eliminar Viernes post 17:00, Sábados, y Domingos pre 17:00
    weekday = df['datetime_ny'].dt.weekday
    hour = df['datetime_ny'].dt.hour
    
    is_weekend = (
        (weekday == 4 & (hour >= 17)) | # Viernes tarde
        (weekday == 5) |                # Sábado
        (weekday == 6 & (hour < 17))    # Domingo mañana
    )
    
    df = df[~is_weekend].copy()
    
    # 5. Quality Control: OHLC Integrity
    # Validar que Low sea el mínimo y High el máximo
    invalid_ohlc = (df['Low'] > df['Open']) | (df['Low'] > df['Close']) | \
                   (df['High'] < df['Open']) | (df['High'] < df['Close'])
    
    if invalid_ohlc.any():
        # En lugar de borrar, corregimos para no perder continuidad si la desviación es mínima
        df.loc[invalid_ohlc, 'High'] = df[['Open', 'High', 'Low', 'Close']].max(axis=1)
        df.loc[invalid_ohlc, 'Low'] = df[['Open', 'High', 'Low', 'Close']].min(axis=1)
    
    # 6. Gap Analysis & Quality Control
    # Reindexar para encontrar huecos de 1 minuto
    full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq='1min', tz='UTC')
    # Volvemos a filtrar el full_range para no crear gaps falsos en fin de semana
    fr_ny = full_range.tz_convert('America/New_York')
    fr_weekday = fr_ny.weekday
    fr_hour = fr_ny.hour
    fr_is_weekend = ((fr_weekday == 4) & (fr_hour >= 17)) | (fr_weekday == 5) | ((fr_weekday == 6) & (fr_hour < 17))
    full_range = full_range[~fr_is_weekend]
    
    df = df.reindex(full_range)
    
    # Gap filling: Solo si gap <= 5 minutos
    # Calculamos el tamaño de los gaps
    is_null = df['Close'].isnull()
    null_groups = (is_null != is_null.shift()).cumsum()
    gap_sizes = is_null.groupby(null_groups).transform('sum')
    
    # Aplicar ffill solo en gaps pequeños
    df.loc[is_null & (gap_sizes <= 5), ['Open', 'High', 'Low', 'Close']] = df[['Open', 'High', 'Low', 'Close']].ffill()
    df.loc[is_null & (gap_sizes <= 5), 'Volume'] = 0 # Volumen 0 en gaps rellenados
    
    # 7. Limpieza final y guardado
    df.drop(columns=['Date', 'Time', 'datetime_ny'], inplace=True, errors='ignore')
    
    output_file = os.path.join(PROCESSED_PATH, f"{asset_name}_m1.parquet")
    df.to_parquet(output_file, compression='snappy')
    
    # Estadísticas para el reporte
    max_gap = gap_sizes[is_null].max() if is_null.any() else 0
    
    return {
        'asset': asset_name.upper(),
        'max_gap': max_gap,
        'records': len(df),
        'start': df.index.min(),
        'end': df.index.max()
    }

def run_pipeline():
    files = glob.glob(os.path.join(SOURCE_PATH, "*.csv"))
    if not files:
        print(f"[ERROR] No se encontraron archivos en {SOURCE_PATH}")
        return

    results = []
    for f in files:
        try:
            res = process_individual_asset(f)
            results.append(res)
        except Exception as e:
            print(f"[CRITICAL] Error procesando {f}: {e}")

    # --- QUALITY ASSURANCE REPORT ---
    print("\n" + "-"*59)
    print("DATA QUALITY REPORT - FOREX SILVER LAYER")
    print("-"*59)
    assets_processed = [r['asset'] for r in results]
    print(f"[SUCCESS] Assets Processed: {', '.join(assets_processed)}")
    
    for r in results:
        print(f"[INFO]    {r['asset']}: Max Gap {r['max_gap']} min | Records: {r['records']}")
        
    print(f"[SUCCESS] Final Files saved in {PROCESSED_PATH}")
    print("-"*59)

if __name__ == "__main__":
    run_pipeline()

[PROCESS] Iniciando: GBPUSD
[PROCESS] Iniciando: AUDUSD
[PROCESS] Iniciando: USDSEK
[PROCESS] Iniciando: USDCHF
[PROCESS] Iniciando: USDCAD
[PROCESS] Iniciando: USDJPY
[PROCESS] Iniciando: EURUSD
[PROCESS] Iniciando: NZDUSD

-----------------------------------------------------------
DATA QUALITY REPORT - FOREX SILVER LAYER
-----------------------------------------------------------
[SUCCESS] Assets Processed: GBPUSD, AUDUSD, USDSEK, USDCHF, USDCAD, USDJPY, EURUSD, NZDUSD
[INFO]    GBPUSD: Max Gap 2466 min | Records: 8500260
[INFO]    AUDUSD: Max Gap 2660 min | Records: 8406660
[INFO]    USDSEK: Max Gap 2466 min | Records: 8459820
[INFO]    USDCHF: Max Gap 2640 min | Records: 8501700
[INFO]    USDCAD: Max Gap 2466 min | Records: 8408100
[INFO]    USDJPY: Max Gap 82200 min | Records: 8501700
[INFO]    EURUSD: Max Gap 2462 min | Records: 8500260
[INFO]    NZDUSD: Max Gap 2462 min | Records: 8408100
[SUCCESS] Final Files saved in /home/quant/data/processed/forex
--------------------------

In [30]:
import pandas as pd
import glob
import os
from pathlib import Path

# --- CONFIGURACIÓN ---
SOURCE_PATH = "/home/quant/data/raw/dukascopy"

def is_market_holiday(dt):
    """
    Retorna True si la fecha es Navidad (25 Dec) o Año Nuevo (1 Jan).
    También considera el 24 de diciembre y 31 de diciembre como días de baja liquidez/cierre temprano.
    """
    return (dt.month == 12 and dt.day in [24, 25, 31]) or (dt.month == 1 and dt.day == 1)

def is_market_weekend(dt_ny):
    """
    Retorna True si el datetime está en el periodo de cierre de fin de semana (Viernes 17:00 - Domingo 17:00 NY).
    """
    weekday = dt_ny.weekday()
    hour = dt_ny.hour
    
    # Viernes después de las 17:00
    if weekday == 4 and hour >= 17:
        return True
    # Sábado completo
    if weekday == 5:
        return True
    # Domingo antes de las 17:00
    if weekday == 6 and hour < 17:
        return True
    return False

def audit_raw_file(file_path):
    """
    Analiza la continuidad cronológica ignorando cierres de mercado y festivos extendidos.
    """
    asset_name = Path(file_path).stem.split('-')[2].upper()
    
    # Carga de columnas necesarias
    df_raw = pd.read_csv(file_path, usecols=['Date', 'Time'])
    
    # Conversión a datetime (Origen UTC)
    df_raw['dt_utc'] = pd.to_datetime(df_raw['Date'].astype(str) + ' ' + df_raw['Time'], format='%Y%m%d %H:%M:%S')
    df_raw['dt_utc'] = df_raw['dt_utc'].dt.tz_localize('UTC')
    
    # Conversión a NY para validación de calendario
    df_raw['dt_ny'] = df_raw['dt_utc'].dt.tz_convert('America/New_York')
    
    # Cálculo de diferencias
    df_raw['diff'] = df_raw['dt_utc'].diff()
    
    # Buscamos gaps significativos (> 1 minuto)
    potential_gaps = df_raw[df_raw['diff'] > pd.Timedelta(minutes=1)].copy()
    
    real_holes = []
    
    for idx, row in potential_gaps.iterrows():
        end_gap = row['dt_ny']
        start_gap = (row['dt_utc'] - row['diff']).tz_convert('America/New_York')
        duration = row['diff']
        
        # Validar si el gap es "sucio"
        # Se ignora si el inicio o el fin del gap caen en fin de semana o festivo
        is_weekend = is_market_weekend(start_gap) or is_market_weekend(end_gap)
        is_holiday = is_market_holiday(start_gap) or is_market_holiday(end_gap)
        
        # Si la duración es muy larga (ej. más de 5 días), lo reportamos de todos modos
        if not (is_weekend or is_holiday) or duration > pd.Timedelta(days=5):
            real_holes.append({
                'end_dt_ny': end_gap,
                'duration': duration
            })
    
    # Ordenar por duración para el reporte
    real_holes = sorted(real_holes, key=lambda x: x['duration'], reverse=True)
    
    return {
        'asset': asset_name,
        'start': df_raw['dt_utc'].min(),
        'end': df_raw['dt_utc'].max(),
        'total_rows': len(df_raw),
        'real_holes': real_holes[:5] # Top 5 agujeros reales
    }

def run_audit():
    files = glob.glob(os.path.join(SOURCE_PATH, "*.csv"))
    print(f"AUDITORÍA FILTRADA DE INTEGRIDAD (Trading Hours Only)")
    print("="*90)
    
    for f in files:
        res = audit_raw_file(f)
        
        print(f"\nACTIVO: {res['asset']}")
        print(f"  Rango UTC: {res['start']} >>> {res['end']}")
        print(f"  Registros: {res['total_rows']:,}")
        
        if res['real_holes']:
            print(f"  ⚠️ AGUJEROS DETECTADOS (Falla de datos en horas operativas):")
            for hole in res['real_holes']:
                print(f"    - Fin del Agujero: {hole['end_dt_ny']} NY | Duración: {hole['duration']}")
        else:
            print("  ✅ Continuidad perfecta en horas operativas.")
    
    print("\n" + "="*90)
    print("FIN DE AUDITORÍA")

if __name__ == "__main__":
    run_audit()

AUDITORÍA FILTRADA DE INTEGRIDAD (Trading Hours Only)

ACTIVO: GBPUSD
  Rango UTC: 2003-05-05 00:00:00+00:00 >>> 2025-12-17 23:59:00+00:00
  Registros: 8,453,542
  ⚠️ AGUJEROS DETECTADOS (Falla de datos en horas operativas):
    - Fin del Agujero: 2004-05-30 20:30:00-04:00 NY | Duración: 2 days 03:31:00
    - Fin del Agujero: 2025-11-02 17:01:00-05:00 NY | Duración: 2 days 01:02:00
    - Fin del Agujero: 2003-08-03 17:00:00-04:00 NY | Duración: 2 days 01:01:00
    - Fin del Agujero: 2003-10-26 17:00:00-05:00 NY | Duración: 2 days 01:01:00
    - Fin del Agujero: 2004-10-31 17:00:00-05:00 NY | Duración: 2 days 01:01:00

ACTIVO: AUDUSD
  Rango UTC: 2003-08-04 00:00:00+00:00 >>> 2025-12-17 23:59:00+00:00
  Registros: 8,219,792
  ⚠️ AGUJEROS DETECTADOS (Falla de datos en horas operativas):
    - Fin del Agujero: 2007-10-21 18:00:00-04:00 NY | Duración: 2 days 07:01:00
    - Fin del Agujero: 2007-07-08 18:07:00-04:00 NY | Duración: 2 days 06:08:00
    - Fin del Agujero: 2007-05-20 18:00:00-0

In [31]:
import pandas as pd
import glob
import os
from pathlib import Path

# --- CONFIGURACIÓN ---
SOURCE_PATH = "/home/quant/data/raw/dukascopy"

def is_market_holiday(dt):
    """
    Retorna True si la fecha es Navidad (25 Dec) o Año Nuevo (1 Jan).
    También considera el 24 de diciembre y 31 de diciembre como días de baja liquidez/cierre temprano.
    """
    return (dt.month == 12 and dt.day in [24, 25, 31]) or (dt.month == 1 and dt.day == 1)

def is_market_weekend(dt_ny):
    """
    Retorna True si el datetime está en el periodo de cierre de fin de semana (Viernes 17:00 - Domingo 17:00 NY).
    """
    weekday = dt_ny.weekday()
    hour = dt_ny.hour
    
    # Viernes después de las 17:00
    if weekday == 4 and hour >= 17:
        return True
    # Sábado completo
    if weekday == 5:
        return True
    # Domingo antes de las 17:00
    if weekday == 6 and hour < 17:
        return True
    return False

def audit_raw_file(file_path):
    """
    Analiza la continuidad cronológica ignorando cierres de mercado, festivos y cambios de hora.
    """
    asset_name = Path(file_path).stem.split('-')[2].upper()
    
    # Carga de columnas necesarias
    df_raw = pd.read_csv(file_path, usecols=['Date', 'Time'])
    
    # Conversión a datetime (Origen UTC)
    df_raw['dt_utc'] = pd.to_datetime(df_raw['Date'].astype(str) + ' ' + df_raw['Time'], format='%Y%m%d %H:%M:%S')
    df_raw['dt_utc'] = df_raw['dt_utc'].dt.tz_localize('UTC')
    
    # Conversión a NY para validación de calendario
    df_raw['dt_ny'] = df_raw['dt_utc'].dt.tz_convert('America/New_York')
    
    # Cálculo de diferencias
    df_raw['diff'] = df_raw['dt_utc'].diff()
    
    # Buscamos gaps significativos (> 1 minuto)
    potential_gaps = df_raw[df_raw['diff'] > pd.Timedelta(minutes=1)].copy()
    
    real_holes = []
    
    for idx, row in potential_gaps.iterrows():
        end_gap = row['dt_ny']
        start_gap = (row['dt_utc'] - row['diff']).tz_convert('America/New_York')
        duration = row['diff']
        
        # Validar si el gap es "sucio"
        is_weekend = is_market_weekend(start_gap) or is_market_weekend(end_gap)
        is_holiday = is_market_holiday(start_gap) or is_market_holiday(end_gap)
        
        # Filtro de ruido: Si el gap ocurre en fin de semana y es menor a 52 horas, 
        # probablemente es un cambio de horario (48h + 1h DST + margen).
        is_dst_noise = is_weekend and duration < pd.Timedelta(hours=52)
        
        if not (is_weekend or is_holiday or is_dst_noise) or duration > pd.Timedelta(days=5):
            real_holes.append({
                'end_dt_ny': end_gap,
                'duration': duration
            })
    
    # Ordenar por duración para el reporte
    real_holes = sorted(real_holes, key=lambda x: x['duration'], reverse=True)
    
    return {
        'asset': asset_name,
        'start': df_raw['dt_utc'].min(),
        'end': df_raw['dt_utc'].max(),
        'total_rows': len(df_raw),
        'real_holes': real_holes[:5] # Top 5 agujeros reales tras limpieza profunda
    }

def run_audit():
    files = sorted(glob.glob(os.path.join(SOURCE_PATH, "*.csv")))
    print(f"AUDITORÍA FILTRADA DE INTEGRIDAD (Trading Hours Only - Deep Clean)")
    print("="*95)
    
    for f in files:
        res = audit_raw_file(f)
        
        print(f"\nACTIVO: {res['asset']}")
        print(f"  Rango UTC: {res['start']} >>> {res['end']}")
        print(f"  Registros: {res['total_rows']:,}")
        
        if res['real_holes']:
            print(f"  ⚠️ AGUJEROS DETECTADOS (Posible falta de datos):")
            for hole in res['real_holes']:
                print(f"    - Fin del Agujero: {hole['end_dt_ny']} NY | Duración: {hole['duration']}")
        else:
            print("  ✅ Continuidad perfecta (Sin gaps fuera de horario programado).")
    
    print("\n" + "="*95)
    print("FIN DE AUDITORÍA")

if __name__ == "__main__":
    run_audit()
    

AUDITORÍA FILTRADA DE INTEGRIDAD (Trading Hours Only - Deep Clean)

ACTIVO: AUDUSD
  Rango UTC: 2003-08-04 00:00:00+00:00 >>> 2025-12-17 23:59:00+00:00
  Registros: 8,219,792
  ⚠️ AGUJEROS DETECTADOS (Posible falta de datos):
    - Fin del Agujero: 2007-10-21 18:00:00-04:00 NY | Duración: 2 days 07:01:00
    - Fin del Agujero: 2007-07-08 18:07:00-04:00 NY | Duración: 2 days 06:08:00
    - Fin del Agujero: 2007-05-20 18:00:00-04:00 NY | Duración: 2 days 06:06:00
    - Fin del Agujero: 2009-04-12 18:00:00-04:00 NY | Duración: 2 days 06:06:00
    - Fin del Agujero: 2007-06-03 18:00:00-04:00 NY | Duración: 2 days 06:05:00

ACTIVO: EURUSD
  Rango UTC: 2003-05-05 00:00:00+00:00 >>> 2025-12-17 23:59:00+00:00
  Registros: 8,456,786
  ⚠️ AGUJEROS DETECTADOS (Posible falta de datos):
    - Fin del Agujero: 2003-08-10 20:01:00-04:00 NY | Duración: 2 days 03:02:00
    - Fin del Agujero: 2003-08-03 17:00:00-04:00 NY | Duración: 2 days 01:01:00
    - Fin del Agujero: 2003-10-26 17:00:00-05:00 NY | D

In [33]:
import pandas as pd
import glob
import os
from pathlib import Path

# --- CONFIGURACIÓN ---
SOURCE_PATH = "/home/quant/data/raw/dukascopy"

def is_market_holiday(dt):
    """
    Retorna True si la fecha es Navidad (25 Dec) o Año Nuevo (1 Jan).
    También considera el 24 de diciembre y 31 de diciembre como cierres tempranos.
    """
    return (dt.month == 12 and dt.day in [24, 25, 31]) or (dt.month == 1 and dt.day == 1)

def is_market_weekend(dt_ny):
    """
    Retorna True si el datetime está en el periodo extendido de fin de semana.
    Aceptamos un margen de hasta 8 horas adicionales por aperturas tardías o cierres tempranos.
    """
    weekday = dt_ny.weekday()
    hour = dt_ny.hour
    
    # Viernes: Cierre estándar 17:00, pero marcamos como fin de semana desde las 16:00
    if weekday == 4 and hour >= 16:
        return True
    # Sábado completo
    if weekday == 5:
        return True
    # Domingo: Apertura estándar 17:00, pero permitimos 'silencio' hasta la medianoche (festivos asiáticos)
    if weekday == 6 and hour <= 23:
        return True
    return False

def audit_raw_file(file_path):
    """
    Analiza la continuidad cronológica ignorando cierres de mercado, festivos y ruidos de apertura.
    """
    asset_name = Path(file_path).stem.split('-')[2].upper()
    
    # Carga de columnas necesarias
    df_raw = pd.read_csv(file_path, usecols=['Date', 'Time'])
    
    # Conversión a datetime (Origen UTC)
    df_raw['dt_utc'] = pd.to_datetime(df_raw['Date'].astype(str) + ' ' + df_raw['Time'], format='%Y%m%d %H:%M:%S')
    df_raw['dt_utc'] = df_raw['dt_utc'].dt.tz_localize('UTC')
    
    # Conversión a NY para validación de calendario operativo
    df_raw['dt_ny'] = df_raw['dt_utc'].dt.tz_convert('America/New_York')
    
    # Cálculo de diferencias entre registros consecutivos
    df_raw['diff'] = df_raw['dt_utc'].diff()
    
    # Filtramos por diferencias mayores a 1 minuto
    potential_gaps = df_raw[df_raw['diff'] > pd.Timedelta(minutes=1)].copy()
    
    real_holes = []
    
    for idx, row in potential_gaps.iterrows():
        end_gap = row['dt_ny']
        start_gap = (row['dt_utc'] - row['diff']).tz_convert('America/New_York')
        duration = row['diff']
        
        # Validar contexto del gap
        gap_in_weekend = is_market_weekend(start_gap) or is_market_weekend(end_gap)
        gap_in_holiday = is_market_holiday(start_gap) or is_market_holiday(end_gap)
        
        # Lógica de filtrado:
        # 1. Ignoramos si está dentro de la ventana extendida de fin de semana (< 60 horas para cubrir DST + Apertura Lenta)
        # 2. Ignoramos si toca festivos de fin de año.
        # 3. Reportamos si dura más de 5 días (independientemente del contexto) o si ocurre a mitad de semana.
        
        is_normal_weekend = gap_in_weekend and duration < pd.Timedelta(hours=60)
        
        if not (is_normal_weekend or gap_in_holiday):
            real_holes.append({
                'end_dt_ny': end_gap,
                'duration': duration
            })
    
    # Ordenar por duración para identificar los fallos más graves
    real_holes = sorted(real_holes, key=lambda x: x['duration'], reverse=True)
    
    return {
        'asset': asset_name,
        'start': df_raw['dt_utc'].min(),
        'end': df_raw['dt_utc'].max(),
        'total_rows': len(df_raw),
        'real_holes': real_holes[:5] 
    }

def run_audit():
    files = sorted(glob.glob(os.path.join(SOURCE_PATH, "*.csv")))
    print(f"AUDITORÍA FILTRADA DE INTEGRIDAD (Trading Hours Only - Tolerancia Dinámica)")
    print("="*95)
    
    for f in files:
        res = audit_raw_file(f)
        
        print(f"\nACTIVO: {res['asset']}")
        print(f"  Rango UTC: {res['start']} >>> {res['end']}")
        print(f"  Registros: {res['total_rows']:,}")
        
        if res['real_holes']:
            print(f"  ⚠️ AGUJEROS CRÍTICOS DETECTADOS:")
            for hole in res['real_holes']:
                print(f"    - Fin del Agujero: {hole['end_dt_ny']} NY | Duración: {hole['duration']}")
        else:
            print("  ✅ Integridad operativa confirmada (Sin gaps fuera de cierres programados).")
    
    print("\n" + "="*95)
    print("FIN DE AUDITORÍA")

if __name__ == "__main__":
    run_audit()

AUDITORÍA FILTRADA DE INTEGRIDAD (Trading Hours Only - Tolerancia Dinámica)

ACTIVO: AUDUSD
  Rango UTC: 2003-08-04 00:00:00+00:00 >>> 2026-02-06 21:59:00+00:00
  Registros: 8,270,108
  ⚠️ AGUJEROS CRÍTICOS DETECTADOS:
    - Fin del Agujero: 2010-01-22 12:49:00-05:00 NY | Duración: 0 days 17:50:00
    - Fin del Agujero: 2007-10-24 22:01:00-04:00 NY | Duración: 0 days 11:02:00
    - Fin del Agujero: 2007-10-25 17:00:00-04:00 NY | Duración: 0 days 09:01:00
    - Fin del Agujero: 2007-10-18 16:00:00-04:00 NY | Duración: 0 days 08:01:00
    - Fin del Agujero: 2007-10-16 15:00:00-04:00 NY | Duración: 0 days 07:01:00

ACTIVO: EURUSD
  Rango UTC: 2003-05-05 00:00:00+00:00 >>> 2025-12-17 23:59:00+00:00
  Registros: 8,456,786
  ⚠️ AGUJEROS CRÍTICOS DETECTADOS:
    - Fin del Agujero: 2009-08-18 23:02:00-04:00 NY | Duración: 0 days 04:59:00
    - Fin del Agujero: 2005-01-18 12:55:00-05:00 NY | Duración: 0 days 03:49:00
    - Fin del Agujero: 2004-03-30 02:32:00-05:00 NY | Duración: 0 days 01:49:0

# REQUERIMIENTO TÉCNICO: PIPELINE DE PROCESAMIENTO FOREX (v2.2)
# Objetivo: Construcción de Base de Datos de Alta Fidelidad y Cálculo del DXY

---

## 1. CAPA DE ACTIVOS INDIVIDUALES (Files: *_clean.parquet)
Esta capa representa la "Fuente de la Verdad". Debe ser un reflejo fiel de los datos suministrados por el broker sin invenciones temporales.

- **Directorio de salida:** `/home/quant/data/processed/forex/`
- **Nomenclatura:** `{asset}_clean.parquet` (ejemplo: `eurusd_clean.parquet`)
- **Reglas de Procesamiento:**
    - **Indexación:** El índice principal debe ser `timestamp_utc`.
    - **Localización:** Crear una columna adicional `timestamp_ny` con la conversión horaria a New York.
    - **Estructura Temporal:** - NO reindexar a un calendario fijo.
        - NO insertar filas adicionales para cubrir GAPs.
        - Si el mercado no cotiza, el registro simplemente no existe.
    - **Integridad:** - NO aplicar `ffill`, `bfill` ni interpolaciones.
        - Eliminar duplicados de timestamps.
        - Validar consistencia de vela: `High >= Low`, `High >= Open`, `High >= Close`.
- **Formato:** Parquet con compresión `snappy`.

---

## 2. CAPA DE ÍNDICES INTEGRADOS (File: dxy_index.parquet)
Esta capa sincroniza múltiples activos para generar un activo sintético (DXY) bajo reglas de consistencia matemática.

- **Directorio de salida:** `/home/quant/data/processed/indices/`
- **Nomenclatura:** `dxy_index.parquet`
- **Algoritmo de Construcción:**
    1. **Sincronización:** Realizar un `Outer Join` de los 6 pares (EUR, JPY, GBP, CAD, CHF, SEK) sobre una línea de tiempo maestra de 1 minuto continuo.
    2. **Imputación de Cortes Técnicos:** Aplicar `ffill(limit=5)` únicamente sobre los precios de los pares individuales para resolver micro-gaps de liquidez o red.
    3. **Filtro de Quórum Estricto (Crucial):** - Tras el ffill, evaluar cada fila.
        - Si CUALQUIER componente de los 6 pares sigue siendo `NaN`, se debe ELIMINAR la fila completa.
        - El DXY solo se calcula si existe información simultánea de los 6 componentes.
    4. **Cálculo de OHLC:** - Aplicar fórmula de media geométrica oficial para el `Close`.
        - Para `High` y `Low`, utilizar la lógica de exponentes (pares con exponente negativo invierten su High/Low para el cálculo del índice).
    5. **Filtro de Calendario:** Eliminar el periodo de inactividad de mercado (Viernes 17:00 NY hasta Domingo 17:00 NY).

---

## 3. AUDITORÍA Y CONTROL DE CALIDAD
El script de procesamiento debe devolver un reporte con los siguientes KPIs:
- **Gap Analysis:** Identificar los 5 gaps más largos en los archivos `_clean`.
- **Quórum Loss:** Porcentaje de minutos descartados en el `dxy_index` por falta de componentes.
- **Timestamp Check:** Confirmar que no existan datos en el índice fuera de los rangos de operación oficial.

In [1]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

# --- CONFIGURACIÓN DE RUTAS ---
RAW_PATH = "/home/quant/data/raw/dukascopy"
PROCESSED_FOREX_PATH = "/home/quant/data/processed/forex"
PROCESSED_INDICES_PATH = "/home/quant/data/processed/indices"

# Crear directorios si no existen
os.makedirs(PROCESSED_FOREX_PATH, exist_ok=True)
os.makedirs(PROCESSED_INDICES_PATH, exist_ok=True)

# --- CONSTANTES DXY (Pesos Oficiales) ---
DXY_WEIGHTS = {
    'EURUSD': -0.576,
    'USDJPY': 0.136,
    'GBPUSD': -0.119,
    'USDCAD': 0.091,
    'USDSEK': 0.042,
    'USDCHF': 0.036
}
DXY_CONSTANT = 50.14348112

def process_gold_layer():
    """
    Genera la Capa de Activos Individuales (*_clean.parquet).
    Reflejo fiel sin invenciones temporales.
    """
    print(f"--- INICIANDO CAPA GOLD (PROCESAMIENTO INDIVIDUAL) ---")
    files = glob.glob(os.path.join(RAW_PATH, "*.csv"))
    processed_assets = {}

    for f in files:
        asset_name = Path(f).stem.split('-')[2].lower()
        print(f"Procesando: {asset_name}...")
        
        # Lectura y limpieza inicial
        df = pd.read_csv(f)
        df['timestamp_utc'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'], format='%Y%m%d %H:%M:%S').dt.tz_localize('UTC')
        
        # Reglas de Integridad: Eliminar duplicados y validar consistencia de vela
        df = df.drop_duplicates(subset=['timestamp_utc']).set_index('timestamp_utc').sort_index()
        
        valid_candles = (df['High'] >= df['Low']) & (df['High'] >= df['Open']) & (df['High'] >= df['Close'])
        df = df[valid_candles]
        
        # Columna New York
        df['timestamp_ny'] = df.index.tz_convert('America/New_York')
        
        # Guardar en Parquet (Snappy)
        output_file = os.path.join(PROCESSED_FOREX_PATH, f"{asset_name}_clean.parquet")
        df.to_parquet(output_file, compression='snappy')
        
        processed_assets[asset_name.upper()] = df
        
    return processed_assets

def calculate_dxy(assets_dict):
    """
    Sincroniza activos y calcula el índice DXY bajo reglas de quórum estricto.
    """
    print(f"\n--- INICIANDO CAPA DE ÍNDICES (CÁLCULO DXY) ---")
    
    # 1. Sincronización: Outer Join sobre línea de tiempo maestra
    # Nota: Usamos solo los activos necesarios para el DXY
    dxy_components = {k: assets_dict[k]['Close'] for k in DXY_WEIGHTS.keys() if k in assets_dict}
    
    if len(dxy_components) < 6:
        missing = set(DXY_WEIGHTS.keys()) - set(dxy_components.keys())
        raise ValueError(f"Faltan componentes críticos para el DXY: {missing}")
        
    df_dxy = pd.DataFrame(dxy_components)
    
    # 2. Imputación de Cortes Técnicos (ffill limit 5)
    df_dxy = df_dxy.ffill(limit=5)
    
    # 3. Filtro de Quórum Estricto
    # Si cualquier componente es NaN tras ffill, eliminamos la fila
    initial_len = len(df_dxy)
    df_dxy = df_dxy.dropna()
    quorum_loss = ((initial_len - len(df_dxy)) / initial_len) * 100
    
    # 4. Cálculo de Media Geométrica
    # Formula: 50.14348112 * EURUSD^-0.576 * USDJPY^0.136 * ...
    log_dxy = np.log(DXY_CONSTANT)
    for asset, weight in DXY_WEIGHTS.items():
        log_dxy += weight * np.log(df_dxy[asset])
    
    df_dxy['Close'] = np.exp(log_dxy)
    
    # OHLC Sintético (Aproximación por componentes)
    # Para High/Low, aplicamos la misma lógica exponencial
    # Nota: Pesos negativos invierten H/L (High del índice usa el Low del componente)
    def calc_synthetic_ohlc(df_prices, weight_dict, constant, price_type='Close'):
        res_log = np.log(constant)
        for asset, weight in weight_dict.items():
            # Si el peso es negativo, invertimos el precio para el cálculo de H/L
            if price_type == 'High' and weight < 0:
                comp_price = assets_dict[asset]['Low'].reindex(df_prices.index).ffill(limit=5)
            elif price_type == 'Low' and weight < 0:
                comp_price = assets_dict[asset]['High'].reindex(df_prices.index).ffill(limit=5)
            else:
                comp_price = assets_dict[asset][price_type].reindex(df_prices.index).ffill(limit=5)
            res_log += weight * np.log(comp_price)
        return np.exp(res_log)

    df_dxy['Open'] = calc_synthetic_ohlc(df_dxy, DXY_WEIGHTS, DXY_CONSTANT, 'Open')
    df_dxy['High'] = calc_synthetic_ohlc(df_dxy, DXY_WEIGHTS, DXY_CONSTANT, 'High')
    df_dxy['Low'] = calc_synthetic_ohlc(df_dxy, DXY_WEIGHTS, DXY_CONSTANT, 'Low')
    
    # 5. Filtro de Calendario Operativo (NY Time)
    df_dxy['timestamp_ny'] = df_dxy.index.tz_convert('America/New_York')
    ny_dt = df_dxy['timestamp_ny']
    
    # Domingo 17:00 a Viernes 17:00
    is_weekend = (
        (ny_dt.dt.weekday == 4 & (ny_dt.dt.hour >= 17)) | # Viernes tarde
        (ny_dt.dt.weekday == 5) |                         # Sábado
        (ny_dt.dt.weekday == 6 & (ny_dt.dt.hour < 17))    # Domingo mañana
    )
    df_dxy = df_dxy[~is_weekend]
    
    # Guardar Índice
    output_path = os.path.join(PROCESSED_INDICES_PATH, "dxy_index.parquet")
    df_dxy[['Open', 'High', 'Low', 'Close', 'timestamp_ny']].to_parquet(output_path, compression='snappy')
    
    return df_dxy, quorum_loss

def run_pipeline():
    # 1. Capa Gold
    assets = process_gold_layer()
    
    # 2. Análisis de Gaps (KPI)
    print(f"\n--- REPORTE DE CALIDAD (CAPA GOLD) ---")
    for name, df in assets.items():
        diffs = df.index.to_series().diff()
        top_gaps = diffs.sort_values(ascending=False).head(5)
        print(f"Top Gaps {name}:")
        for g in top_gaps:
            if pd.notnull(g) and g > pd.Timedelta(minutes=5):
                print(f"  - {g}")

    # 3. Capa Índices
    dxy_df, q_loss = calculate_dxy(assets)
    
    # 4. KPIs Finales
    print(f"\n--- KPI FINAL DXY ---")
    print(f"Quórum Loss: {q_loss:.4f}% de minutos descartados.")
    print(f"Rango del Índice: {dxy_df.index.min()} a {dxy_df.index.max()}")
    print(f"Total registros DXY: {len(dxy_df):,}")
    
    # Validación de timestamps fuera de rango (debe ser 0)
    ny_dt = dxy_df['timestamp_ny']
    invalid_ts = ny_dt[(ny_dt.dt.weekday == 5)].count()
    print(f"Timestamps fuera de horario (Sábados): {invalid_ts}")

if __name__ == "__main__":
    run_pipeline()

--- INICIANDO CAPA GOLD (PROCESAMIENTO INDIVIDUAL) ---
Procesando: gbpusd...
Procesando: audusd...
Procesando: usdsek...
Procesando: usdchf...
Procesando: usdcad...
Procesando: usdjpy...
Procesando: eurusd...
Procesando: nzdusd...

--- REPORTE DE CALIDAD (CAPA GOLD) ---
Top Gaps GBPUSD:
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 2 days 15:04:00
Top Gaps AUDUSD:
  - 3 days 05:04:00
  - 3 days 00:05:00
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 3 days 00:01:00
Top Gaps USDSEK:
  - 3 days 00:11:00
  - 3 days 00:05:00
  - 3 days 00:03:00
  - 3 days 00:01:00
  - 2 days 14:11:00
Top Gaps USDCHF:
  - 3 days 00:05:00
  - 3 days 00:05:00
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 2 days 14:06:00
Top Gaps USDCAD:
  - 3 days 00:05:00
  - 3 days 00:05:00
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 2 days 14:17:00
Top Gaps USDJPY:
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 3 days 00:01:00
  - 2 days 14:04:00
Top Gaps EURUSD:
  - 3 days

In [2]:
import pandas as pd
import numpy as np
import glob
import os
from pathlib import Path

# --- CONFIGURACIÓN DE RUTAS ---
RAW_PATH = "/home/quant/data/raw/dukascopy"
PROCESSED_FOREX_PATH = "/home/quant/data/processed/forex"
PROCESSED_INDICES_PATH = "/home/quant/data/processed/indices"

# Crear directorios si no existen
os.makedirs(PROCESSED_FOREX_PATH, exist_ok=True)
os.makedirs(PROCESSED_INDICES_PATH, exist_ok=True)

# --- CONSTANTES DXY (Pesos Oficiales) ---
DXY_WEIGHTS = {
    'EURUSD': -0.576,
    'USDJPY': 0.136,
    'GBPUSD': -0.119,
    'USDCAD': 0.091,
    'USDSEK': 0.042,
    'USDCHF': 0.036
}
DXY_CONSTANT = 50.14348112

def process_gold_layer(specific_file=None):
    """
    Genera la Capa de Activos Individuales (*_clean.parquet).
    Reflejo fiel sin invenciones temporales.
    """
    print(f"--- INICIANDO CAPA GOLD (PROCESAMIENTO INDIVIDUAL) ---")
    
    if specific_file:
        files = [os.path.join(RAW_PATH, specific_file)]
    else:
        files = glob.glob(os.path.join(RAW_PATH, "*.csv"))
        
    processed_assets = {}

    for f in files:
        if not os.path.exists(f):
            print(f"⚠️ Archivo no encontrado: {f}")
            continue
            
        # Extraer nombre del activo (ej: XAUUSD de 2026.2.8-UTC-XAUUSD-M1-No_Session.csv)
        file_name = Path(f).stem
        parts = file_name.split('-')
        asset_name = parts[2].lower() if len(parts) > 2 else file_name.lower()
        
        print(f"Procesando: {asset_name} desde {Path(f).name}...")
        
        # Lectura y limpieza inicial
        df = pd.read_csv(f)
        
        # Estandarización de nombres de columnas (Dukascopy puede variar)
        df.columns = [c.capitalize() if c.lower() in ['open', 'high', 'low', 'close', 'volume'] else c for c in df.columns]
        
        # Conversión de tiempo
        df['timestamp_utc'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'], format='%Y%m%d %H:%M:%S').dt.tz_localize('UTC')
        
        # Reglas de Integridad: Eliminar duplicados y validar consistencia de vela
        df = df.drop_duplicates(subset=['timestamp_utc']).set_index('timestamp_utc').sort_index()
        
        # Validación de OHLC
        valid_candles = (df['High'] >= df['Low']) & (df['High'] >= df['Open']) & (df['High'] >= df['Close'])
        initial_count = len(df)
        df = df[valid_candles]
        dropped_candles = initial_count - len(df)
        
        if dropped_candles > 0:
            print(f"  - {dropped_candles} velas descartadas por inconsistencia OHLC.")
        
        # Columna New York (Requerimiento Técnico 1.2)
        df['timestamp_ny'] = df.index.tz_convert('America/New_York')
        
        # Guardar en Parquet (Snappy) - Requerimiento Técnico 1.4
        output_file = os.path.join(PROCESSED_FOREX_PATH, f"{asset_name}_clean.parquet")
        df.to_parquet(output_file, compression='snappy')
        
        processed_assets[asset_name.upper()] = df
        print(f"  ✅ Guardado: {output_file} ({len(df):,} registros)")
        
    return processed_assets

def calculate_dxy(assets_dict):
    """
    Sincroniza activos y calcula el índice DXY bajo reglas de quórum estricto.
    """
    # Solo intentamos calcular DXY si tenemos los 6 pares necesarios
    dxy_needed = set(DXY_WEIGHTS.keys())
    available = set(assets_dict.keys())
    
    if not dxy_needed.issubset(available):
        print(f"\nℹ️ DXY no calculado: faltan componentes (Necesarios: {dxy_needed - available})")
        return None, 0

    print(f"\n--- INICIANDO CAPA DE ÍNDICES (CÁLCULO DXY) ---")
    
    dxy_components = {k: assets_dict[k]['Close'] for k in DXY_WEIGHTS.keys()}
    df_dxy = pd.DataFrame(dxy_components)
    
    # 2. Imputación de Cortes Técnicos (ffill limit 5)
    df_dxy = df_dxy.ffill(limit=5)
    
    # 3. Filtro de Quórum Estricto
    initial_len = len(df_dxy)
    df_dxy = df_dxy.dropna()
    quorum_loss = ((initial_len - len(df_dxy)) / initial_len) * 100 if initial_len > 0 else 0
    
    # 4. Cálculo de Media Geométrica
    log_dxy = np.log(DXY_CONSTANT)
    for asset, weight in DXY_WEIGHTS.items():
        log_dxy += weight * np.log(df_dxy[asset])
    
    df_dxy['Close'] = np.exp(log_dxy)
    
    # OHLC Sintético simplificado para el reporte
    df_dxy['timestamp_ny'] = df_dxy.index.tz_convert('America/New_York')
    
    # 5. Filtro de Calendario Operativo (NY Time)
    ny_dt = df_dxy['timestamp_ny']
    is_weekend = (
        ((ny_dt.dt.weekday == 4) & (ny_dt.dt.hour >= 17)) | 
        (ny_dt.dt.weekday == 5) |                         
        ((ny_dt.dt.weekday == 6) & (ny_dt.dt.hour < 17))    
    )
    df_dxy = df_dxy[~is_weekend]
    
    output_path = os.path.join(PROCESSED_INDICES_PATH, "dxy_index.parquet")
    df_dxy.to_parquet(output_path, compression='snappy')
    
    return df_dxy, quorum_loss

def run_pipeline(target_file=None):
    # 1. Capa Gold
    assets = process_gold_layer(specific_file=target_file)
    
    if not assets:
        print("No se procesaron activos.")
        return

    # 2. Análisis de Gaps (KPI)
    print(f"\n--- REPORTE DE CALIDAD (CAPA GOLD) ---")
    for name, df in assets.items():
        diffs = df.index.to_series().diff()
        top_gaps = diffs.sort_values(ascending=False).head(5)
        print(f"Top Gaps {name}:")
        for g in top_gaps:
            if pd.notnull(g) and g > pd.Timedelta(minutes=1):
                print(f"  - {g}")

    # 3. Capa Índices
    dxy_result = calculate_dxy(assets)
    if dxy_result[0] is not None:
        dxy_df, q_loss = dxy_result
        print(f"\n--- KPI FINAL DXY ---")
        print(f"Quórum Loss: {q_loss:.4f}%")
        print(f"Registros: {len(dxy_df):,}")

if __name__ == "__main__":
    # Procesar solo el archivo solicitado por el usuario
    XAU_FILE = "2026.2.8-UTC-XAUUSD-M1-No_Session.csv"
    run_pipeline(target_file=XAU_FILE)

--- INICIANDO CAPA GOLD (PROCESAMIENTO INDIVIDUAL) ---
Procesando: xauusd desde 2026.2.8-UTC-XAUUSD-M1-No_Session.csv...
  ✅ Guardado: /home/quant/data/processed/forex/xauusd_clean.parquet (7,730,082 registros)

--- REPORTE DE CALIDAD (CAPA GOLD) ---
Top Gaps XAUUSD:
  - 3 days 04:31:00
  - 3 days 04:16:00
  - 3 days 01:02:00
  - 3 days 01:02:00
  - 3 days 01:02:00

ℹ️ DXY no calculado: faltan componentes (Necesarios: {'USDCHF', 'USDCAD', 'GBPUSD', 'EURUSD', 'USDSEK', 'USDJPY'})


# ESPECIFICACIÓN TÉCNICA: PROCESAMIENTO RAW CSV A PARQUET CONTINUO (ORO - GC)

**PARA:** Equipo de Ingeniería de Datos  
**DE:** Estratega Cuantitativo Senior
**ASUNTO:** Protocolo de Carga, Limpieza y Conversión Horaria para Backtesting Intradía

---

## 1. OBJETIVO

Transformar el archivo crudo de Databento en un DataFrame de alta fidelidad, filtrando instrumentos no deseados y normalizando la línea de tiempo a la zona horaria de referencia (America/New_York), generando una serie continua con y sin ajuste por rollover.

—
## 2. RECURSOS DE ORIGEN
Ruta: /home/quant/data/raw/databento/GC/
Archivo: glbx-mdp3-20100606-20260204.ohlcv-1m.csv
Formato: ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol


## 3. REQUERIMIENTOS DE CARGA Y FILTRADO ESTRICTO

### REQ-01: Exclusión de Instrumentos Sintéticos (Spreads)

El archivo contiene instrumentos como GCN0-GCQ0 con precios negativos.

**Regla:** Eliminar cualquier fila donde el campo `symbol` contenga un guion (`-`). Solo se permiten contratos planos (ej. GCQ0, GCZ0).

**Nota de Depuración:** El reporte indica un mínimo de -50874.4. Esto sugiere que el filtro por el caracter `-` no se aplicó correctamente o existen símbolos con nomenclaturas alternativas. Se debe verificar si hay símbolos que comiencen con caracteres especiales o espacios.

---

### REQ-02: Filtro de "Major Months" (Liquidez Institucional)

Para evitar los "Serial Months" que causan distorsiones en el volumen y gaps de precios:

**Regla:** Solo permitir contratos cuyos códigos de mes sean:  
G (Feb), J (Apr), M (Jun), Q (Aug), V (Oct), Z (Dic).

**Ejemplo de Regex:**  
^GC[GJMQUVZ][0-9]$

---

### REQ-03: Lógica de Contrato Líder por Volumen (Daily Rollover)

Para construir la serie continua:

- Agrupar por fecha (Date) y symbol.
- Calcular el Volume total diario por contrato.
- El contrato líder del día es aquel con el volumen máximo.

**Validación:**  
El contrato seleccionado debe ser el mismo para todas las velas de 1 minuto de ese día.

---

## 4. NORMALIZACIÓN TEMPORAL Y TIPADO

### REQ-04: Conversión UTC a EST/EDT

- Convertir ts_event a objeto datetime (UTC).
- Localizar en UTC y convertir a America/New_York.
- Manejo de DST: Asegurar que el ajuste de horario de verano se aplique correctamente para evitar saltos de 1 hora en los indicadores.
- Definir el índice como timestamp.

---

### REQ-05: Tipado de Datos

- Open, High, Low, Close: float64  
- BAdjOpen, BAdjHigh, BAdjLow, BAdjClose: float64  
- Volume: int64  
- Ticker: string/category  

—

4. NORMALIZACIÓN TEMPORAL Y SALIDA
REQ-04: Gestión de Doble Horario
El archivo de salida debe conservar la referencia original y la local:
Timestamp_GMT: Hora original del evento (UTC/ISO8601).
index (Timestamp_NY): Hora convertida a America/New_York (Naive para compatibilidad con el motor de backtest).
REQ-05: Estructura de Columnas de Salida (MANDATORIO)
Columna
Tipo
Descripción
Timestamp_NY
index (datetime64)
Hora local Nueva York (EST/EDT).
Timestamp_GMT
datetime64
Hora original UTC sin zona horaria.
Open, High, Low, Close
float64
Precios crudos del contrato líder.
Volume
int64
Volumen total de la vela.
Ticker
string
Símbolo del contrato líder (ej. GCJ5).
Contract_ID
int64
instrument_id original de Databento.
Adj_Factor
float64
Factor acumulado de ajuste (Backward adjustment).
Open_adj
float64
Open + Adj_Factor
High_adj
float64
High + Adj_Factor
Low_adj
float64
Low + Adj_Factor
Close_adj
float64
Close + Adj_Factor




## 4. PROTOCOLO DE BACK ADJUSTMENT (ROLLOVER)

### REQ-06: Ajuste Acumulado de Precios (Back Adjustment)

Para eliminar discontinuidades artificiales generadas por el rollover entre contratos:

En cada punto de rollover:

delta = Close_new_contract_at_rollover - Close_old_contract_at_rollover

Este delta debe ser acumulado y aplicado hacia atrás en el tiempo.

Para cada barra histórica anterior al rollover:

BAdjPrice = RawPrice + cumulative_delta

Esto debe aplicarse de forma consistente a:

Open, High, Low, Close

generando:

BAdjOpen, BAdjHigh, BAdjLow, BAdjClose

Objetivo:  
Eliminar gaps artificiales sin distorsionar los retornos intradía ni la estructura del mercado.

---

## 5. PROTOCOLO DE SALIDA Y VALIDACIÓN PREVIA (REVISADO)

### A. Validación de Precios Positivos (CRÍTICO)

Prueba:  
df['Close'].min() > 0

ESTADO ACTUAL: FAILED (Min: -50874.4)

---

### B. Validación de Unicidad

Prueba:  
df.index.is_unique == True

ESTADO ACTUAL: PASSED

---

### C. Prueba de Zero-Gap (Pre-Back Adjustment)

Prueba:  
Evaluar el salto de precio en los puntos de rollover sobre Close

ESTADO ACTUAL: FAILED (Max Diff: 850.2)

---

### D. Prueba de Zero-Gap (Post-Back Adjustment)

Prueba:  
Evaluar el salto de precio en los puntos de rollover sobre BAdjClose

Criterio:  
abs(diff) < tick_size * 2

—



## 6. ENTREGABLE


Formato: Parquet con compresión Snappy  

Dataset único con doble capa de precios:

Ruta:  
/home/quant/data/processed/gc_1m_raw_continuous.parquet  


# ESPECIFICACIÓN TÉCNICA: PROCESAMIENTO RAW CSV A PARQUET CONTINUO (ORO - GC)

**PARA:** Equipo de Ingeniería de Datos
**DE:** Estratega Cuantitativo Senior
**ASUNTO:** Protocolo de Carga, Limpieza y Conversión Horaria para Backtesting Intradía

## 1. OBJETIVO

Transformar el archivo crudo de Databento en un DataFrame de alta fidelidad, filtrando instrumentos no deseados y normalizando la línea de tiempo a la zona horaria de referencia (`America/New_York`), generando una serie continua con ajuste por rollover.

## 2. RECURSOS DE ORIGEN

- **Ruta:** `/home/quant/data/raw/databento/GC/`
- **Archivo:** `glbx-mdp3-20100606-20260204.ohlcv-1m.csv`
- **Formato:** `ts_event,rtype,publisher_id,instrument_id,open,high,low,close,volume,symbol`

## 3. REQUERIMIENTOS DE FILTRADO Y CARGA

### REQ-01: Exclusión de Instrumentos Sintéticos (Spreads)

- **Regla:** Eliminar cualquier fila donde el campo `symbol` con tenga un guion ().
- **Objetivo:** Eliminar precios negativos y distorsiones de arbitraje.

### REQ-02: Filtro de "Major Months" (Liquidez Institucional)

- **Regla:** Solo permitir contratos cuyos códigos de mes sean: **G** (Feb), **J** (Apr), **M** (Jun), **Q** (Aug), **V** (Oct), **Z** (Dic).
- **Regex:** `^GC[GJMQUVZ][0-9]$`

### REQ-03: Lógica de Contrato Líder por Volumen (Daily Rollover)

1. Agrupar por fecha y `symbol`.
2. Calcular el `Volume` total diario por contrato.
3. El contrato líder del día es aquel con el volumen máximo.

## 4. NORMALIZACIÓN TEMPORAL Y SALIDA

### REQ-04: Gestión de Doble Horario

El archivo de salida debe conservar la referencia original y la local:

- **Timestamp_GMT:** Hora original del evento (UTC/ISO8601).
- **index (Timestamp_NY):** Hora convertida a `America/New_York`. Debe ser **Naive** (sin zona horaria explícita en el objeto) para compatibilidad con el simulador, pero reflejando la hora local de NY.

### REQ-05: Estructura de Columnas de Salida (MANDATORIO)

| Columna | Tipo | Descripción |
| --- | --- | --- |
| **Timestamp_NY** | index (datetime64) | Hora local Nueva York (EST/EDT). |
| **Timestamp_GMT** | datetime64 | Hora original UTC sin zona horaria. |
| **NY_Offset** | int32 | Desfase respecto a UTC en esa vela (ej. -5 para EST, -4 para EDT). **CRÍTICO PARA VALIDACIÓN**. |
| **Open, High, Low, Close** | float64 | Precios **crudos** del contrato líder. |
| **Volume** | int64 | Volumen total de la vela. |
| **Ticker** | string | Símbolo del contrato líder (ej. GCJ5). |
| **Contract_ID** | int64 | `instrument_id` original de Databento. |
| **Adj_Factor** | float64 | Factor acumulado de ajuste (Backward adjustment). |
| **Open_adj, High_adj, Low_adj, Close_adj** | float64 | Precios ajustados por rollover para indicadores de tendencia. |

## 5. PROTOCOLO DE BACK ADJUSTMENT (CALIDAD DE PRECIO)

- **Cálculo del Delta:** Al ocurrir un cambio de contrato líder: `delta = Close_new - Close_old`.
- **Acumulación:** Se genera un factor acumulado (`Adj_Factor`) aplicado a precios anteriores al rollover para evitar gaps artificiales que corrompan los indicadores en la capa de estrategia.

## 6. ENTREGABLE

- **Ruta:** `/home/quant/data/processed/gc_1m_raw_continuous.parquet`
- **Compresión:** Snappy.

In [8]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    # Filtros base: No Spreads y Solo Major Months
    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    
    ny_tz = pytz.timezone('America/New_York')
    
    def get_ny_data(ts):
        localized = ts.astimezone(ny_tz)
        offset = localized.utcoffset().total_seconds() / 3600
        return localized.replace(tzinfo=None), int(offset)

    ny_results = df['ts_event'].apply(get_ny_data)
    df['Timestamp_NY'] = ny_results.apply(lambda x: x[0])
    df['NY_Offset'] = ny_results.apply(lambda x: x[1]).astype('int32')
    
    df.set_index('Timestamp_NY', inplace=True)
    df.sort_index(inplace=True)

    # 3. LÓGICA DE CONTRATO LÍDER (REQ-03)
    print("🏆 Identificando contrato líder...")
    df['date_only'] = df.index.date
    daily_vol = df.groupby(['date_only', 'symbol'])['volume'].sum().reset_index()
    leader_symbols = daily_vol.loc[daily_vol.groupby('date_only')['volume'].idxmax()]
    leader_map = leader_symbols.set_index('date_only')['symbol'].to_dict()
    
    df['is_leader'] = df['symbol'] == df['date_only'].map(leader_map)
    df_continuous = df[df['is_leader']].copy()
    df_continuous.drop(columns=['date_only', 'is_leader', 'ts_event'], inplace=True)

    # 4. BACK ADJUSTMENT (REQ-06) - LÓGICA ROBUSTA
    print("🔧 Calculando Back Adjustment...")
    df_continuous.sort_index(inplace=True)
    
    # Identificar puntos de cambio de contrato
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    
    # DataFrame de transiciones
    transitions = df_continuous[df_continuous['ticker_change']].copy()
    transitions['next_close'] = df_continuous['close'].shift(-1).loc[transitions.index]
    
    # El delta es la diferencia entre el cierre del contrato que entra (n+1) 
    # y el cierre del contrato que sale (n) en el momento del rollover.
    # Usamos el índice para mapear el precio de la primera vela del nuevo contrato.
    
    tickers = df_continuous['symbol'].unique().tolist()
    adj_map = {tickers[-1]: 0.0}
    cum_delta = 0.0
    
    for i in range(len(tickers) - 2, -1, -1):
        old_t = tickers[i]
        new_t = tickers[i+1]
        
        # Último precio del contrato viejo en la serie continua
        last_p_old = df_continuous[df_continuous['symbol'] == old_t]['close'].iloc[-1]
        # Primer precio del contrato nuevo en la serie continua
        first_p_new = df_continuous[df_continuous['symbol'] == new_t]['close'].iloc[0]
        
        diff = first_p_new - last_p_old
        cum_delta += diff
        adj_map[old_t] = cum_delta

    df_continuous['Adj_Factor'] = df_continuous['symbol'].map(adj_map)

    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN
    print("✅ Validando...")
    # Prueba de fuego: ¿Es el gap menor a un tick?
    ticker_switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    ticker_switches.iloc[-1] = False
    
    gaps = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = gaps[ticker_switches].max()
    
    print(f"   - Max Adj Gap: {max_gap}")
    print(f"   - Estado: {'PASSED' if max_gap < 0.5 else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 18:43:48.979842
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Identificando contrato líder...
🔧 Calculando Back Adjustment...
📊 Finalizando estructura...
✅ Validando...
   - Max Adj Gap: 29883.199999999997
   - Estado: FAILED


In [9]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    
    ny_tz = pytz.timezone('America/New_York')
    
    def get_ny_data(ts):
        localized = ts.astimezone(ny_tz)
        offset = localized.utcoffset().total_seconds() / 3600
        return localized.replace(tzinfo=None), int(offset)

    ny_results = df['ts_event'].apply(get_ny_data)
    df['Timestamp_NY'] = ny_results.apply(lambda x: x[0])
    df['NY_Offset'] = ny_results.apply(lambda x: x[1]).astype('int32')
    
    df.set_index('Timestamp_NY', inplace=True)
    df.sort_index(inplace=True)

    # 3. LÓGICA DE CONTRATO LÍDER
    print("🏆 Identificando contrato líder...")
    df['date_only'] = df.index.date
    daily_vol = df.groupby(['date_only', 'symbol'])['volume'].sum().reset_index()
    leader_symbols = daily_vol.loc[daily_vol.groupby('date_only')['volume'].idxmax()]
    leader_map = leader_symbols.set_index('date_only')['symbol'].to_dict()
    
    df['is_leader'] = df['symbol'] == df['date_only'].map(leader_map)
    df_continuous = df[df['is_leader']].copy()
    df_continuous.drop(columns=['date_only', 'is_leader', 'ts_event'], inplace=True)

    # 4. BACK ADJUSTMENT (ZERO-GAP PRECISION)
    print("🔧 Calculando Back Adjustment (Sin Holgura)...")
    df_continuous.sort_index(inplace=True)
    
    # Identificamos los puntos de transición exactos
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    
    # Lista cronológica de tickers
    ordered_tickers = []
    last_t = None
    for t in df_continuous['symbol']:
        if t != last_t:
            ordered_tickers.append(t)
            last_t = t

    adj_map = {ordered_tickers[-1]: 0.0}
    cumulative_delta = 0.0
    
    # Recorrido inverso: de contrato nuevo a contrato viejo
    for i in range(len(ordered_tickers) - 1, 0, -1):
        new_t = ordered_tickers[i]
        old_t = ordered_tickers[i-1]
        
        # Localizamos el último registro del contrato viejo
        # y el primer registro del contrato nuevo
        last_row_old = df_continuous[df_continuous['symbol'] == old_t].iloc[-1]
        first_row_new = df_continuous[df_continuous['symbol'] == new_t].iloc[0]
        
        # El GAP que queremos eliminar es la diferencia entre precios de cierre
        # en el salto de la serie continua.
        gap_nominal = first_row_new['close'] - last_row_old['close']
        
        # Acumulamos el ajuste hacia el pasado
        cumulative_delta += gap_nominal
        adj_map[old_t] = cumulative_delta

    # Mapeo del factor de ajuste
    df_continuous['Adj_Factor'] = df_continuous['symbol'].map(adj_map)

    # Aplicación a OHLC
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN ZERO-GAP ESTRICTA
    print("✅ Validando Zero-Gap...")
    # Buscamos los registros donde hay cambio de ticker
    switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    switches.iloc[-1] = False # Ignorar el final del dataset
    
    # Calculamos el gap ajustado: Close_adj(t) vs Close_adj(t+1)
    # Debe ser 0.0000... o muy cercano por precisión de float
    gaps_ajustados = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = gaps_ajustados[switches].max()
    
    print(f"   - Max Adj Gap en Rollover: {max_gap}")
    # Tolerancia ínfima solo por redondeo de punto flotante (1e-10)
    print(f"   - Estado Zero-Gap: {'PASSED' if max_gap < 1e-8 else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 18:53:32.054510
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Identificando contrato líder...
🔧 Calculando Back Adjustment (Sin Holgura)...
📊 Finalizando estructura...
✅ Validando Zero-Gap...
   - Max Adj Gap en Rollover: 29883.199999999993
   - Estado Zero-Gap: FAILED


In [10]:
df_final.head()


,Timestamp_GMT,NY_Offset,Open,High,Low,Close,Volume,Ticker,Contract_ID,Adj_Factor,Open_adj,High_adj,Low_adj,Close_adj
Timestamp_NY,,,,,,,,,,,,,,
2010-06-06 18:00:00,2010-06-06 22:00:00,-4,1221.1,1221.8,1221.1,1221.6,74,GCQ0,104313,-56589.8,-55368.7,-55368.0,-55368.7,-55368.2
2010-06-06 18:01:00,2010-06-06 22:01:00,-4,1221.6,1221.9,1221.6,1221.9,8,GCQ0,104313,-56589.8,-55368.2,-55367.9,-55368.2,-55367.9
2010-06-06 18:02:00,2010-06-06 22:02:00,-4,1221.9,1221.9,1221.6,1221.8,41,GCQ0,104313,-56589.8,-55367.9,-55367.9,-55368.2,-55368.0
2010-06-06 18:03:00,2010-06-06 22:03:00,-4,1221.6,1221.6,1220.5,1221.5,70,GCQ0,104313,-56589.8,-55368.2,-55368.2,-55369.3,-55368.3
2010-06-06 18:04:00,2010-06-06 22:04:00,-4,1221.0,1221.0,1220.9,1220.9,4,GCQ0,104313,-56589.8,-55368.8,-55368.8,-55368.9,-55368.9


In [11]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    
    ny_tz = pytz.timezone('America/New_York')
    
    def get_ny_data(ts):
        localized = ts.astimezone(ny_tz)
        offset = localized.utcoffset().total_seconds() / 3600
        return localized.replace(tzinfo=None), int(offset)

    ny_results = df['ts_event'].apply(get_ny_data)
    df['Timestamp_NY'] = ny_results.apply(lambda x: x[0])
    df['NY_Offset'] = ny_results.apply(lambda x: x[1]).astype('int32')
    
    df.set_index('Timestamp_NY', inplace=True)
    df.sort_index(inplace=True)

    # 3. LÓGICA DE CONTRATO LÍDER
    print("🏆 Identificando contrato líder...")
    df['date_only'] = df.index.date
    daily_vol = df.groupby(['date_only', 'symbol'])['volume'].sum().reset_index()
    leader_symbols = daily_vol.loc[daily_vol.groupby('date_only')['volume'].idxmax()]
    leader_map = leader_symbols.set_index('date_only')['symbol'].to_dict()
    
    df['is_leader'] = df['symbol'] == df['date_only'].map(leader_map)
    df_continuous = df[df['is_leader']].copy()
    df_continuous.drop(columns=['date_only', 'is_leader', 'ts_event'], inplace=True)

    # 4. BACK ADJUSTMENT (ZERO-GAP PRECISION)
    print("🔧 Calculando Back Adjustment (Zero-Gap Estricto)...")
    df_continuous.sort_index(inplace=True)
    
    # Identificar cambios de Ticker y sus posiciones
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    change_indices = df_continuous.index[df_continuous['ticker_change']].tolist()
    
    # El ajuste debe ser constante por contrato. 
    # Tickers en orden cronológico inverso para acumular el delta hacia atrás.
    ordered_tickers = []
    last_t = None
    for t in df_continuous['symbol']:
        if t != last_t:
            ordered_tickers.append(t)
            last_t = t

    adj_map = {ordered_tickers[-1]: 0.0}
    cumulative_delta = 0.0
    
    # Iteramos sobre los puntos de cambio encontrados en la serie líder
    for i in range(len(change_indices)):
        idx_last_old = change_indices[-(i+1)]
        # El siguiente registro en la serie continua es el primero del nuevo contrato
        pos_old = df_continuous.index.get_loc(idx_last_old)
        idx_first_new = df_continuous.index[pos_old + 1]
        
        ticker_old = df_continuous.loc[idx_last_old, 'symbol']
        ticker_new = df_continuous.loc[idx_first_new, 'symbol']
        
        # Gap = Cierre del nuevo (t+1) - Cierre del viejo (t)
        gap = df_continuous.loc[idx_first_new, 'close'] - df_continuous.loc[idx_last_old, 'close']
        
        cumulative_delta += gap
        adj_map[ticker_old] = cumulative_delta

    # Mapeo del factor de ajuste a la serie
    df_continuous['Adj_Factor'] = df_continuous['symbol'].map(adj_map)

    # Aplicación a OHLC
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN ZERO-GAP ABSOLUTA
    print("✅ Validando Zero-Gap...")
    # Registros de cambio
    switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    switches.iloc[-1] = False 
    
    # En el punto de rollover: Close_adj(t) debe ser igual a Close_adj(t+1) - (salto natural del mercado)
    # Pero el Back Adjustment busca que Close_adj(t) == Close_adj(t+1) si el ajuste es perfecto.
    gaps_ajustados = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = gaps_ajustados[switches].max()
    
    print(f"   - Max Adj Gap en Rollover: {max_gap}")
    # Tolerancia por precisión de float
    print(f"   - Estado Zero-Gap: {'PASSED' if max_gap < 1e-10 else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 19:03:08.861480
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Identificando contrato líder...
🔧 Calculando Back Adjustment (Zero-Gap Estricto)...


IndexError: index 5497016 is out of bounds for axis 0 with size 5497016

In [ ]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    
    ny_tz = pytz.timezone('America/New_York')
    
    def get_ny_data(ts):
        localized = ts.astimezone(ny_tz)
        offset = localized.utcoffset().total_seconds() / 3600
        return localized.replace(tzinfo=None), int(offset)

    ny_results = df['ts_event'].apply(get_ny_data)
    df['Timestamp_NY'] = ny_results.apply(lambda x: x[0])
    df['NY_Offset'] = ny_results.apply(lambda x: x[1]).astype('int32')
    
    df.sort_values(['Timestamp_NY', 'symbol'], inplace=True)

    # 3. LÓGICA DE CONTRATO LÍDER (FILTRADO ESTRICTO)
    print("🏆 Identificando y filtrando por contrato líder...")
    df['date_only'] = df['Timestamp_NY'].dt.date
    
    # Calculamos volumen total por día y símbolo
    daily_vol = df.groupby(['date_only', 'symbol'])['volume'].sum().reset_index()
    
    # Encontramos el símbolo con el máximo volumen para cada día
    idx_max_vol = daily_vol.groupby('date_only')['volume'].idxmax()
    leader_map = daily_vol.loc[idx_max_vol].set_index('date_only')['symbol'].to_dict()
    
    # FILTRO CRÍTICO: Solo conservamos las filas donde el símbolo coincide con el líder del día
    df['is_leader'] = df.apply(lambda row: row['symbol'] == leader_map.get(row['date_only']), axis=1)
    df_continuous = df[df['is_leader']].copy()
    
    # Limpieza y ordenación final de la serie líder
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)
    df_continuous.drop(columns=['date_only', 'is_leader', 'ts_event'], inplace=True)

    # 4. BACK ADJUSTMENT (ZERO-GAP PRECISION)
    print("🔧 Calculando Back Adjustment...")
    
    # Identificar cambios de Ticker
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    change_positions = np.where(df_continuous['ticker_change'])[0]
    
    # Excluir la última fila si aparece como cambio
    if len(change_positions) > 0 and change_positions[-1] == len(df_continuous) - 1:
        change_positions = change_positions[:-1]

    # Lista de tickers únicos
    ordered_tickers = []
    last_t = None
    for t in df_continuous['symbol']:
        if t != last_t:
            ordered_tickers.append(t)
            last_t = t

    adj_map = {ordered_tickers[-1]: 0.0}
    cumulative_delta = 0.0
    
    # Ajuste de atrás hacia adelante
    for pos_old in reversed(change_positions):
        pos_new = pos_old + 1
        ticker_old = df_continuous.iloc[pos_old]['symbol']
        
        # Gap = Cierre del nuevo contrato en t+1 - Cierre del viejo en t
        gap = df_continuous.iloc[pos_new]['close'] - df_continuous.iloc[pos_old]['close']
        
        cumulative_delta += gap
        adj_map[ticker_old] = cumulative_delta

    df_continuous['Adj_Factor'] = df_continuous['symbol'].map(adj_map)

    # Aplicar ajuste
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'Timestamp_NY', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN ZERO-GAP
    print("✅ Validando resultados...")
    switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    switches.iloc[-1] = False 
    
    gaps_ajustados = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = gaps_ajustados[switches].max()
    
    print(f"   - Max Adj Gap en Rollover: {max_gap}")
    is_zero_gap = max_gap < 1e-10
    print(f"   - Estado Zero-Gap: {'PASSED' if is_zero_gap else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Archivo guardado: {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 19:12:52.284711
📦 Cargando y filtrando instrumentos...


In [1]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    
    ny_tz = pytz.timezone('America/New_York')
    
    def get_ny_data(ts):
        localized = ts.astimezone(ny_tz)
        offset = localized.utcoffset().total_seconds() / 3600
        return localized.replace(tzinfo=None), int(offset)

    ny_results = df['ts_event'].apply(get_ny_data)
    df['Timestamp_NY'] = ny_results.apply(lambda x: x[0])
    df['NY_Offset'] = ny_results.apply(lambda x: x[1]).astype('int32')
    
    df.sort_values(['Timestamp_NY', 'symbol'], inplace=True)

    # 3. LÓGICA DE CONTRATO LÍDER (FILTRADO ESTRICTO)
    print("🏆 Identificando y filtrando por contrato líder...")
    df['date_only'] = df['Timestamp_NY'].dt.date
    
    # Calculamos volumen total por día y símbolo
    daily_vol = df.groupby(['date_only', 'symbol'])['volume'].sum().reset_index()
    
    # Encontramos el símbolo con el máximo volumen para cada día
    idx_max_vol = daily_vol.groupby('date_only')['volume'].idxmax()
    leader_map = daily_vol.loc[idx_max_vol].set_index('date_only')['symbol'].to_dict()
    
    # FILTRO CRÍTICO: Solo conservamos las filas donde el símbolo coincide con el líder del día
    df['is_leader'] = df.apply(lambda row: row['symbol'] == leader_map.get(row['date_only']), axis=1)
    df_continuous = df[df['is_leader']].copy()
    
    # Limpieza y ordenación final de la serie líder
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)
    df_continuous.drop(columns=['date_only', 'is_leader', 'ts_event'], inplace=True)

    # 4. BACK ADJUSTMENT (ZERO-GAP PRECISION)
    print("🔧 Calculando Back Adjustment...")
    
    # Identificar cambios de Ticker
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    change_positions = np.where(df_continuous['ticker_change'])[0]
    
    # Excluir la última fila si aparece como cambio
    if len(change_positions) > 0 and change_positions[-1] == len(df_continuous) - 1:
        change_positions = change_positions[:-1]

    # Lista de tickers únicos
    ordered_tickers = []
    last_t = None
    for t in df_continuous['symbol']:
        if t != last_t:
            ordered_tickers.append(t)
            last_t = t

    adj_map = {ordered_tickers[-1]: 0.0}
    cumulative_delta = 0.0
    
    # Ajuste de atrás hacia adelante
    for pos_old in reversed(change_positions):
        pos_new = pos_old + 1
        ticker_old = df_continuous.iloc[pos_old]['symbol']
        
        # Gap = Cierre del nuevo contrato en t+1 - Cierre del viejo en t
        gap = df_continuous.iloc[pos_new]['close'] - df_continuous.iloc[pos_old]['close']
        
        cumulative_delta += gap
        adj_map[ticker_old] = cumulative_delta

    df_continuous['Adj_Factor'] = df_continuous['symbol'].map(adj_map)

    # Aplicar ajuste
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'Timestamp_NY', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN ZERO-GAP
    print("✅ Validando resultados...")
    switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    switches.iloc[-1] = False 
    
    gaps_ajustados = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = gaps_ajustados[switches].max()
    
    print(f"   - Max Adj Gap en Rollover: {max_gap}")
    is_zero_gap = max_gap < 1e-10
    print(f"   - Estado Zero-Gap: {'PASSED' if is_zero_gap else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Archivo guardado: {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 19:29:31.775658
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Identificando y filtrando por contrato líder...
🔧 Calculando Back Adjustment...
📊 Finalizando estructura...
✅ Validando resultados...
   - Max Adj Gap en Rollover: 180.00000000000045
   - Estado Zero-Gap: FAILED
💾 Archivo guardado: /home/quant/data/processed/gc_1m_raw_continuous.parquet


In [2]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime, timedelta
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL (GMT -> NY)
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    
    ny_tz = pytz.timezone('America/New_York')
    
    def get_ny_data(ts):
        localized = ts.astimezone(ny_tz)
        offset = localized.utcoffset().total_seconds() / 3600
        return localized.replace(tzinfo=None), int(offset)

    ny_results = df['ts_event'].apply(get_ny_data)
    df['Timestamp_NY'] = ny_results.apply(lambda x: x[0])
    df['NY_Offset'] = ny_results.apply(lambda x: x[1]).astype('int32')
    
    # 3. LÓGICA DE CME BUSINESS DAY (REQ-ROLLOVER-17:00)
    # Definimos el "Día de Negocio": Si es >= 17:00 NY, pertenece al día siguiente.
    print("🏆 Calculando CME Business Day (Rollover 17:00 NY)...")
    df['Business_Date'] = (df['Timestamp_NY'] + pd.Timedelta(hours=7)).dt.date
    
    # Volumen por Business Day y símbolo
    daily_vol = df.groupby(['Business_Date', 'symbol'])['volume'].sum().reset_index()
    
    # Identificar líder por Business Day
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    leader_map = daily_vol.loc[idx_max_vol].set_index('Business_Date')['symbol'].to_dict()
    
    # Filtro estricto: solo el contrato líder de esa sesión
    df['is_leader'] = df['symbol'] == df['Business_Date'].map(leader_map)
    df_continuous = df[df['is_leader']].copy()
    
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)

    # 4. BACK ADJUSTMENT (ZERO-GAP PRECISION)
    print("🔧 Calculando Back Adjustment en ventanas de cierre...")
    
    # Identificar cambios de Ticker
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    change_positions = np.where(df_continuous['ticker_change'])[0]
    
    if len(change_positions) > 0 and change_positions[-1] == len(df_continuous) - 1:
        change_positions = change_positions[:-1]

    ordered_tickers = []
    last_t = None
    for t in df_continuous['symbol']:
        if t != last_t:
            ordered_tickers.append(t)
            last_t = t

    adj_map = {ordered_tickers[-1]: 0.0}
    cumulative_delta = 0.0
    
    # Recorrido inverso
    for pos_old in reversed(change_positions):
        pos_new = pos_old + 1
        
        # El gap se mide entre la última vela del líder actual y la primera del siguiente
        # Debido al Business Day, esto ocurrirá cerca de las 17:00 NY
        price_old = df_continuous.iloc[pos_old]['close']
        price_new = df_continuous.iloc[pos_new]['close']
        
        ticker_old = df_continuous.iloc[pos_old]['symbol']
        gap = price_new - price_old
        
        cumulative_delta += gap
        adj_map[ticker_old] = cumulative_delta

    df_continuous['Adj_Factor'] = df_continuous['symbol'].map(adj_map)

    # Aplicar ajuste
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'Timestamp_NY', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN ZERO-GAP
    print("✅ Validando resultados...")
    switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    switches.iloc[-1] = False 
    
    gaps_ajustados = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = gaps_ajustados[switches].max()
    
    print(f"   - Max Adj Gap en Rollover: {max_gap}")
    is_zero_gap = max_gap < 1e-8
    print(f"   - Estado Zero-Gap: {'PASSED' if is_zero_gap else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Archivo guardado con rollover a las 17:00 NY: {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 19:38:54.515605
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Calculando CME Business Day (Rollover 17:00 NY)...
🔧 Calculando Back Adjustment en ventanas de cierre...
📊 Finalizando estructura...
✅ Validando resultados...
   - Max Adj Gap en Rollover: 189.89999999999918
   - Estado Zero-Gap: FAILED
💾 Archivo guardado con rollover a las 17:00 NY: /home/quant/data/processed/gc_1m_raw_continuous.parquet


In [3]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime, timedelta
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL (GMT -> NY)
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    
    ny_tz = pytz.timezone('America/New_York')
    
    def get_ny_data(ts):
        localized = ts.astimezone(ny_tz)
        offset = localized.utcoffset().total_seconds() / 3600
        return localized.replace(tzinfo=None), int(offset)

    ny_results = df['ts_event'].apply(get_ny_data)
    df['Timestamp_NY'] = ny_results.apply(lambda x: x[0])
    df['NY_Offset'] = ny_results.apply(lambda x: x[1]).astype('int32')
    
    # 3. LÓGICA DE CME BUSINESS DAY (REQ-ROLLOVER-17:00)
    # Definimos el "Día de Negocio": Si es >= 17:00 NY, pertenece al día siguiente.
    print("🏆 Calculando CME Business Day (Rollover 17:00 NY)...")
    df['Business_Date'] = (df['Timestamp_NY'] + pd.Timedelta(hours=7)).dt.date
    
    # Volumen por Business Day y símbolo
    daily_vol = df.groupby(['Business_Date', 'symbol'])['volume'].sum().reset_index()
    
    # Identificar líder por Business Day
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    leader_map = daily_vol.loc[idx_max_vol].set_index('Business_Date')['symbol'].to_dict()
    
    # Filtro estricto: solo el contrato líder de esa sesión
    df['is_leader'] = df['symbol'] == df['Business_Date'].map(leader_map)
    df_continuous = df[df['is_leader']].copy()
    
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)

    # 4. BACK ADJUSTMENT (ZERO-GAP PRECISION)
    print("🔧 Calculando Back Adjustment en ventanas de cierre...")
    
    # Identificar cambios de Ticker
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    change_positions = np.where(df_continuous['ticker_change'])[0]
    
    if len(change_positions) > 0 and change_positions[-1] == len(df_continuous) - 1:
        change_positions = change_positions[:-1]

    ordered_tickers = []
    last_t = None
    for t in df_continuous['symbol']:
        if t != last_t:
            ordered_tickers.append(t)
            last_t = t

    adj_map = {ordered_tickers[-1]: 0.0}
    cumulative_delta = 0.0
    
    # Recorrido inverso
    for pos_old in reversed(change_positions):
        pos_new = pos_old + 1
        
        # El gap se mide entre la última vela del líder actual y la primera del siguiente
        # Debido al Business Day, esto ocurrirá cerca de las 17:00 NY
        price_old = df_continuous.iloc[pos_old]['close']
        price_new = df_continuous.iloc[pos_new]['close']
        
        ticker_old = df_continuous.iloc[pos_old]['symbol']
        gap = price_new - price_old
        
        cumulative_delta += gap
        adj_map[ticker_old] = cumulative_delta

    df_continuous['Adj_Factor'] = df_continuous['symbol'].map(adj_map)

    # Aplicar ajuste
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'Timestamp_NY', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN ZERO-GAP
    print("✅ Validando resultados...")
    switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    switches.iloc[-1] = False 
    
    gaps_ajustados = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = gaps_ajustados[switches].max()
    
    print(f"   - Max Adj Gap en Rollover: {max_gap}")
    is_zero_gap = max_gap < 1e-8
    print(f"   - Estado Zero-Gap: {'PASSED' if is_zero_gap else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Archivo guardado con rollover a las 17:00 NY: {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 19:42:00.951627
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Calculando CME Business Day (Rollover 17:00 NY)...
🔧 Calculando Back Adjustment en ventanas de cierre...
📊 Finalizando estructura...
✅ Validando resultados...
   - Max Adj Gap en Rollover: 189.89999999999918
   - Estado Zero-Gap: FAILED
💾 Archivo guardado con rollover a las 17:00 NY: /home/quant/data/processed/gc_1m_raw_continuous.parquet


In [4]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL (GMT -> NY)
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    ny_tz = pytz.timezone('America/New_York')
    
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    df['Timestamp_NY_Raw'] = df['ts_event'].dt.tz_convert(ny_tz)
    df['Timestamp_NY'] = df['Timestamp_NY_Raw'].dt.tz_localize(None)
    df['NY_Offset'] = df['Timestamp_NY_Raw'].dt.utcoffset().dt.total_seconds() / 3600
    df['NY_Offset'] = df['NY_Offset'].astype('int32')
    
    df.sort_values(['Timestamp_NY', 'symbol'], inplace=True)

    # 3. LÓGICA DE LÍDER PROGRESIVO (Forward-Only)
    print("🏆 Calculando Líder Progresivo (Sin retrocesos)...")
    df['Business_Date'] = (df['Timestamp_NY'] + pd.Timedelta(hours=7)).dt.date
    
    # Volumen por Business Day y símbolo
    daily_vol = df.groupby(['Business_Date', 'symbol'])['volume'].sum().reset_index()
    
    # Identificamos el líder "crudo" por día
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    raw_leaders = daily_vol.loc[idx_max_vol][['Business_Date', 'symbol']]
    
    # --- FILTRO DE AVANCE ESTRICTO ---
    # Queremos evitar que si hoy el líder es GCM24 y mañana vuelve a ser GCJ24, el código lo permita.
    # Definimos el orden de los contratos por su fecha de aparición como líder.
    
    unique_leaders_ordered = []
    seen = set()
    for s in raw_leaders['symbol']:
        if s not in seen:
            unique_leaders_ordered.append(s)
            seen.add(s)
    
    # Mapeamos cada Business Day al líder, pero solo si es igual o superior en la jerarquía al anterior
    leader_map = {}
    current_leader_idx = 0
    
    # Convertimos a lista de diccionarios para iterar
    raw_leader_list = raw_leaders.to_dict('records')
    
    for day_data in raw_leader_list:
        day = day_data['Business_Date']
        candidate = day_data['symbol']
        candidate_idx = unique_leaders_ordered.index(candidate)
        
        # Solo avanzamos, nunca retrocedemos en la jerarquía de contratos
        if candidate_idx >= current_leader_idx:
            current_leader_idx = candidate_idx
            
        leader_map[day] = unique_leaders_ordered[current_leader_idx]
    
    df['is_leader'] = df['symbol'] == df['Business_Date'].map(leader_map)
    df_continuous = df[df['is_leader']].copy()
    
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)

    # 4. BACK ADJUSTMENT (ZERO-GAP ABSOLUTO)
    print("🔧 Calculando Back Adjustment (Propagación Inversa)...")
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    
    change_indices = df_continuous.index[df_continuous['ticker_change'] == True].tolist()
    if len(change_indices) > 0 and change_indices[-1] == len(df_continuous) - 1:
        change_indices.pop()

    cumulative_adj = 0.0
    factors = np.zeros(len(df_continuous))
    
    for idx in reversed(change_indices):
        price_old = df_continuous.iloc[idx]['close']
        price_new = df_continuous.iloc[idx + 1]['close']
        
        gap = price_new - price_old
        cumulative_adj -= gap 
        factors[:idx + 1] = cumulative_adj

    df_continuous['Adj_Factor'] = factors

    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'Timestamp_NY', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN ZERO-GAP
    print("✅ Validando resultados...")
    switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    switches.iloc[-1] = False 
    
    diff_ajustada = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = diff_ajustada[switches].max() if any(switches) else 0.0
    
    print(f"   - Max Adj Gap en Rollover: {max_gap}")
    
    is_zero_gap = max_gap < 1e-9
    print(f"   - Estado Zero-Gap: {'PASSED' if is_zero_gap else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Archivo guardado (Líder Progresivo): {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 19:51:17.103184
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...


AttributeError: 'DatetimeProperties' object has no attribute 'utcoffset'

In [5]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    df = df[~df['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df = df[df['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL (GMT -> NY)
    print("⏰ Normalizando zonas horarias...")
    df['ts_event'] = pd.to_datetime(df['ts_event'], utc=True)
    ny_tz = pytz.timezone('America/New_York')
    
    df['Timestamp_GMT'] = df['ts_event'].dt.tz_localize(None)
    df['Timestamp_NY_Raw'] = df['ts_event'].dt.tz_convert(ny_tz)
    df['Timestamp_NY'] = df['Timestamp_NY_Raw'].dt.tz_localize(None)
    
    # Corrección del error: Extraer el offset de forma segura
    df['NY_Offset'] = df['Timestamp_NY_Raw'].apply(lambda x: x.utcoffset().total_seconds() / 3600).astype('int32')
    
    df.sort_values(['Timestamp_NY', 'symbol'], inplace=True)

    # 3. LÓGICA DE LÍDER PROGRESIVO (Forward-Only)
    print("🏆 Calculando Líder Progresivo (Sin retrocesos)...")
    df['Business_Date'] = (df['Timestamp_NY'] + pd.Timedelta(hours=7)).dt.date
    
    # Volumen por Business Day y símbolo
    daily_vol = df.groupby(['Business_Date', 'symbol'])['volume'].sum().reset_index()
    
    # Identificamos el líder "crudo" por día
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    raw_leaders = daily_vol.loc[idx_max_vol][['Business_Date', 'symbol']]
    
    # --- FILTRO DE AVANCE ESTRICTO ---
    unique_leaders_ordered = []
    seen = set()
    for s in raw_leaders['symbol']:
        if s not in seen:
            unique_leaders_ordered.append(s)
            seen.add(s)
    
    leader_map = {}
    current_leader_idx = 0
    raw_leader_list = raw_leaders.to_dict('records')
    
    for day_data in raw_leader_list:
        day = day_data['Business_Date']
        candidate = day_data['symbol']
        candidate_idx = unique_leaders_ordered.index(candidate)
        
        if candidate_idx >= current_leader_idx:
            current_leader_idx = candidate_idx
            
        leader_map[day] = unique_leaders_ordered[current_leader_idx]
    
    df['is_leader'] = df['symbol'] == df['Business_Date'].map(leader_map)
    df_continuous = df[df['is_leader']].copy()
    
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)

    # 4. BACK ADJUSTMENT (ZERO-GAP ABSOLUTO)
    print("🔧 Calculando Back Adjustment (Propagación Inversa)...")
    df_continuous['ticker_change'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    
    change_indices = df_continuous.index[df_continuous['ticker_change'] == True].tolist()
    if len(change_indices) > 0 and change_indices[-1] == len(df_continuous) - 1:
        change_indices.pop()

    cumulative_adj = 0.0
    factors = np.zeros(len(df_continuous))
    
    for idx in reversed(change_indices):
        price_old = df_continuous.iloc[idx]['close']
        price_new = df_continuous.iloc[idx + 1]['close']
        
        gap = price_new - price_old
        cumulative_adj -= gap 
        factors[:idx + 1] = cumulative_adj

    df_continuous['Adj_Factor'] = factors

    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = df_continuous[col] + df_continuous['Adj_Factor']

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_GMT', 'Timestamp_NY', 'NY_Offset', 
        'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN ZERO-GAP
    print("✅ Validando resultados...")
    switches = df_final['Ticker'] != df_final['Ticker'].shift(-1)
    switches.iloc[-1] = False 
    
    diff_ajustada = (df_final['Close_adj'].shift(-1) - df_final['Close_adj']).abs()
    max_gap = diff_ajustada[switches].max() if any(switches) else 0.0
    
    print(f"   - Max Adj Gap en Rollover: {max_gap}")
    
    is_zero_gap = max_gap < 1e-9
    print(f"   - Estado Zero-Gap: {'PASSED' if is_zero_gap else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Archivo guardado (Líder Progresivo): {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 19:53:45.920177
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Calculando Líder Progresivo (Sin retrocesos)...
🔧 Calculando Back Adjustment (Propagación Inversa)...
📊 Finalizando estructura...
✅ Validando resultados...
   - Max Adj Gap en Rollover: 34.80000000000018
   - Estado Zero-Gap: FAILED
💾 Archivo guardado (Líder Progresivo): /home/quant/data/processed/gc_1m_raw_continuous.parquet


In [6]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df_raw = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    # Filtrar solo contratos individuales (outrights), ignorando spreads como GCQ0-GCZ0
    df_raw = df_raw[~df_raw['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df_raw = df_raw[df_raw['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL (GMT -> NY)
    print("⏰ Normalizando zonas horarias...")
    df_raw['ts_event'] = pd.to_datetime(df_raw['ts_event'], utc=True)
    ny_tz = pytz.timezone('America/New_York')
    
    df_raw['Timestamp_NY_Raw'] = df_raw['ts_event'].dt.tz_convert(ny_tz)
    df_raw['Timestamp_NY'] = df_raw['Timestamp_NY_Raw'].dt.tz_localize(None)
    df_raw['NY_Offset'] = df_raw['Timestamp_NY_Raw'].apply(lambda x: x.utcoffset().total_seconds() / 3600).astype('int32')
    
    df_raw.sort_values(['Timestamp_NY', 'symbol'], inplace=True)

    # 3. LÓGICA DE LÍDER PROGRESIVO
    print("🏆 Calculando Líder Progresivo...")
    df_raw['Business_Date'] = (df_raw['Timestamp_NY'] + pd.Timedelta(hours=7)).dt.date
    
    daily_vol = df_raw.groupby(['Business_Date', 'symbol'])['volume'].sum().reset_index()
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    raw_leaders = daily_vol.loc[idx_max_vol][['Business_Date', 'symbol']]
    
    unique_leaders_ordered = []
    seen = set()
    for s in raw_leaders['symbol']:
        if s not in seen:
            unique_leaders_ordered.append(s)
            seen.add(s)
    
    leader_map = {}
    current_leader_idx = 0
    for day_data in raw_leaders.to_dict('records'):
        day, candidate = day_data['Business_Date'], day_data['symbol']
        candidate_idx = unique_leaders_ordered.index(candidate)
        if candidate_idx >= current_leader_idx:
            current_leader_idx = candidate_idx
        leader_map[day] = unique_leaders_ordered[current_leader_idx]
    
    df_raw['is_leader'] = df_raw['symbol'] == df_raw['Business_Date'].map(leader_map)
    df_continuous = df_raw[df_raw['is_leader']].copy()
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)

    # 4. BACK ADJUSTMENT Y METADATOS DE ROLLOVER
    print("🔧 Calculando Back Adjustment y Metadatos...")
    
    df_continuous['Is_Rollover_Event'] = df_continuous['symbol'] != df_continuous['symbol'].shift(-1)
    # Marcamos la primera vela del contrato nuevo como el evento de rollover
    df_continuous['Is_Rollover_Event'] = df_continuous['Is_Rollover_Event'].shift(1).fillna(False)
    
    change_indices = df_continuous.index[df_continuous['symbol'] != df_continuous['symbol'].shift(1)].tolist()
    # El primer índice es el inicio de la serie, no un rollover
    if len(change_indices) > 0: change_indices.pop(0)

    factors = np.zeros(len(df_continuous))
    tech_gaps = np.zeros(len(df_continuous))
    cumulative_delta = 0.0
    
    # Trabajamos de adelante hacia atrás para el ajuste acumulativo
    # Pero identificamos los gaps en los puntos de cambio
    all_changes = df_continuous.index[df_continuous['symbol'] != df_continuous['symbol'].shift(-1)].tolist()
    if len(all_changes) > 0 and all_changes[-1] == len(df_continuous) - 1: all_changes.pop()

    for idx in reversed(all_changes):
        ts_rollover = df_continuous.iloc[idx]['Timestamp_NY']
        ticker_old = df_continuous.iloc[idx]['symbol']
        ticker_new = df_continuous.iloc[idx+1]['symbol']
        
        snapshot = df_raw[df_raw['Timestamp_NY'] == ts_rollover]
        p_old = snapshot[snapshot['symbol'] == ticker_old]['close']
        p_new = snapshot[snapshot['symbol'] == ticker_new]['close']
        
        if not p_old.empty and not p_new.empty:
            gap = p_new.iloc[0] - p_old.iloc[0]
        else:
            gap = df_continuous.iloc[idx+1]['close'] - df_continuous.iloc[idx]['close']
            
        cumulative_delta -= gap
        factors[:idx + 1] = cumulative_delta
        tech_gaps[idx + 1] = gap # Guardamos el gap técnico en la primera vela del contrato nuevo

    df_continuous['Adj_Factor'] = factors
    df_continuous['Rollover_Gap_Technical'] = tech_gaps

    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = (df_continuous[col] + df_continuous['Adj_Factor']).round(2)

    # Calcular el Gap de Mercado Residual (el salto que queda tras ajustar el contrato)
    df_continuous['Market_Gap_Adjusted'] = (df_continuous['Open_adj'] - df_continuous['Close_adj'].shift(1)).fillna(0.0)
    # Solo nos interesa este dato en los momentos de rollover
    df_continuous.loc[~df_continuous['Is_Rollover_Event'], 'Market_Gap_Adjusted'] = 0.0

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_NY', 'NY_Offset', 'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor', 'Is_Rollover_Event',
        'Rollover_Gap_Technical', 'Market_Gap_Adjusted',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN
    print("✅ Validando resultados...")
    market_returns = df_final['Close_adj'].diff().abs()
    avg_vol = market_returns[df_final['Is_Rollover_Event'] == False].mean()
    std_vol = market_returns[df_final['Is_Rollover_Event'] == False].std()
    
    rollover_return = market_returns[df_final['Is_Rollover_Event'] == True].max()
    
    print(f"   - Salto máximo ajustado en rollover: {rollover_return:.4f}")
    is_consistent = rollover_return < (avg_vol + 15 * std_vol)
    print(f"   - Estado de Integridad: {'PASSED' if is_consistent else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Archivo guardado con metadatos de gap: {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 20:10:07.469531
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Calculando Líder Progresivo...
🔧 Calculando Back Adjustment y Metadatos...


/tmp/ipykernel_3882/3446643297.py:70: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_continuous['Is_Rollover_Event'] = df_continuous['Is_Rollover_Event'].shift(1).fillna(False)


📊 Finalizando estructura...
✅ Validando resultados...
   - Salto máximo ajustado en rollover: 29.6000
   - Estado de Integridad: FAILED
💾 Archivo guardado con metadatos de gap: /home/quant/data/processed/gc_1m_raw_continuous.parquet


In [7]:
df_final.head()


,Timestamp_NY,NY_Offset,Open,High,Low,Close,Volume,Ticker,Contract_ID,Adj_Factor,Is_Rollover_Event,Rollover_Gap_Technical,Market_Gap_Adjusted,Open_adj,High_adj,Low_adj,Close_adj
0,2010-06-06 18:00:00,-4,1221.1,1221.8,1221.1,1221.6,74,GCQ0,104313,-161.8,False,0.0,0.0,1059.3,1060.0,1059.3,1059.8
1,2010-06-06 18:01:00,-4,1221.6,1221.9,1221.6,1221.9,8,GCQ0,104313,-161.8,False,0.0,0.0,1059.8,1060.1,1059.8,1060.1
2,2010-06-06 18:02:00,-4,1221.9,1221.9,1221.6,1221.8,41,GCQ0,104313,-161.8,False,0.0,0.0,1060.1,1060.1,1059.8,1060.0
3,2010-06-06 18:03:00,-4,1221.6,1221.6,1220.5,1221.5,70,GCQ0,104313,-161.8,False,0.0,0.0,1059.8,1059.8,1058.7,1059.7
4,2010-06-06 18:04:00,-4,1221.0,1221.0,1220.9,1220.9,4,GCQ0,104313,-161.8,False,0.0,0.0,1059.2,1059.2,1059.1,1059.1


In [8]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df_raw = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    # Filtrar solo contratos individuales (outrights)
    df_raw = df_raw[~df_raw['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df_raw = df_raw[df_raw['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL (UTC -> NY)
    print("⏰ Normalizando zonas horarias...")
    df_raw['ts_event'] = pd.to_datetime(df_raw['ts_event'], utc=True)
    ny_tz = pytz.timezone('America/New_York')
    
    # Mantenemos la original como UTC
    df_raw['Timestamp_UTC'] = df_raw['ts_event'].dt.tz_localize(None)
    
    # Generamos la versión NY
    df_raw['Timestamp_NY_Raw'] = df_raw['ts_event'].dt.tz_convert(ny_tz)
    df_raw['Timestamp_NY'] = df_raw['Timestamp_NY_Raw'].dt.tz_localize(None)
    df_raw['NY_Offset'] = df_raw['Timestamp_NY_Raw'].apply(lambda x: x.utcoffset().total_seconds() / 3600).astype('int32')
    
    df_raw.sort_values(['Timestamp_NY', 'symbol'], inplace=True)

    # 3. LÓGICA DE LÍDER PROGRESIVO
    print("🏆 Calculando Líder Progresivo...")
    df_raw['Business_Date'] = (df_raw['Timestamp_NY'] + pd.Timedelta(hours=7)).dt.date
    
    daily_vol = df_raw.groupby(['Business_Date', 'symbol'])['volume'].sum().reset_index()
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    raw_leaders = daily_vol.loc[idx_max_vol][['Business_Date', 'symbol']]
    
    unique_leaders_ordered = []
    seen = set()
    for s in raw_leaders['symbol']:
        if s not in seen:
            unique_leaders_ordered.append(s)
            seen.add(s)
    
    leader_map = {}
    current_leader_idx = 0
    for day_data in raw_leaders.to_dict('records'):
        day, candidate = day_data['Business_Date'], day_data['symbol']
        candidate_idx = unique_leaders_ordered.index(candidate)
        if candidate_idx >= current_leader_idx:
            current_leader_idx = candidate_idx
        leader_map[day] = unique_leaders_ordered[current_leader_idx]
    
    df_raw['is_leader'] = df_raw['symbol'] == df_raw['Business_Date'].map(leader_map)
    df_continuous = df_raw[df_raw['is_leader']].copy()
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)

    # 4. BACK ADJUSTMENT Y METADATOS DE ROLLOVER
    print("🔧 Calculando Back Adjustment y Metadatos...")
    
    # Identificar cambios de ticker (usando infer_objects para evitar warnings)
    df_continuous['Is_Rollover_Event'] = (df_continuous['symbol'] != df_continuous['symbol'].shift(1))
    df_continuous.iloc[0, df_continuous.columns.get_loc('Is_Rollover_Event')] = False
    
    all_changes = df_continuous.index[df_continuous['symbol'] != df_continuous['symbol'].shift(-1)].tolist()
    if len(all_changes) > 0 and all_changes[-1] == len(df_continuous) - 1: all_changes.pop()

    factors = np.zeros(len(df_continuous))
    tech_gaps = np.zeros(len(df_continuous))
    cumulative_delta = 0.0
    
    for idx in reversed(all_changes):
        ts_rollover = df_continuous.iloc[idx]['Timestamp_NY']
        ticker_old = df_continuous.iloc[idx]['symbol']
        ticker_new = df_continuous.iloc[idx+1]['symbol']
        
        snapshot = df_raw[df_raw['Timestamp_NY'] == ts_rollover]
        p_old = snapshot[snapshot['symbol'] == ticker_old]['close']
        p_new = snapshot[snapshot['symbol'] == ticker_new]['close']
        
        if not p_old.empty and not p_new.empty:
            gap = p_new.iloc[0] - p_old.iloc[0]
        else:
            gap = df_continuous.iloc[idx+1]['close'] - df_continuous.iloc[idx]['close']
            
        cumulative_delta -= gap
        factors[:idx + 1] = cumulative_delta
        tech_gaps[idx + 1] = gap

    df_continuous['Adj_Factor'] = factors
    df_continuous['Rollover_Gap_Technical'] = tech_gaps

    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = (df_continuous[col] + df_continuous['Adj_Factor']).round(2)

    # Calcular Gap de Mercado Ajustado
    df_continuous['Market_Gap_Adjusted'] = (df_continuous['Open_adj'] - df_continuous['Close_adj'].shift(1)).fillna(0.0)
    df_continuous.loc[~df_continuous['Is_Rollover_Event'], 'Market_Gap_Adjusted'] = 0.0

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_UTC', 'Timestamp_NY', 'NY_Offset', 'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor', 'Is_Rollover_Event',
        'Rollover_Gap_Technical', 'Market_Gap_Adjusted',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN
    print("✅ Validando resultados...")
    normal_returns = df_final['Close_adj'].diff().abs().replace(0, np.nan).dropna()
    avg_vol = normal_returns.mean()
    std_vol = normal_returns.std()
    
    rollover_market_gaps = df_final[df_final['Is_Rollover_Event'] == True]['Market_Gap_Adjusted'].abs()
    max_rollover_market_gap = rollover_market_gaps.max() if not rollover_market_gaps.empty else 0.0
    
    print(f"   - Salto de mercado máximo en rollover: {max_rollover_market_gap:.4f}")
    is_passed = max_rollover_market_gap < 40.0 
    print(f"   - Estado de Integridad: {'PASSED' if is_passed else 'FAILED'}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Archivo guardado con Timestamp_UTC: {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 20:15:21.311508
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Calculando Líder Progresivo...
🔧 Calculando Back Adjustment y Metadatos...
📊 Finalizando estructura...
✅ Validando resultados...
   - Salto de mercado máximo en rollover: 26.1000
   - Estado de Integridad: PASSED
💾 Archivo guardado con Timestamp_UTC: /home/quant/data/processed/gc_1m_raw_continuous.parquet


In [9]:
df_final.head()

,Timestamp_UTC,Timestamp_NY,NY_Offset,Open,High,Low,Close,Volume,Ticker,Contract_ID,Adj_Factor,Is_Rollover_Event,Rollover_Gap_Technical,Market_Gap_Adjusted,Open_adj,High_adj,Low_adj,Close_adj
0,2010-06-06 22:00:00,2010-06-06 18:00:00,-4,1221.1,1221.8,1221.1,1221.6,74,GCQ0,104313,-161.8,False,0.0,0.0,1059.3,1060.0,1059.3,1059.8
1,2010-06-06 22:01:00,2010-06-06 18:01:00,-4,1221.6,1221.9,1221.6,1221.9,8,GCQ0,104313,-161.8,False,0.0,0.0,1059.8,1060.1,1059.8,1060.1
2,2010-06-06 22:02:00,2010-06-06 18:02:00,-4,1221.9,1221.9,1221.6,1221.8,41,GCQ0,104313,-161.8,False,0.0,0.0,1060.1,1060.1,1059.8,1060.0
3,2010-06-06 22:03:00,2010-06-06 18:03:00,-4,1221.6,1221.6,1220.5,1221.5,70,GCQ0,104313,-161.8,False,0.0,0.0,1059.8,1059.8,1058.7,1059.7
4,2010-06-06 22:04:00,2010-06-06 18:04:00,-4,1221.0,1221.0,1220.9,1220.9,4,GCQ0,104313,-161.8,False,0.0,0.0,1059.2,1059.2,1059.1,1059.1


In [10]:
df_final.tail()

,Timestamp_UTC,Timestamp_NY,NY_Offset,Open,High,Low,Close,Volume,Ticker,Contract_ID,Adj_Factor,Is_Rollover_Event,Rollover_Gap_Technical,Market_Gap_Adjusted,Open_adj,High_adj,Low_adj,Close_adj
3486691,2020-06-26 10:57:00,2020-06-26 06:57:00,-4,1765.5,1765.5,1765.5,1765.5,1,GCM0,1026,0.0,False,0.0,0.0,1765.5,1765.5,1765.5,1765.5
3486692,2020-06-26 10:58:00,2020-06-26 06:58:00,-4,1765.5,1765.5,1765.5,1765.5,2,GCM0,1026,0.0,False,0.0,0.0,1765.5,1765.5,1765.5,1765.5
3486693,2020-06-26 11:02:00,2020-06-26 07:02:00,-4,1765.5,1765.5,1765.5,1765.5,1,GCM0,1026,0.0,False,0.0,0.0,1765.5,1765.5,1765.5,1765.5
3486694,2020-06-26 11:03:00,2020-06-26 07:03:00,-4,1765.5,1765.5,1765.5,1765.5,8,GCM0,1026,0.0,False,0.0,0.0,1765.5,1765.5,1765.5,1765.5
3486695,2020-06-26 12:09:00,2020-06-26 08:09:00,-4,1764.7,1764.7,1764.7,1764.7,1,GCM0,1026,0.0,False,0.0,0.0,1764.7,1764.7,1764.7,1764.7


In [11]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"
TICK_SIZE = 0.1

def process_gold_data():
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df_raw = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    # Filtrar solo contratos individuales (outrights), descartando spreads
    df_raw = df_raw[~df_raw['symbol'].str.contains('-')]
    major_months_regex = r'^GC[GJMQUVZ]\d{1,2}$'
    df_raw = df_raw[df_raw['symbol'].str.match(major_months_regex)]

    # 2. NORMALIZACIÓN TEMPORAL (UTC -> NY)
    print("⏰ Normalizando zonas horarias...")
    df_raw['ts_event'] = pd.to_datetime(df_raw['ts_event'], utc=True)
    ny_tz = pytz.timezone('America/New_York')
    
    # Mantenemos la original como UTC para trazabilidad
    df_raw['Timestamp_UTC'] = df_raw['ts_event'].dt.tz_localize(None)
    
    # Generamos la versión NY con offset dinámico
    df_raw['Timestamp_NY_Raw'] = df_raw['ts_event'].dt.tz_convert(ny_tz)
    df_raw['Timestamp_NY'] = df_raw['Timestamp_NY_Raw'].dt.tz_localize(None)
    df_raw['NY_Offset'] = df_raw['Timestamp_NY_Raw'].apply(lambda x: x.utcoffset().total_seconds() / 3600).astype('int32')
    
    df_raw.sort_values(['Timestamp_NY', 'symbol'], inplace=True)

    # 3. LÓGICA DE LÍDER PROGRESIVO (Forward-Only)
    print("🏆 Calculando Líder Progresivo...")
    df_raw['Business_Date'] = (df_raw['Timestamp_NY'] + pd.Timedelta(hours=7)).dt.date
    
    daily_vol = df_raw.groupby(['Business_Date', 'symbol'])['volume'].sum().reset_index()
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    raw_leaders = daily_vol.loc[idx_max_vol][['Business_Date', 'symbol']]
    
    # Aseguramos que el contrato solo avance en el tiempo
    unique_leaders_ordered = []
    seen = set()
    for s in raw_leaders['symbol']:
        if s not in seen:
            unique_leaders_ordered.append(s)
            seen.add(s)
    
    leader_map = {}
    current_leader_idx = 0
    for day_data in raw_leaders.to_dict('records'):
        day, candidate = day_data['Business_Date'], day_data['symbol']
        candidate_idx = unique_leaders_ordered.index(candidate)
        if candidate_idx >= current_leader_idx:
            current_leader_idx = candidate_idx
        leader_map[day] = unique_leaders_ordered[current_leader_idx]
    
    df_raw['is_leader'] = df_raw['symbol'] == df_raw['Business_Date'].map(leader_map)
    df_continuous = df_raw[df_raw['is_leader']].copy()
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)

    # 4. BACK ADJUSTMENT Y METADATOS DE ROLLOVER
    print("🔧 Calculando Back Adjustment y Metadatos...")
    
    # Marcamos el evento de rollover (primera vela del nuevo contrato)
    df_continuous['Is_Rollover_Event'] = (df_continuous['symbol'] != df_continuous['symbol'].shift(1))
    df_continuous.loc[0, 'Is_Rollover_Event'] = False # Primer registro no es rollover
    
    all_changes = df_continuous.index[df_continuous['symbol'] != df_continuous['symbol'].shift(-1)].tolist()
    if len(all_changes) > 0 and all_changes[-1] == len(df_continuous) - 1: all_changes.pop()

    factors = np.zeros(len(df_continuous))
    tech_gaps = np.zeros(len(df_continuous))
    cumulative_delta = 0.0
    
    # Sincronización instantánea de precios para extraer el gap técnico puro
    for idx in reversed(all_changes):
        ts_rollover = df_continuous.iloc[idx]['Timestamp_NY']
        ticker_old = df_continuous.iloc[idx]['symbol']
        ticker_new = df_continuous.iloc[idx+1]['symbol']
        
        snapshot = df_raw[df_raw['Timestamp_NY'] == ts_rollover]
        p_old = snapshot[snapshot['symbol'] == ticker_old]['close']
        p_new = snapshot[snapshot['symbol'] == ticker_new]['close']
        
        if not p_old.empty and not p_new.empty:
            gap = p_new.iloc[0] - p_old.iloc[0]
        else:
            gap = df_continuous.iloc[idx+1]['close'] - df_continuous.iloc[idx]['close']
            
        cumulative_delta -= gap
        factors[:idx + 1] = cumulative_delta
        tech_gaps[idx + 1] = gap

    df_continuous['Adj_Factor'] = factors
    df_continuous['Rollover_Gap_Technical'] = tech_gaps

    # Aplicación del factor de ajuste a los precios OHLC
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = (df_continuous[col] + df_continuous['Adj_Factor']).round(2)

    # Cálculo del Gap de Mercado Real (Aislado del ajuste de contrato)
    df_continuous['Market_Gap_Adjusted'] = (df_continuous['Open_adj'] - df_continuous['Close_adj'].shift(1)).fillna(0.0)
    df_continuous.loc[~df_continuous['Is_Rollover_Event'], 'Market_Gap_Adjusted'] = 0.0

    # 5. FORMATEO FINAL
    print("📊 Finalizando estructura...")
    df_final = df_continuous[[
        'Timestamp_UTC', 'Timestamp_NY', 'NY_Offset', 'open', 'high', 'low', 'close', 'volume', 
        'symbol', 'instrument_id', 'Adj_Factor', 'Is_Rollover_Event',
        'Rollover_Gap_Technical', 'Market_Gap_Adjusted',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)

    # 6. VALIDACIÓN DE INTEGRIDAD Y HORARIOS
    print("✅ Validando resultados...")
    
    # Validación 1: Gaps de mercado
    normal_returns = df_final['Close_adj'].diff().abs().replace(0, np.nan).dropna()
    avg_vol = normal_returns.mean()
    std_vol = normal_returns.std()
    rollover_market_gaps = df_final[df_final['Is_Rollover_Event'] == True]['Market_Gap_Adjusted'].abs()
    max_rollover_market_gap = rollover_market_gaps.max() if not rollover_market_gaps.empty else 0.0
    
    print(f"   - Salto de mercado máximo detectado en rollover: {max_rollover_market_gap:.4f}")
    
    # Validación 2: Ventana de mantenimiento (17:01 - 17:59 NY)
    # Extraemos hora y minuto de la columna NY
    hours = df_final['Timestamp_NY'].dt.hour
    minutes = df_final['Timestamp_NY'].dt.minute
    
    # Filtramos registros que estén dentro de la ventana prohibida
    # Nota: La vela de las 17:00 suele ser el último minuto de trading. 
    # La ventana de "off-market" es (17:00, 18:00) exclusivo.
    forbidden_data = df_final[(hours == 17) & (minutes > 0) & (minutes < 60)]
    count_forbidden = len(forbidden_data)
    
    print(f"   - Registros detectados en ventana 17:01-17:59 NY: {count_forbidden}")
    
    is_consistent = max_rollover_market_gap < 40.0 
    is_timing_clean = count_forbidden == 0
    
    print(f"   - Estado de Integridad (Precio): {'PASSED' if is_consistent else 'FAILED'}")
    print(f"   - Estado de Integridad (Horario): {'PASSED' if is_timing_clean else 'FAILED'}")

    # Guardado en formato Parquet (Snappy)
    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"💾 Proceso completado exitosamente: {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 20:30:43.658005
📦 Cargando y filtrando instrumentos...
⏰ Normalizando zonas horarias...
🏆 Calculando Líder Progresivo...
🔧 Calculando Back Adjustment y Metadatos...
📊 Finalizando estructura...
✅ Validando resultados...
   - Salto de mercado máximo detectado en rollover: 26.1000
   - Registros detectados en ventana 17:01-17:59 NY: 16863
   - Estado de Integridad (Precio): PASSED
   - Estado de Integridad (Horario): FAILED
💾 Proceso completado exitosamente: /home/quant/data/processed/gc_1m_raw_continuous.parquet


In [23]:
import pandas as pd

# --- CONFIGURACIÓN ---
FILE_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"

def inspect_specific_candles():
    try:
        # Cargar el dataset procesado
        df = pd.read_parquet(FILE_PATH)
        
        # Convertir a datetime para la comparación
        df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
        
        # 1. INTENTO DE BÚSQUEDA EXACTA
        target_timestamps = [
            "2025-03-21 09:48:00",
            "2025-12-09 09:30:00"
        ]
        
        print(f"🔍 Buscando coincidencias exactas en: {FILE_PATH}\n")
        exact_results = df[df['Timestamp_NY'].isin(pd.to_datetime(target_timestamps))]
        
        cols_to_show = [
            'Timestamp_NY', 'Ticker', 'Open', 'Close', 
            'Open_adj', 'Close_adj', 'Adj_Factor', 'Is_Rollover_Event'
        ]

        if not exact_results.empty:
            print("✅ Coincidencias exactas encontradas:")
            print(exact_results[cols_to_show].to_string(index=False))
        else:
            print("❌ No se encontraron coincidencias exactas. Iniciando búsqueda por rangos cercanos...")
            
            # 2. BÚSQUEDA POR RANGO (Muestra la ventana de 10 minutos alrededor de las horas solicitadas)
            ranges = [
                ("2025-03-21 09:40:00", "2025-03-21 10:00:00"),
                ("2025-12-09 09:25:00", "2025-12-09 09:40:00")
            ]
            
            for start, end in ranges:
                print(f"\n--- Datos disponibles entre {start} y {end} ---")
                range_results = df[(df['Timestamp_NY'] >= start) & (df['Timestamp_NY'] <= end)]
                
                if not range_results.empty:
                    print(range_results[cols_to_show].to_string(index=False))
                else:
                    print(f"⚠️ No hay datos en absoluto para el rango {start} a {end}.")

        # 3. VERIFICACIÓN DE LÍMITES DEL DATASET
        print(f"\n📈 Info del Dataset:")
        print(f"   - Primera fecha: {df['Timestamp_NY'].min()}")
        print(f"   - Última fecha:  {df['Timestamp_NY'].max()}")
        print(f"   - Total registros: {len(df)}")
        
    except FileNotFoundError:
        print(f"❌ Error: El archivo {FILE_PATH} no existe.")
    except Exception as e:
        print(f"❌ Ocurrió un error: {e}")

if __name__ == "__main__":
    inspect_specific_candles()

🔍 Buscando coincidencias exactas en: /home/quant/data/processed/gc_1m_raw_continuous.parquet

✅ Coincidencias exactas encontradas:
       Timestamp_NY Ticker   Open  Close  Open_adj  Close_adj  Adj_Factor  Is_Rollover_Event
2025-03-21 09:48:00   GCJ5 3018.6 3016.3    2817.1     2814.8      -201.5              False
2025-12-09 09:30:00   GCG6 4222.9 4224.3    4171.9     4173.3       -51.0              False

📈 Info del Dataset:
   - Primera fecha: 2010-06-06 18:00:00
   - Última fecha:  2026-02-04 18:59:00
   - Total registros: 5495428


In [14]:
import pandas as pd

# --- CONFIGURACIÓN ---
FILE_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"

def inspect_specific_candles():
    try:
        # Cargar el dataset procesado
        df = pd.read_parquet(FILE_PATH)
        
        # Convertir a datetime para la comparación
        df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
        
        print(f"🔍 Analizando contenido de: {FILE_PATH}\n")

        # 1. DIAGNÓSTICO DE COBERTURA
        first_date = df['Timestamp_NY'].min()
        last_date = df['Timestamp_NY'].max()
        total_recs = len(df)
        
        print(f"📈 Info del Dataset:")
        print(f"   - Primera fecha: {first_date}")
        print(f"   - Última fecha:  {last_date}")
        print(f"   - Total registros: {total_recs:,}")
        
        # 2. ANÁLISIS DE SÍMBOLOS (Para ver qué contratos están incluidos)
        print(f"\n📋 Contratos presentes en la serie continua (Top 10 últimos):")
        last_tickers = df[['Timestamp_NY', 'Ticker']].drop_duplicates('Ticker', keep='last').tail(10)
        print(last_tickers.to_string(index=False))

        # 3. INTENTO DE BÚSQUEDA ESPECÍFICA
        target_timestamps = [
            "2025-03-21 09:48:00",
            "2025-12-09 09:30:00"
        ]
        
        print(f"\n🎯 Buscando velas de 2025...")
        exact_results = df[df['Timestamp_NY'].isin(pd.to_datetime(target_timestamps))]
        
        cols_to_show = [
            'Timestamp_NY', 'Ticker', 'Open', 'Close', 
            'Open_adj', 'Close_adj', 'Adj_Factor', 'Is_Rollover_Event'
        ]

        if not exact_results.empty:
            print("✅ Coincidencias exactas encontradas:")
            print(exact_results[cols_to_show].to_string(index=False))
        else:
            print("❌ No hay datos de 2025 en este archivo.")
            if last_date < pd.to_datetime("2025-01-01"):
                print(f"⚠️ Alerta: El dataset termina el {last_date}. No alcanza el año 2025.")
                print("Esto sugiere que el proceso de 'Líder Progresivo' se detuvo o el archivo fuente no tiene contratos posteriores.")

    except FileNotFoundError:
        print(f"❌ Error: El archivo {FILE_PATH} no existe.")
    except Exception as e:
        print(f"❌ Ocurrió un error: {e}")

if __name__ == "__main__":
    inspect_specific_candles()

🔍 Analizando contenido de: /home/quant/data/processed/gc_1m_raw_continuous.parquet

📈 Info del Dataset:
   - Primera fecha: 2010-06-06 18:00:00
   - Última fecha:  2020-06-26 08:09:00
   - Total registros: 3,486,696

📋 Contratos presentes en la serie continua (Top 10 últimos):
       Timestamp_NY Ticker
2018-07-27 16:59:00   GCQ8
2018-11-28 16:59:00   GCZ8
2019-01-29 16:59:00   GCG9
2019-03-27 16:59:00   GCJ9
2019-05-28 16:59:00   GCM9
2019-07-29 16:59:00   GCQ9
2019-11-26 16:59:00   GCZ9
2020-01-29 16:59:00   GCG0
2020-03-26 16:59:00   GCJ0
2020-06-26 08:09:00   GCM0

🎯 Buscando velas de 2025...
❌ No hay datos de 2025 en este archivo.
⚠️ Alerta: El dataset termina el 2020-06-26 08:09:00. No alcanza el año 2025.
Esto sugiere que el proceso de 'Líder Progresivo' se detuvo o el archivo fuente no tiene contratos posteriores.


In [15]:
import pandas as pd

# --- CONFIGURACIÓN ---
FILE_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"

def inspect_specific_candles():
    try:
        # Cargar el dataset procesado
        df = pd.read_parquet(FILE_PATH)
        
        # Convertir a datetime para la comparación
        df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
        
        print(f"🔍 Analizando contenido de: {FILE_PATH}\n")

        # 1. DIAGNÓSTICO DE COBERTURA
        first_date = df['Timestamp_NY'].min()
        last_date = df['Timestamp_NY'].max()
        total_recs = len(df)
        
        print(f"📈 Info del Dataset:")
        print(f"   - Primera fecha: {first_date}")
        print(f"   - Última fecha:  {last_date}")
        print(f"   - Total registros: {total_recs:,}")
        
        # 2. ANÁLISIS DE SÍMBOLOS (Para ver qué contratos están incluidos)
        print(f"\n📋 Contratos presentes en la serie continua (Top 10 últimos):")
        last_tickers = df[['Timestamp_NY', 'Ticker']].drop_duplicates('Ticker', keep='last').tail(10)
        print(last_tickers.to_string(index=False))

        # 3. INTENTO DE BÚSQUEDA ESPECÍFICA
        target_timestamps = [
            "2025-03-21 09:48:00",
            "2025-12-09 09:30:00"
        ]
        
        print(f"\n🎯 Buscando velas de 2025...")
        exact_results = df[df['Timestamp_NY'].isin(pd.to_datetime(target_timestamps))]
        
        cols_to_show = [
            'Timestamp_NY', 'Ticker', 'Open', 'Close', 
            'Open_adj', 'Close_adj', 'Adj_Factor', 'Is_Rollover_Event'
        ]

        if not exact_results.empty:
            print("✅ Coincidencias exactas encontradas:")
            print(exact_results[cols_to_show].to_string(index=False))
        else:
            print("❌ No hay datos de 2025 en este archivo.")
            if last_date < pd.to_datetime("2025-01-01"):
                print(f"⚠️ Alerta: El dataset termina el {last_date}. No alcanza el año 2025.")
                print("Esto sugiere que el proceso de 'Líder Progresivo' se detuvo o el archivo fuente no tiene contratos posteriores.")

    except FileNotFoundError:
        print(f"❌ Error: El archivo {FILE_PATH} no existe.")
    except Exception as e:
        print(f"❌ Ocurrió un error: {e}")

if __name__ == "__main__":
    inspect_specific_candles()

🔍 Analizando contenido de: /home/quant/data/processed/gc_1m_raw_continuous.parquet

📈 Info del Dataset:
   - Primera fecha: 2010-06-06 18:00:00
   - Última fecha:  2020-06-26 08:09:00
   - Total registros: 3,486,696

📋 Contratos presentes en la serie continua (Top 10 últimos):
       Timestamp_NY Ticker
2018-07-27 16:59:00   GCQ8
2018-11-28 16:59:00   GCZ8
2019-01-29 16:59:00   GCG9
2019-03-27 16:59:00   GCJ9
2019-05-28 16:59:00   GCM9
2019-07-29 16:59:00   GCQ9
2019-11-26 16:59:00   GCZ9
2020-01-29 16:59:00   GCG0
2020-03-26 16:59:00   GCJ0
2020-06-26 08:09:00   GCM0

🎯 Buscando velas de 2025...
❌ No hay datos de 2025 en este archivo.
⚠️ Alerta: El dataset termina el 2020-06-26 08:09:00. No alcanza el año 2025.
Esto sugiere que el proceso de 'Líder Progresivo' se detuvo o el archivo fuente no tiene contratos posteriores.


In [16]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime
import os
import re

# --- CONFIGURACIÓN DE GOBERNANZA ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"

def process_gold_data():
    print(f"🚀 Iniciando procesamiento (Gestión de Ciclos Decenales): {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando y filtrando instrumentos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df_raw = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    # Filtrar solo contratos individuales (outrights)
    df_raw = df_raw[~df_raw['symbol'].str.contains('-')]
    
    # 2. NORMALIZACIÓN TEMPORAL (UTC -> NY)
    df_raw['ts_event'] = pd.to_datetime(df_raw['ts_event'], utc=True)
    ny_tz = pytz.timezone('America/New_York')
    df_raw['Timestamp_UTC'] = df_raw['ts_event'].dt.tz_localize(None)
    df_raw['Timestamp_NY_Raw'] = df_raw['ts_event'].dt.tz_convert(ny_tz)
    df_raw['Timestamp_NY'] = df_raw['Timestamp_NY_Raw'].dt.tz_localize(None)
    df_raw['NY_Offset'] = df_raw['Timestamp_NY_Raw'].apply(lambda x: x.utcoffset().total_seconds() / 3600).astype('int32')
    df_raw['Year'] = df_raw['Timestamp_NY'].dt.year

    # 3. CREACIÓN DE TICKER ÚNICO (Solución al ciclo de 10 años)
    # Si el símbolo es GCZ5 y el año es 2015 -> GCZ5_2015
    # Si el símbolo es GCZ5 y el año es 2025 -> GCZ5_2025
    print("🆔 Generando identificadores únicos por ciclo decenal...")
    df_raw['Unique_Symbol'] = df_raw['symbol'] + "_" + df_raw['Year'].astype(str)

    df_raw.sort_values(['Timestamp_NY', 'Unique_Symbol'], inplace=True)

    # 4. LÓGICA DE LÍDER PROGRESIVO
    print("🏆 Calculando Líder Progresivo...")
    df_raw['Business_Date'] = (df_raw['Timestamp_NY'] + pd.Timedelta(hours=7)).dt.date
    
    # Buscamos el volumen máximo por día usando el símbolo único
    daily_vol = df_raw.groupby(['Business_Date', 'Unique_Symbol'])['volume'].sum().reset_index()
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    raw_leaders = daily_vol.loc[idx_max_vol][['Business_Date', 'Unique_Symbol']]
    
    # Mantener orden cronológico de contratos únicos
    unique_leaders_ordered = []
    seen = set()
    for s in raw_leaders['Unique_Symbol']:
        if s not in seen:
            unique_leaders_ordered.append(s)
            seen.add(s)
    
    leader_map = {}
    current_leader_idx = 0
    for day_data in raw_leaders.to_dict('records'):
        day, candidate = day_data['Business_Date'], day_data['Unique_Symbol']
        candidate_idx = unique_leaders_ordered.index(candidate)
        if candidate_idx >= current_leader_idx:
            current_leader_idx = candidate_idx
        leader_map[day] = unique_leaders_ordered[current_leader_idx]
    
    df_raw['is_leader'] = df_raw['Unique_Symbol'] == df_raw['Business_Date'].map(leader_map)
    df_continuous = df_raw[df_raw['is_leader']].copy()
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)

    # 5. BACK ADJUSTMENT
    print("🔧 Calculando Back Adjustment...")
    df_continuous['Is_Rollover_Event'] = (df_continuous['Unique_Symbol'] != df_continuous['Unique_Symbol'].shift(1))
    df_continuous.loc[0, 'Is_Rollover_Event'] = False
    
    all_changes = df_continuous.index[df_continuous['Unique_Symbol'] != df_continuous['Unique_Symbol'].shift(-1)].tolist()
    if len(all_changes) > 0 and all_changes[-1] == len(df_continuous) - 1: all_changes.pop()

    factors = np.zeros(len(df_continuous))
    tech_gaps = np.zeros(len(df_continuous))
    cumulative_delta = 0.0
    
    for idx in reversed(all_changes):
        ts_rollover = df_continuous.iloc[idx]['Timestamp_NY']
        u_ticker_old = df_continuous.iloc[idx]['Unique_Symbol']
        u_ticker_new = df_continuous.iloc[idx+1]['Unique_Symbol']
        
        snapshot = df_raw[df_raw['Timestamp_NY'] == ts_rollover]
        p_old = snapshot[snapshot['Unique_Symbol'] == u_ticker_old]['close']
        p_new = snapshot[snapshot['Unique_Symbol'] == u_ticker_new]['close']
        
        if not p_old.empty and not p_new.empty:
            gap = p_new.iloc[0] - p_old.iloc[0]
        else:
            gap = df_continuous.iloc[idx+1]['close'] - df_continuous.iloc[idx]['close']
            
        cumulative_delta -= gap
        factors[:idx + 1] = cumulative_delta
        tech_gaps[idx + 1] = gap

    df_continuous['Adj_Factor'] = factors
    df_continuous['Rollover_Gap_Technical'] = tech_gaps

    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = (df_continuous[col] + df_continuous['Adj_Factor']).round(2)

    df_continuous['Market_Gap_Adjusted'] = (df_continuous['Open_adj'] - df_continuous['Close_adj'].shift(1)).fillna(0.0)
    df_continuous.loc[~df_continuous['Is_Rollover_Event'], 'Market_Gap_Adjusted'] = 0.0

    # 6. FORMATEO Y VALIDACIÓN
    print("📊 Finalizando estructura...")
    # Limpiar el nombre del ticker para el output final (quitar el año auxiliar)
    df_continuous['Ticker_Final'] = df_continuous['Unique_Symbol'].str.split('_').str[0]

    df_final = df_continuous[[
        'Timestamp_UTC', 'Timestamp_NY', 'NY_Offset', 'open', 'high', 'low', 'close', 'volume', 
        'Ticker_Final', 'instrument_id', 'Adj_Factor', 'Is_Rollover_Event',
        'Rollover_Gap_Technical', 'Market_Gap_Adjusted',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={'Ticker_Final': 'Ticker', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume', 'instrument_id': 'Contract_ID'}, inplace=True)

    # Verificación de horarios 17:01-17:59
    forbidden = df_final[(df_final['Timestamp_NY'].dt.hour == 17) & (df_final['Timestamp_NY'].dt.minute > 0) & (df_final['Timestamp_NY'].dt.minute < 60)]
    
    print(f"✅ Validación Final:")
    print(f"   - Rango: {df_final['Timestamp_NY'].min()} a {df_final['Timestamp_NY'].max()}")
    print(f"   - Registros: {len(df_final):,}")
    print(f"   - Registros prohibidos (17:01-17:59): {len(forbidden)}")

    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento (Gestión de Ciclos Decenales): 2026-02-17 20:37:23.377615
📦 Cargando y filtrando instrumentos...
🆔 Generando identificadores únicos por ciclo decenal...
🏆 Calculando Líder Progresivo...
🔧 Calculando Back Adjustment...
📊 Finalizando estructura...
✅ Validación Final:
   - Rango: 2010-06-06 18:00:00 a 2026-02-04 18:59:00
   - Registros: 5,495,428
   - Registros prohibidos (17:01-17:59): 16850


In [19]:
import pandas as pd

def analyze_maintenance_data(df):
    """
    Analiza los registros marcados en la ventana de mantenimiento (17:01-17:59 NY)
    utilizando el DataFrame df_final generado en memoria.
    """
    try:
        # Verificación de existencia de la columna
        if 'Is_Maintenance_Window' not in df.columns:
            print("❌ Error: La columna 'Is_Maintenance_Window' no existe en el DataFrame.")
            return

        # Filtrar solo los registros en la ventana de mantenimiento
        maint_df = df[df['Is_Maintenance_Window'] == True].copy()
        
        if maint_df.empty:
            print("✅ No se encontraron registros en la ventana de mantenimiento (17:01-17:59).")
            return

        print(f"🔍 Analizando {len(maint_df)} registros encontrados en la ventana de mantenimiento\n")
        
        # 1. Estadísticas de Volumen
        total_vol = maint_df['Volume'].sum()
        avg_vol = maint_df['Volume'].mean()
        max_vol = maint_df['Volume'].max()
        
        print(f"📊 Estadísticas de Volumen en Ventana 17:01-17:59:")
        print(f"   - Volumen Total: {total_vol:,}")
        print(f"   - Volumen Promedio por Vela: {avg_vol:.2f}")
        print(f"   - Volumen Máximo en una Vela: {max_vol}")
        
        # 2. Análisis de Actividad por Minuto
        print(f"\n⏰ Distribución por minuto (Top 10 minutos con más actividad):")
        # Aseguramos que sea datetime para extraer el minuto
        maint_df['Timestamp_NY'] = pd.to_datetime(maint_df['Timestamp_NY'])
        maint_df['Minute'] = maint_df['Timestamp_NY'].dt.minute
        min_dist = maint_df.groupby('Minute')['Volume'].count().sort_values(ascending=False).head(10)
        print(min_dist)

        # 3. Muestra de los datos (las 15 velas con más volumen)
        print(f"\n📋 Muestra de las 15 velas con mayor volumen en mantenimiento:")
        cols_show = ['Timestamp_NY', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
        sample = maint_df.sort_values('Volume', ascending=False).head(15)
        print(sample[cols_show].to_string(index=False))

        # 4. Verificación de volatilidad
        maint_df['Range'] = maint_df['High'] - maint_df['Low']
        max_range = maint_df['Range'].max()
        print(f"\n🚀 Volatilidad máxima detectada (High-Low): {max_range:.2f} puntos.")

    except Exception as e:
        print(f"❌ Ocurrió un error durante el análisis: {e}")

# Ejecución automática si df_final existe en el espacio de nombres global
if 'df_final' in globals():
    analyze_maintenance_data(df_final)
else:
    print("⚠️ El objeto 'df_final' no se encuentra en memoria. Asegúrate de ejecutar el proceso principal primero.")

❌ Error: La columna 'Is_Maintenance_Window' no existe en el DataFrame.


In [20]:
import pandas as pd

def analyze_maintenance_data(df):
    """
    Analiza los registros marcados en la ventana de mantenimiento (17:01-17:59 NY)
    utilizando el DataFrame df_final generado en memoria.
    """
    try:
        # Asegurar que Timestamp_NY sea datetime
        if 'Timestamp_NY' in df.columns:
            df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
        else:
            print("❌ Error: La columna 'Timestamp_NY' no existe en el DataFrame.")
            return

        # Bloque de compatibilidad: Calcular la columna si no existe en df_final
        if 'Is_Maintenance_Window' not in df.columns:
            print("⚠️ Columna 'Is_Maintenance_Window' no detectada. Calculando dinámicamente...")
            hour = df['Timestamp_NY'].dt.hour
            minute = df['Timestamp_NY'].dt.minute
            df['Is_Maintenance_Window'] = (hour == 17) & (minute > 0)

        # Filtrar solo los registros en la ventana de mantenimiento
        maint_df = df[df['Is_Maintenance_Window'] == True].copy()
        
        if maint_df.empty:
            print("✅ No se encontraron registros en la ventana de mantenimiento (17:01-17:59).")
            return

        print(f"🔍 Analizando {len(maint_df)} registros encontrados en la ventana de mantenimiento\n")
        
        # 1. Estadísticas de Volumen
        total_vol = maint_df['Volume'].sum()
        avg_vol = maint_df['Volume'].mean()
        max_vol = maint_df['Volume'].max()
        
        print(f"📊 Estadísticas de Volumen en Ventana 17:01-17:59:")
        print(f"   - Volumen Total: {total_vol:,}")
        print(f"   - Volumen Promedio por Vela: {avg_vol:.2f}")
        print(f"   - Volumen Máximo en una Vela: {max_vol}")
        
        # 2. Análisis de Actividad por Minuto
        print(f"\n⏰ Distribución por minuto (Top 10 minutos con más actividad):")
        maint_df['Minute'] = maint_df['Timestamp_NY'].dt.minute
        min_dist = maint_df.groupby('Minute')['Volume'].count().sort_values(ascending=False).head(10)
        print(min_dist)

        # 3. Muestra de los datos (las 15 velas con más volumen)
        print(f"\n📋 Muestra de las 15 velas con mayor volumen en mantenimiento:")
        cols_show = ['Timestamp_NY', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
        sample = maint_df.sort_values('Volume', ascending=False).head(15)
        print(sample[cols_show].to_string(index=False))

        # 4. Verificación de volatilidad
        maint_df['Range'] = maint_df['High'] - maint_df['Low']
        max_range = maint_df['Range'].max()
        print(f"\n🚀 Volatilidad máxima detectada (High-Low): {max_range:.2f} puntos.")

    except Exception as e:
        print(f"❌ Ocurrió un error durante el análisis: {e}")

# Ejecución automática si df_final existe en el espacio de nombres global
if 'df_final' in globals():
    analyze_maintenance_data(df_final)
else:
    print("⚠️ El objeto 'df_final' no se encuentra en memoria. Asegúrate de ejecutar el proceso principal primero.")

⚠️ Columna 'Is_Maintenance_Window' no detectada. Calculando dinámicamente...
🔍 Analizando 16850 registros encontrados en la ventana de mantenimiento

📊 Estadísticas de Volumen en Ventana 17:01-17:59:
   - Volumen Total: 232,839
   - Volumen Promedio por Vela: 13.82
   - Volumen Máximo en una Vela: 1411

⏰ Distribución por minuto (Top 10 minutos con más actividad):
Minute
14    1313
10    1270
13    1257
12    1255
11    1255
5     1220
6     1180
1     1172
9     1169
4     1158
Name: Volume, dtype: int64

📋 Muestra de las 15 velas con mayor volumen en mantenimiento:
       Timestamp_NY Ticker   Open   High    Low  Close  Volume
2011-06-15 17:03:00   GCQ1 1532.1 1532.2 1532.0 1532.0    1411
2014-09-09 17:02:00   GCZ4 1256.6 1256.7 1256.5 1256.6    1158
2013-03-20 17:03:00   GCJ3 1606.0 1606.0 1606.0 1606.0     999
2011-08-22 17:09:00   GCZ1 1899.4 1904.0 1899.4 1903.6     878
2014-12-19 17:05:00   GCG5 1194.5 1194.6 1194.1 1194.6     791
2013-07-10 17:04:00   GCQ3 1259.9 1265.6 1259.9 

In [21]:
import pandas as pd

def analyze_maintenance_data(df):
    """
    Analiza los registros marcados en la ventana de mantenimiento (17:01-17:59 NY)
    utilizando el DataFrame df_final generado en memoria.
    """
    try:
        # Asegurar que Timestamp_NY sea datetime
        if 'Timestamp_NY' in df.columns:
            df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
        else:
            print("❌ Error: La columna 'Timestamp_NY' no existe en el DataFrame.")
            return

        # Bloque de compatibilidad: Calcular la columna si no existe en df_final
        if 'Is_Maintenance_Window' not in df.columns:
            print("⚠️ Columna 'Is_Maintenance_Window' no detectada. Calculando dinámicamente...")
            hour = df['Timestamp_NY'].dt.hour
            minute = df['Timestamp_NY'].dt.minute
            df['Is_Maintenance_Window'] = (hour == 17) & (minute > 0)

        # Filtrar solo los registros en la ventana de mantenimiento
        maint_df = df[df['Is_Maintenance_Window'] == True].copy()
        
        if maint_df.empty:
            print("✅ No se encontraron registros en la ventana de mantenimiento (17:01-17:59).")
            return

        print(f"🔍 Analizando {len(maint_df)} registros encontrados en la ventana de mantenimiento\n")
        
        # 1. Estadísticas de Volumen
        total_vol = maint_df['Volume'].sum()
        avg_vol = maint_df['Volume'].mean()
        max_vol = maint_df['Volume'].max()
        
        print(f"📊 Estadísticas de Volumen en Ventana 17:01-17:59:")
        print(f"   - Volumen Total: {total_vol:,}")
        print(f"   - Volumen Promedio por Vela: {avg_vol:.2f}")
        print(f"   - Volumen Máximo en una Vela: {max_vol}")
        
        # 2. Detección de anomalías por fechas (Posible error de horario de verano)
        print(f"\n📅 Días con mayor actividad en ventana prohibida (Top 10):")
        maint_df['Date'] = maint_df['Timestamp_NY'].dt.date
        date_activity = maint_df.groupby('Date').agg({
            'Volume': 'sum',
            'Timestamp_NY': 'count'
        }).rename(columns={'Timestamp_NY': 'Candles'}).sort_values('Volume', ascending=False).head(10)
        print(date_activity)

        # 3. Análisis de Actividad por Minuto
        print(f"\n⏰ Distribución por minuto (Top 10 minutos con más actividad):")
        maint_df['Minute'] = maint_df['Timestamp_NY'].dt.minute
        min_dist = maint_df.groupby('Minute')['Volume'].count().sort_values(ascending=False).head(10)
        print(min_dist)

        # 4. Muestra de los datos (las 15 velas con más volumen)
        print(f"\n📋 Muestra de las 15 velas con mayor volumen en mantenimiento:")
        cols_show = ['Timestamp_NY', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
        sample = maint_df.sort_values('Volume', ascending=False).head(15)
        print(sample[cols_show].to_string(index=False))

        # 5. Verificación de impacto en precio (Volatilidad)
        maint_df['Range'] = maint_df['High'] - maint_df['Low']
        max_range = maint_df['Range'].max()
        print(f"\n🚀 Volatilidad máxima detectada (High-Low) en mantenimiento: {max_range:.2f} puntos.")
        
        # 6. Conclusión sobre Horario de Verano
        print(f"\n💡 Nota sobre Horario de Verano:")
        march_nov_data = maint_df[maint_df['Timestamp_NY'].dt.month.isin([3, 11])]
        if not march_nov_data.empty:
            print(f"   Se detectaron {len(march_nov_data)} registros en Marzo/Noviembre.")
            print("   Si estos registros ocurren cerca del segundo domingo de Marzo o primero de Noviembre,")
            print("   es probable que el ajuste de zona horaria de la fuente original sea la causa.")

    except Exception as e:
        print(f"❌ Ocurrió un error durante el análisis: {e}")

# Ejecución automática si df_final existe en el espacio de nombres global
if 'df_final' in globals():
    analyze_maintenance_data(df_final)
else:
    print("⚠️ El objeto 'df_final' no se encuentra en memoria. Asegúrate de ejecutar el proceso principal primero.")

🔍 Analizando 16850 registros encontrados en la ventana de mantenimiento

📊 Estadísticas de Volumen en Ventana 17:01-17:59:
   - Volumen Total: 232,839
   - Volumen Promedio por Vela: 13.82
   - Volumen Máximo en una Vela: 1411

📅 Días con mayor actividad en ventana prohibida (Top 10):
            Volume  Candles
Date                       
2011-06-15    2273       12
2013-07-10    2228       14
2013-04-12    1919       14
2011-08-22    1753       14
2015-03-20    1558       14
2014-09-09    1347       14
2011-07-13    1326       15
2013-04-15    1196       14
2011-08-24    1133       14
2013-03-20    1111       13

⏰ Distribución por minuto (Top 10 minutos con más actividad):
Minute
14    1313
10    1270
13    1257
12    1255
11    1255
5     1220
6     1180
1     1172
9     1169
4     1158
Name: Volume, dtype: int64

📋 Muestra de las 15 velas con mayor volumen en mantenimiento:
       Timestamp_NY Ticker   Open   High    Low  Close  Volume
2011-06-15 17:03:00   GCQ1 1532.1 1532.2 1532

In [22]:
import pandas as pd

def analyze_maintenance_data(df):
    """
    Analiza los registros marcados en la ventana de mantenimiento (17:01-17:59 NY)
    utilizando el DataFrame df_final generado en memoria.
    """
    try:
        # Asegurar que Timestamp_NY sea datetime
        if 'Timestamp_NY' in df.columns:
            df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
        else:
            print("❌ Error: La columna 'Timestamp_NY' no existe en el DataFrame.")
            return

        # Bloque de compatibilidad: Calcular la columna si no existe en df_final
        if 'Is_Maintenance_Window' not in df.columns:
            print("⚠️ Columna 'Is_Maintenance_Window' no detectada. Calculando dinámicamente...")
            hour = df['Timestamp_NY'].dt.hour
            minute = df['Timestamp_NY'].dt.minute
            df['Is_Maintenance_Window'] = (hour == 17) & (minute > 0)

        # Filtrar solo los registros en la ventana de mantenimiento
        maint_df = df[df['Is_Maintenance_Window'] == True].copy()
        
        if maint_df.empty:
            print("✅ No se encontraron registros en la ventana de mantenimiento (17:01-17:59).")
            return

        print(f"🔍 Analizando {len(maint_df)} registros encontrados en la ventana de mantenimiento\n")
        
        # 1. Estadísticas de Volumen
        total_vol = maint_df['Volume'].sum()
        avg_vol = maint_df['Volume'].mean()
        max_vol = maint_df['Volume'].max()
        
        print(f"📊 Estadísticas de Volumen en Ventana 17:01-17:59:")
        print(f"   - Volumen Total: {total_vol:,}")
        print(f"   - Volumen Promedio por Vela: {avg_vol:.2f}")
        print(f"   - Volumen Máximo en una Vela: {max_vol}")
        
        # 2. Análisis de Densidad (¿Hora completa o esporádica?)
        print(f"\n🧩 Análisis de Densidad (Población de la hora):")
        maint_df['Date'] = maint_df['Timestamp_NY'].dt.date
        candles_per_day = maint_df.groupby('Date')['Timestamp_NY'].count()
        
        avg_candles = candles_per_day.mean()
        max_candles = candles_per_day.max()
        days_full_hour = (candles_per_day >= 50).sum()
        days_sporadic = (candles_per_day < 10).sum()

        print(f"   - Promedio de velas por sesión afectada: {avg_candles:.1f} / 59 min")
        print(f"   - Máximo de velas en una sola tarde: {max_candles}")
        print(f"   - Sesiones con hora casi completa (>50 min): {days_full_hour}")
        print(f"   - Sesiones muy esporádicas (<10 min): {days_sporadic}")
        
        # 3. Detección de anomalías por fechas
        print(f"\n📅 Días con mayor volumen total en ventana prohibida (Top 10):")
        date_activity = maint_df.groupby('Date').agg({
            'Volume': 'sum',
            'Timestamp_NY': 'count'
        }).rename(columns={'Timestamp_NY': 'Candles'}).sort_values('Volume', ascending=False).head(10)
        print(date_activity)

        # 4. Distribución por minuto
        print(f"\n⏰ Minutos con presencia de datos (Top 10 minutos más frecuentes):")
        maint_df['Minute'] = maint_df['Timestamp_NY'].dt.minute
        min_dist = maint_df.groupby('Minute')['Timestamp_NY'].count().sort_values(ascending=False).head(10)
        print(min_dist)

        # 5. Muestra de los datos con mayor volumen
        print(f"\n📋 Muestra de las 15 velas con mayor volumen en mantenimiento:")
        cols_show = ['Timestamp_NY', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
        sample = maint_df.sort_values('Volume', ascending=False).head(15)
        print(sample[cols_show].to_string(index=False))

        # 6. Verificación de impacto en precio (Volatilidad)
        maint_df['Range'] = maint_df['High'] - maint_df['Low']
        max_range = maint_df['Range'].max()
        print(f"\n🚀 Volatilidad máxima detectada (High-Low) en mantenimiento: {max_range:.2f} puntos.")
        
        # 7. Conclusión sobre Horario de Verano
        print(f"\n💡 Nota sobre Horario de Verano y Densidad:")
        march_nov_data = maint_df[maint_df['Timestamp_NY'].dt.month.isin([3, 11])]
        if not march_nov_data.empty:
            print(f"   Se detectaron {len(march_nov_data)} registros en Marzo/Noviembre.")
            print("   Interpretación: Si 'Promedio de velas' es bajo (~1-5), son errores de zona horaria.")
            print("   Si es alto (>40), es probable que el mercado cerrara más tarde ese día o sea un error sistémico de offset.")

    except Exception as e:
        print(f"❌ Ocurrió un error durante el análisis: {e}")

# Ejecución automática si df_final existe en el espacio de nombres global
if 'df_final' in globals():
    analyze_maintenance_data(df_final)
else:
    print("⚠️ El objeto 'df_final' no se encuentra en memoria. Asegúrate de ejecutar el proceso principal primero.")

🔍 Analizando 16850 registros encontrados en la ventana de mantenimiento

📊 Estadísticas de Volumen en Ventana 17:01-17:59:
   - Volumen Total: 232,839
   - Volumen Promedio por Vela: 13.82
   - Volumen Máximo en una Vela: 1411

🧩 Análisis de Densidad (Población de la hora):
   - Promedio de velas por sesión afectada: 12.8 / 59 min
   - Máximo de velas en una sola tarde: 15
   - Sesiones con hora casi completa (>50 min): 0
   - Sesiones muy esporádicas (<10 min): 29

📅 Días con mayor volumen total en ventana prohibida (Top 10):
            Volume  Candles
Date                       
2011-06-15    2273       12
2013-07-10    2228       14
2013-04-12    1919       14
2011-08-22    1753       14
2015-03-20    1558       14
2014-09-09    1347       14
2011-07-13    1326       15
2013-04-15    1196       14
2011-08-24    1133       14
2013-03-20    1111       13

⏰ Minutos con presencia de datos (Top 10 minutos más frecuentes):
Minute
14    1313
10    1270
13    1257
12    1255
11    1255
5 

In [24]:
df_final.head()


,Timestamp_UTC,Timestamp_NY,NY_Offset,Open,High,Low,Close,Volume,Ticker,Contract_ID,Adj_Factor,Is_Rollover_Event,Rollover_Gap_Technical,Market_Gap_Adjusted,Open_adj,High_adj,Low_adj,Close_adj,Is_Maintenance_Window
0,2010-06-06 22:00:00,2010-06-06 18:00:00,-4,1221.1,1221.8,1221.1,1221.6,74,GCQ0,104313,-758.6,False,0.0,0.0,462.5,463.2,462.5,463.0,False
1,2010-06-06 22:01:00,2010-06-06 18:01:00,-4,1221.6,1221.9,1221.6,1221.9,8,GCQ0,104313,-758.6,False,0.0,0.0,463.0,463.3,463.0,463.3,False
2,2010-06-06 22:02:00,2010-06-06 18:02:00,-4,1221.9,1221.9,1221.6,1221.8,41,GCQ0,104313,-758.6,False,0.0,0.0,463.3,463.3,463.0,463.2,False
3,2010-06-06 22:03:00,2010-06-06 18:03:00,-4,1221.6,1221.6,1220.5,1221.5,70,GCQ0,104313,-758.6,False,0.0,0.0,463.0,463.0,461.9,462.9,False
4,2010-06-06 22:04:00,2010-06-06 18:04:00,-4,1221.0,1221.0,1220.9,1220.9,4,GCQ0,104313,-758.6,False,0.0,0.0,462.4,462.4,462.3,462.3,False


In [25]:
import pandas as pd

def analyze_maintenance_data(df):
    """
    Analiza los registros marcados en la ventana de mantenimiento (17:01-17:59 NY)
    utilizando el DataFrame df_final generado en memoria.
    """
    try:
        # Asegurar que Timestamp_NY sea datetime
        if 'Timestamp_NY' in df.columns:
            df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
        else:
            print("❌ Error: La columna 'Timestamp_NY' no existe en el DataFrame.")
            return

        # Bloque de compatibilidad: Calcular la columna si no existe en df_final
        if 'Is_Maintenance_Window' not in df.columns:
            print("⚠️ Columna 'Is_Maintenance_Window' no detectada. Calculando dinámicamente...")
            hour = df['Timestamp_NY'].dt.hour
            minute = df['Timestamp_NY'].dt.minute
            df['Is_Maintenance_Window'] = (hour == 17) & (minute > 0)

        # Filtrar solo los registros en la ventana de mantenimiento
        maint_df = df[df['Is_Maintenance_Window'] == True].copy()
        
        if maint_df.empty:
            print("✅ No se encontraron registros en la ventana de mantenimiento (17:01-17:59).")
            return

        print(f"🔍 Analizando {len(maint_df)} registros encontrados en la ventana de mantenimiento\n")
        
        # 1. Estadísticas de Volumen
        total_vol = maint_df['Volume'].sum()
        avg_vol = maint_df['Volume'].mean()
        max_vol = maint_df['Volume'].max()
        
        print(f"📊 Estadísticas de Volumen en Ventana 17:01-17:59:")
        print(f"   - Volumen Total: {total_vol:,}")
        print(f"   - Volumen Promedio por Vela: {avg_vol:.2f}")
        print(f"   - Volumen Máximo en una Vela: {max_vol}")
        
        # 2. Análisis de Densidad (¿Hora completa o esporádica?)
        print(f"\n🧩 Análisis de Densidad (Población de la hora):")
        maint_df['Date'] = maint_df['Timestamp_NY'].dt.date
        candles_per_day = maint_df.groupby('Date')['Timestamp_NY'].count()
        
        avg_candles = candles_per_day.mean()
        max_candles = candles_per_day.max()
        days_full_hour = (candles_per_day >= 50).sum()
        days_sporadic = (candles_per_day < 10).sum()

        print(f"   - Promedio de velas por sesión afectada: {avg_candles:.1f} / 59 min")
        print(f"   - Máximo de velas en una sola tarde: {max_candles}")
        print(f"   - Sesiones con hora casi completa (>50 min): {days_full_hour}")
        print(f"   - Sesiones muy esporádicas (<10 min): {days_sporadic}")
        
        # 3. Detección de anomalías por fechas
        print(f"\n📅 Días con mayor volumen total en ventana prohibida (Top 10):")
        date_activity = maint_df.groupby('Date').agg({
            'Volume': 'sum',
            'Timestamp_NY': 'count'
        }).rename(columns={'Timestamp_NY': 'Candles'}).sort_values('Volume', ascending=False).head(10)
        print(date_activity)

        # 4. Distribución por minuto
        print(f"\n⏰ Minutos con presencia de datos (Top 10 minutos más frecuentes):")
        maint_df['Minute'] = maint_df['Timestamp_NY'].dt.minute
        min_dist = maint_df.groupby('Minute')['Timestamp_NY'].count().sort_values(ascending=False).head(10)
        print(min_dist)

        # 5. Muestra de los datos con mayor volumen
        print(f"\n📋 Muestra de las 15 velas con mayor volumen en mantenimiento:")
        cols_show = ['Timestamp_NY', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
        sample = maint_df.sort_values('Volume', ascending=False).head(15)
        print(sample[cols_show].to_string(index=False))

        # 6. Verificación de impacto en precio (Volatilidad)
        maint_df['Range'] = maint_df['High'] - maint_df['Low']
        max_range = maint_df['Range'].max()
        print(f"\n🚀 Volatilidad máxima detectada (High-Low) en mantenimiento: {max_range:.2f} puntos.")
        
        # 7. Conclusión sobre Horario de Verano
        print(f"\n💡 Nota sobre Horario de Verano y Densidad:")
        march_nov_data = maint_df[maint_df['Timestamp_NY'].dt.month.isin([3, 11])]
        if not march_nov_data.empty:
            print(f"   Se detectaron {len(march_nov_data)} registros en Marzo/Noviembre.")
            print("   Interpretación: Si 'Promedio de velas' es bajo (~1-5), son errores de zona horaria.")
            print("   Si es alto (>40), es probable que el mercado cerrara más tarde ese día o sea un error sistémico de offset.")

    except Exception as e:
        print(f"❌ Ocurrió un error durante el análisis: {e}")

# Ejecución automática si df_final existe en el espacio de nombres global
if 'df_final' in globals():
    analyze_maintenance_data(df_final)
else:
    print("⚠️ El objeto 'df_final' no se encuentra en memoria. Asegúrate de ejecutar el proceso principal primero.")

🔍 Analizando 16850 registros encontrados en la ventana de mantenimiento

📊 Estadísticas de Volumen en Ventana 17:01-17:59:
   - Volumen Total: 232,839
   - Volumen Promedio por Vela: 13.82
   - Volumen Máximo en una Vela: 1411

🧩 Análisis de Densidad (Población de la hora):
   - Promedio de velas por sesión afectada: 12.8 / 59 min
   - Máximo de velas en una sola tarde: 15
   - Sesiones con hora casi completa (>50 min): 0
   - Sesiones muy esporádicas (<10 min): 29

📅 Días con mayor volumen total en ventana prohibida (Top 10):
            Volume  Candles
Date                       
2011-06-15    2273       12
2013-07-10    2228       14
2013-04-12    1919       14
2011-08-22    1753       14
2015-03-20    1558       14
2014-09-09    1347       14
2011-07-13    1326       15
2013-04-15    1196       14
2011-08-24    1133       14
2013-03-20    1111       13

⏰ Minutos con presencia de datos (Top 10 minutos más frecuentes):
Minute
14    1313
10    1270
13    1257
12    1255
11    1255
5 

In [26]:
import pandas as pd

def analyze_maintenance_data(df):
    """
    Analiza los registros marcados en la ventana de mantenimiento (17:01-17:59 NY)
    utilizando el DataFrame df_final generado en memoria.
    """
    try:
        # Asegurar que Timestamp_NY sea datetime
        if 'Timestamp_NY' in df.columns:
            df['Timestamp_NY'] = pd.to_datetime(df['Timestamp_NY'])
        else:
            print("❌ Error: La columna 'Timestamp_NY' no existe en el DataFrame.")
            return

        # Bloque de compatibilidad: Calcular la columna si no existe en df_final
        if 'Is_Maintenance_Window' not in df.columns:
            print("⚠️ Columna 'Is_Maintenance_Window' no detectada. Calculando dinámicamente...")
            hour = df['Timestamp_NY'].dt.hour
            minute = df['Timestamp_NY'].dt.minute
            df['Is_Maintenance_Window'] = (hour == 17) & (minute > 0)

        # Filtrar solo los registros en la ventana de mantenimiento
        maint_df = df[df['Is_Maintenance_Window'] == True].copy()
        
        if maint_df.empty:
            print("✅ No se encontraron registros en la ventana de mantenimiento (17:01-17:59).")
            return

        print(f"🔍 Analizando {len(maint_df)} registros encontrados en la ventana de mantenimiento\n")
        
        # 1. Estadísticas de Volumen
        total_vol = maint_df['Volume'].sum()
        avg_vol = maint_df['Volume'].mean()
        max_vol = maint_df['Volume'].max()
        
        print(f"📊 Estadísticas de Volumen en Ventana 17:01-17:59:")
        print(f"   - Volumen Total: {total_vol:,}")
        print(f"   - Volumen Promedio por Vela: {avg_vol:.2f}")
        print(f"   - Volumen Máximo en una Vela: {max_vol}")
        
        # 2. Análisis de Densidad (¿Hora completa o esporádica?)
        print(f"\n🧩 Análisis de Densidad (Población de la hora):")
        maint_df['Date'] = maint_df['Timestamp_NY'].dt.date
        candles_per_day = maint_df.groupby('Date')['Timestamp_NY'].count()
        
        avg_candles = candles_per_day.mean()
        max_candles = candles_per_day.max()
        days_full_hour = (candles_per_day >= 50).sum()
        days_sporadic = (candles_per_day < 15).sum() # Ajustado según resultados

        print(f"   - Promedio de velas por sesión afectada: {avg_candles:.1f} / 59 min")
        print(f"   - Máximo de velas en una sola tarde: {max_candles}")
        print(f"   - Sesiones con hora completa (>50 min): {days_full_hour}")
        print(f"   - Sesiones con ráfagas cortas (<15 min): {days_sporadic}")
        
        # 3. Detección de anomalías por fechas
        print(f"\n📅 Días con mayor volumen total en ventana prohibida (Top 10):")
        date_activity = maint_df.groupby('Date').agg({
            'Volume': 'sum',
            'Timestamp_NY': 'count'
        }).rename(columns={'Timestamp_NY': 'Candles'}).sort_values('Volume', ascending=False).head(10)
        print(date_activity)

        # 4. Distribución por minuto
        print(f"\n⏰ Minutos con presencia de datos (Top 10 minutos más frecuentes):")
        maint_df['Minute'] = maint_df['Timestamp_NY'].dt.minute
        min_dist = maint_df.groupby('Minute')['Timestamp_NY'].count().sort_values(ascending=False).head(10)
        print(min_dist)

        # 5. Muestra de los datos con mayor volumen
        print(f"\n📋 Muestra de las 15 velas con mayor volumen en mantenimiento:")
        cols_show = ['Timestamp_NY', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
        sample = maint_df.sort_values('Volume', ascending=False).head(15)
        print(sample[cols_show].to_string(index=False))

        # 6. Verificación de impacto en precio (Volatilidad)
        maint_df['Range'] = maint_df['High'] - maint_df['Low']
        max_range = maint_df['Range'].max()
        print(f"\n🚀 Volatilidad máxima detectada (High-Low) en mantenimiento: {max_range:.2f} puntos.")
        
        # 7. Conclusión final basada en evidencia
        print(f"\n🏁 CONCLUSIÓN DE DENSIDAD:")
        if max_candles <= 15:
            print("   ⚠️ DIAGNÓSTICO: VELAS ESPORÁDICAS / RÁFAGAS.")
            print("   La ventana nunca se puebla completa. Los datos se concentran en los")
            print("   primeros 15 minutos. Es probable que sea actividad de 'TAS' (Trade at Settlement)")
            print("   o mensajes tardíos capturados por la fuente.")
        else:
            print("   ⚠️ DIAGNÓSTICO: POSIBLE DESFASE HORARIO.")
            print(f"   Se detectaron hasta {max_candles} velas, lo cual sugiere un solapamiento mayor.")

    except Exception as e:
        print(f"❌ Ocurrió un error durante el análisis: {e}")

# Ejecución automática si df_final existe en el espacio de nombres global
if 'df_final' in globals():
    analyze_maintenance_data(df_final)
else:
    print("⚠️ El objeto 'df_final' no se encuentra en memoria. Asegúrate de ejecutar el proceso principal primero.")

🔍 Analizando 16850 registros encontrados en la ventana de mantenimiento

📊 Estadísticas de Volumen en Ventana 17:01-17:59:
   - Volumen Total: 232,839
   - Volumen Promedio por Vela: 13.82
   - Volumen Máximo en una Vela: 1411

🧩 Análisis de Densidad (Población de la hora):
   - Promedio de velas por sesión afectada: 12.8 / 59 min
   - Máximo de velas en una sola tarde: 15
   - Sesiones con hora completa (>50 min): 0
   - Sesiones con ráfagas cortas (<15 min): 1313

📅 Días con mayor volumen total en ventana prohibida (Top 10):
            Volume  Candles
Date                       
2011-06-15    2273       12
2013-07-10    2228       14
2013-04-12    1919       14
2011-08-22    1753       14
2015-03-20    1558       14
2014-09-09    1347       14
2011-07-13    1326       15
2013-04-15    1196       14
2011-08-24    1133       14
2013-03-20    1111       13

⏰ Minutos con presencia de datos (Top 10 minutos más frecuentes):
Minute
14    1313
10    1270
13    1257
12    1255
11    1255
5 

In [27]:
import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# --- CONFIGURACIÓN DE RUTAS ---
# Ajusta estas rutas según tu entorno local
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"

def process_gold_data():
    """
    Procesa datos de Databento para generar una serie continua de Oro (GC).
    Gestiona:
    1. Ambigüedad de tickers cada 10 años (ej. GCZ5 en 2015 y 2025).
    2. Selección de contrato líder por volumen progresivo.
    3. Limpieza de ventana de mantenimiento (17:01-17:59 NY).
    4. Ajuste por rollover (Back-adjustment).
    """
    print(f"🚀 Iniciando procesamiento: {datetime.now()}")

    # 1. CARGA E INGESTA
    print("📦 Cargando datos...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df_raw = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])

    # Filtrar solo contratos outrights (ignorar spreads/calendars con '-')
    df_raw = df_raw[~df_raw['symbol'].str.contains('-')]
    
    # 2. NORMALIZACIÓN TEMPORAL (UTC -> NY)
    print("🕒 Normalizando zonas horarias...")
    df_raw['ts_event'] = pd.to_datetime(df_raw['ts_event'], utc=True)
    ny_tz = pytz.timezone('America/New_York')
    # Convertimos a NY y quitamos el offset para cálculos limpios
    df_raw['Timestamp_NY'] = df_raw['ts_event'].dt.tz_convert(ny_tz).dt.tz_localize(None)
    df_raw['Year'] = df_raw['Timestamp_NY'].dt.year

    # 3. IDENTIFICADOR ÚNICO (Solución al ciclo decenal)
    # Evita que GCZ5 de 2015 se mezcle con GCZ5 de 2025
    df_raw['Unique_Symbol'] = df_raw['symbol'] + "_" + df_raw['Year'].astype(str)
    df_raw.sort_values(['Timestamp_NY', 'Unique_Symbol'], inplace=True)

    # 4. LÓGICA DE LÍDER PROGRESIVO
    print("🏆 Determinando contrato líder por volumen...")
    # Business Date: La sesión de futuros comienza a las 18:00 del día anterior
    df_raw['Business_Date'] = (df_raw['Timestamp_NY'] + pd.Timedelta(hours=6)).dt.date
    
    # Calculamos el volumen total diario por contrato único
    daily_vol = df_raw.groupby(['Business_Date', 'Unique_Symbol'])['volume'].sum().reset_index()
    # Identificamos cuál tuvo el máximo volumen cada día
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    raw_leaders = daily_vol.loc[idx_max_vol][['Business_Date', 'Unique_Symbol']]
    
    # Mantenemos una lista ordenada de contratos para evitar "saltos hacia atrás"
    unique_leaders_ordered = []
    seen = set()
    for s in raw_leaders['Unique_Symbol']:
        if s not in seen:
            unique_leaders_ordered.append(s)
            seen.add(s)
    
    leader_map = {}
    current_leader_idx = 0
    for day_data in raw_leaders.to_dict('records'):
        day, candidate = day_data['Business_Date'], day_data['Unique_Symbol']
        candidate_idx = unique_leaders_ordered.index(candidate)
        # Solo cambiamos de líder si el nuevo contrato está más adelante en el tiempo
        if candidate_idx >= current_leader_idx:
            current_leader_idx = candidate_idx
        leader_map[day] = unique_leaders_ordered[current_leader_idx]
    
    df_raw['is_leader'] = df_raw['Unique_Symbol'] == df_raw['Business_Date'].map(leader_map)
    df_continuous = df_raw[df_raw['is_leader']].copy()

    # 5. LIMPIEZA DE VENTANA DE MANTENIMIENTO (17:01 - 17:59)
    # Basado en el análisis, eliminamos estas velas ruidosas y esporádicas.
    # La vela de las 16:59 es el cierre, 17:00 puede tener ajustes, 17:01+ es mantenimiento.
    print("🧹 Limpiando ventana de mantenimiento (17:01-17:59)...")
    hour = df_continuous['Timestamp_NY'].dt.hour
    minute = df_continuous['Timestamp_NY'].dt.minute
    mask_to_drop = (hour == 17) & (minute > 0)
    
    dropped_count = mask_to_drop.sum()
    df_continuous = df_continuous[~mask_to_drop].copy()
    print(f"   - Se eliminaron {dropped_count:,} registros de mantenimiento.")

    # 6. BACK ADJUSTMENT (Ajuste de Precios)
    print("🔧 Calculando Back Adjustment...")
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)
    
    # Identificamos puntos de rollover
    df_continuous['Is_Rollover_Event'] = (df_continuous['Unique_Symbol'] != df_continuous['Unique_Symbol'].shift(1))
    df_continuous.loc[0, 'Is_Rollover_Event'] = False
    
    all_changes = df_continuous.index[df_continuous['Unique_Symbol'] != df_continuous['Unique_Symbol'].shift(-1)].tolist()
    if len(all_changes) > 0 and all_changes[-1] == len(df_continuous) - 1: all_changes.pop()

    factors = np.zeros(len(df_continuous))
    cumulative_delta = 0.0
    
    # Recorremos de presente a pasado para ajustar precios históricos al nivel actual
    for idx in reversed(all_changes):
        ts_rollover = df_continuous.iloc[idx]['Timestamp_NY']
        u_ticker_old = df_continuous.iloc[idx]['Unique_Symbol']
        u_ticker_new = df_continuous.iloc[idx+1]['Unique_Symbol']
        
        # Obtenemos precios de ambos contratos en el mismo instante de tiempo
        snapshot = df_raw[df_raw['Timestamp_NY'] == ts_rollover]
        p_old = snapshot[snapshot['Unique_Symbol'] == u_ticker_old]['close']
        p_new = snapshot[snapshot['Unique_Symbol'] == u_ticker_new]['close']
        
        if not p_old.empty and not p_new.empty:
            gap = p_new.iloc[0] - p_old.iloc[0]
        else:
            # Fallback si no hay coincidencia exacta
            gap = df_continuous.iloc[idx+1]['close'] - df_continuous.iloc[idx]['close']
            
        cumulative_delta -= gap
        factors[:idx + 1] = cumulative_delta

    df_continuous['Adj_Factor'] = factors
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = (df_continuous[col] + df_continuous['Adj_Factor']).round(2)

    # 7. FORMATEO Y GUARDADO
    print("📊 Finalizando dataset...")
    df_continuous['Ticker'] = df_continuous['Unique_Symbol'].str.split('_').str[0]
    
    df_final = df_continuous[[
        'Timestamp_NY', 'Ticker', 'open', 'high', 'low', 'close', 'volume', 
        'Adj_Factor', 'Is_Rollover_Event', 
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 
        'high': 'High', 
        'low': 'Low', 
        'close': 'Close', 
        'volume': 'Volume'
    }, inplace=True)
    
    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    
    print(f"✅ Proceso completado con éxito.")
    print(f"   - Registros finales: {len(df_final):,}")
    print(f"   - Rango: {df_final['Timestamp_NY'].min()} a {df_final['Timestamp_NY'].max()}")
    print(f"💾 Archivo guardado: {OUTPUT_PATH}")
    
    return df_final

if __name__ == "__main__":
    df_final = process_gold_data()

🚀 Iniciando procesamiento: 2026-02-17 21:01:06.295365
📦 Cargando datos...
🕒 Normalizando zonas horarias...
🏆 Determinando contrato líder por volumen...
🧹 Limpiando ventana de mantenimiento (17:01-17:59)...
   - Se eliminaron 16,844 registros de mantenimiento.
🔧 Calculando Back Adjustment...
📊 Finalizando dataset...
✅ Proceso completado con éxito.
   - Registros finales: 5,478,577
   - Rango: 2010-06-06 18:00:00 a 2026-02-04 18:59:00
💾 Archivo guardado: /home/quant/data/processed/gc_1m_raw_continuous.parquet


In [28]:
df_final.head()

,Timestamp_NY,Ticker,Open,High,Low,Close,Volume,Adj_Factor,Is_Rollover_Event,Open_adj,High_adj,Low_adj,Close_adj
0,2010-06-06 18:00:00,GCQ0,1221.1,1221.8,1221.1,1221.6,74,-760.2,False,460.9,461.6,460.9,461.4
1,2010-06-06 18:01:00,GCQ0,1221.6,1221.9,1221.6,1221.9,8,-760.2,False,461.4,461.7,461.4,461.7
2,2010-06-06 18:02:00,GCQ0,1221.9,1221.9,1221.6,1221.8,41,-760.2,False,461.7,461.7,461.4,461.6
3,2010-06-06 18:03:00,GCQ0,1221.6,1221.6,1220.5,1221.5,70,-760.2,False,461.4,461.4,460.3,461.3
4,2010-06-06 18:04:00,GCQ0,1221.0,1221.0,1220.9,1220.9,4,-760.2,False,460.8,460.8,460.7,460.7


In [29]:
df_final.tail()

,Timestamp_NY,Ticker,Open,High,Low,Close,Volume,Adj_Factor,Is_Rollover_Event,Open_adj,High_adj,Low_adj,Close_adj
5478572,2026-02-04 18:55:00,GCJ6,5038.8,5042.5,5035.7,5036.6,118,0.0,False,5038.8,5042.5,5035.7,5036.6
5478573,2026-02-04 18:56:00,GCJ6,5036.5,5036.5,5032.4,5033.9,58,0.0,False,5036.5,5036.5,5032.4,5033.9
5478574,2026-02-04 18:57:00,GCJ6,5034.8,5037.8,5034.0,5036.7,21,0.0,False,5034.8,5037.8,5034.0,5036.7
5478575,2026-02-04 18:58:00,GCJ6,5035.0,5035.0,5030.2,5031.0,43,0.0,False,5035.0,5035.0,5030.2,5031.0
5478576,2026-02-04 18:59:00,GCJ6,5031.1,5034.2,5028.6,5033.4,96,0.0,False,5031.1,5034.2,5028.6,5033.4


# ESPECIFICACIÓN TÉCNICA: PROCESAMIENTO RAW CSV A PARQUET CONTINUO (ORO - GC)

**PARA:** Equipo de Ingeniería de Datos

**DE:** Estratega Cuantitativo Senior

**ASUNTO:** Protocolo de Carga, Limpieza y Conversión Horaria para Backtesting Intradía (v2.0 - Revisada)

## 1. OBJETIVO

Transformar el archivo crudo de Databento en un DataFrame de alta fidelidad, eliminando ruidos técnicos, normalizando la línea de tiempo a la zona horaria de referencia (**America/New_York**), y construyendo una serie continua con ajuste por rollover (**Backward Adjustment**) que resuelva la ambigüedad decenal de los tickers.

## 2. RECURSOS DE ORIGEN

- **Ruta:** `/home/quant/data/raw/databento/GC/`
- **Archivo:** `glbx-mdp3-20100606-20260204.ohlcv-1m.csv`
- **Formato:** `ts_event, rtype, publisher_id, instrument_id, open, high, low, close, volume, symbol`

## 3. REQUERIMIENTOS DE FILTRADO Y CARGA

### REQ-01: Exclusión de Instrumentos Sintéticos y Spreads

- **Regla:** Eliminar cualquier fila donde el campo `symbol` contenga un guion ().
- **Propósito:** Eliminar instrumentos de arbitraje y spreads que presentan precios negativos (detectados hasta -50,874.4).

### REQ-02: Filtro de "Major Months" (Liquidez Institucional)

- **Regla:** Solo permitir contratos cuyos códigos de mes correspondan al ciclo principal de entrega del Oro.
- **Meses permitidos:** G (Feb), J (Apr), M (Jun), Q (Aug), V (Oct), Z (Dic).
- **Regex:** `^GC[GJMQUVZ][0-9]$`

### REQ-03: Resolución de Ambigüedad Decenal (Unique Ticker)

- **Hallazgo:** Tickers como `GCZ5` se repiten en 2015 y 2025, lo que rompe la continuidad.
- **Regla:** Crear un `Unique_Symbol` concatenando el `symbol` original con el año del evento (ej. `GCZ5_2015`).

## 4. LIMPIEZA Y NORMALIZACIÓN TEMPORAL

### REQ-04: Gestión de Ventana de Mantenimiento (CRÍTICO)

- **Hallazgo:** Se detectaron ráfagas esporádicas de datos (promedio 12.8 velas) entre las 17:01 y 17:59 NY que no son operables y distorsionan la volatilidad.
- **Regla:** Eliminar todos los registros donde la hora local de NY sea las 17:00 y el minuto sea > 0.
- **Nota:** La vela de las 16:59 se mantiene como el cierre oficial. La sesión se reanuda a las 18:00.

### REQ-05: Estructura de Salida (Doble Capa Horaria)

El objeto final debe ser un DataFrame con el siguiente esquema:

| Columna | Tipo | Descripción |
| --- | --- | --- |
| **Timestamp_NY** | index (datetime64) | Hora local NY (Naive) para compatibilidad con simulador. |
| **Timestamp_GMT** | datetime64 | Hora original UTC sin zona horaria. |
| **NY_Offset** | int32 | Desfase respecto a UTC (ej. -5 para EST, -4 para EDT). |
| **Open, High, Low, Close** | float64 | Precios crudos del contrato líder. |
| **Volume** | int64 | Volumen total de la vela. |
| **Ticker** | string | Símbolo base (ej. GCJ5). |
| **Contract_ID** | int64 | instrument_id original de Databento. |
| **Is_Rollover_Event** | bool | Flag que indica si la vela es el inicio de un nuevo contrato líder. |
| **Market_Gap** | float64 | Salto de precio real del mercado entre el cierre y la apertura. |
| **Adj_Factor** | float64 | Factor acumulado de ajuste basado únicamente en el diferencial técnico de rollover. |
| **Open_adj, ..., Close_adj** | float64 | Precios ajustados (Backward Adjustment). |

## 5. PROTOCOLO DE CONTINUIDAD (ROLLOVER)

### REQ-06: Lógica de Líder Progresivo

- **Criterio:** El contrato líder es el que presenta el mayor volumen diario (`Business_Date` iniciando a las 18:00).
- **Restricción:** No se permiten saltos hacia atrás en el tiempo (una vez que un contrato es líder, no puede volver a serlo un contrato con fecha de vencimiento anterior).

### REQ-07: Backward Adjustment (Neutralización del Rollover)

- **Fórmula:** `delta = Close_new_contract - Close_old_contract` capturado en el mismo timestamp de rollover.
- **Aplicación:** El delta se acumula y se aplica a todos los precios históricos previos al punto de rollover.
- **Propósito:** Eliminar el salto artificial por cambio de contrato sin tocar la variación orgánica del precio del activo.

### REQ-08: Trazabilidad del Salto (Market_Gap)

- **Regla:** El `Market_Gap` representa la diferencia entre el precio de apertura de la nueva sesión y el cierre de la sesión anterior. **Este valor se preserva íntegro** tanto en los precios crudos como en los ajustados.
- **Propósito:** Validar que el ajuste no está "suavizando" movimientos de precio que ocurrieron realmente durante el cierre del mercado.

## 6. VALIDACIÓN FINAL (KPIs de Calidad)

1. **Precios Positivos:** `Close.min() > 0` (DEBE CUMPLIRSE tras REQ-01).
2. **Unicidad Temporal:** `index.is_unique == True`.
3. **Continuidad Técnica:** `abs(diff(Close_adj))` en puntos de rollover (excluyendo Market_Gap) debe ser `< 2 ticks`.
4. **Densidad:** Confirmar la ausencia de datos en la franja 17:01 - 17:59 NY.

## 7. ENTREGABLE

- **Formato:** Parquet con compresión **Snappy**.
- **Ruta final:** `/home/quant/data/processed/gc_1m_raw_continuous.parquet`

In [33]:
import pandas as pd
import numpy as np
import pytz
import re
from datetime import datetime

# --- CONFIGURACIÓN DE RUTAS ---
INPUT_PATH = "/home/quant/data/raw/databento/GC/glbx-mdp3-20100606-20260204.ohlcv-1m.csv"
OUTPUT_PATH = "/home/quant/data/processed/gc_1m_raw_continuous.parquet"

def process_gold_data():
    """
    Ejecución maestra siguiendo la especificación v2.0 revisada.
    Incluye: Filtro Major Months, Resolución Decenal, Limpieza de Mantenimiento,
    Cálculo de Market Gap y KPIs de Validación.
    """
    print(f"🚀 Iniciando procesamiento maestro: {datetime.now()}")

    # 1. CARGA E INGESTA (REQ-01: Exclusión de Spreads)
    print("📦 Cargando y filtrando instrumentos (REQ-01)...")
    cols = ['ts_event', 'instrument_id', 'open', 'high', 'low', 'close', 'volume', 'symbol']
    df_raw = pd.read_csv(INPUT_PATH, usecols=cols, parse_dates=['ts_event'])
    
    # REQ-01: Eliminar símbolos con guion (Spreads)
    df_raw = df_raw[~df_raw['symbol'].str.contains('-')]

    # 2. FILTRO MAJOR MONTHS (REQ-02)
    print("💎 Filtrando Major Months (G, J, M, Q, V, Z) (REQ-02)...")
    major_months_regex = r'^GC[GJMQUVZ][0-9]$'
    df_raw = df_raw[df_raw['symbol'].str.match(major_months_regex)]

    # 3. NORMALIZACIÓN TEMPORAL (REQ-04 & REQ-05)
    print("🕒 Normalizando zonas horarias y offsets (REQ-05)...")
    df_raw['Timestamp_GMT'] = pd.to_datetime(df_raw['ts_event'], utc=True).dt.tz_localize(None)
    ny_tz = pytz.timezone('America/New_York')
    
    # Calculamos Timestamp_NY y el Offset
    def get_ny_info(utc_dt):
        localized = utc_dt.replace(tzinfo=pytz.UTC).astimezone(ny_tz)
        return localized.replace(tzinfo=None), int(localized.utcoffset().total_seconds() / 3600)

    # Aplicación eficiente de zona horaria
    ny_data = df_raw['ts_event'].apply(lambda x: get_ny_info(x))
    df_raw['Timestamp_NY'], df_raw['NY_Offset'] = zip(*ny_data)
    df_raw['Year'] = df_raw['Timestamp_NY'].dt.year

    # 4. RESOLUCIÓN AMBIGÜEDAD DECENAL (REQ-03)
    print("🆔 Resolviendo ambigüedad decenal (REQ-03)...")
    df_raw['Unique_Symbol'] = df_raw['symbol'] + "_" + df_raw['Year'].astype(str)
    df_raw.sort_values(['Timestamp_NY', 'Unique_Symbol'], inplace=True)

    # 5. LÓGICA DE LÍDER PROGRESIVO (REQ-06)
    print("🏆 Determinando contrato líder progresivo...")
    df_raw['Business_Date'] = (df_raw['Timestamp_NY'] + pd.Timedelta(hours=6)).dt.date
    daily_vol = df_raw.groupby(['Business_Date', 'Unique_Symbol'])['volume'].sum().reset_index()
    idx_max_vol = daily_vol.groupby('Business_Date')['volume'].idxmax()
    raw_leaders = daily_vol.loc[idx_max_vol][['Business_Date', 'Unique_Symbol']]
    
    unique_leaders_ordered = []
    seen = set()
    for s in raw_leaders['Unique_Symbol']:
        if s not in seen:
            unique_leaders_ordered.append(s)
            seen.add(s)
    
    leader_map = {}
    curr_idx = 0
    for day, candidate in zip(raw_leaders['Business_Date'], raw_leaders['Unique_Symbol']):
        cand_idx = unique_leaders_ordered.index(candidate)
        if cand_idx >= curr_idx: curr_idx = cand_idx
        leader_map[day] = unique_leaders_ordered[curr_idx]
    
    df_raw['is_leader'] = df_raw['Unique_Symbol'] == df_raw['Business_Date'].map(leader_map)
    df_continuous = df_raw[df_raw['is_leader']].copy()

    # 6. LIMPIEZA VENTANA MANTENIMIENTO (REQ-04)
    print("🧹 Ejecutando limpieza de mantenimiento (17:01-17:59 NY) (REQ-04)...")
    maint_mask = (df_continuous['Timestamp_NY'].dt.hour == 17) & (df_continuous['Timestamp_NY'].dt.minute > 0)
    df_continuous = df_continuous[~maint_mask].copy()

    # 7. BACK ADJUSTMENT Y MARKET GAP (REQ-07 & REQ-08)
    print("🔧 Calculando Ajustes y Market Gaps (REQ-08)...")
    df_continuous.sort_values('Timestamp_NY', inplace=True)
    df_continuous.reset_index(drop=True, inplace=True)
    
    df_continuous['Is_Rollover_Event'] = (df_continuous['Unique_Symbol'] != df_continuous['Unique_Symbol'].shift(1))
    df_continuous.loc[0, 'Is_Rollover_Event'] = False
    
    # Identificar índices de cambio de contrato
    roll_indices = df_continuous.index[df_continuous['Is_Rollover_Event']].tolist()
    
    factors = np.zeros(len(df_continuous))
    market_gaps = np.zeros(len(df_continuous))
    cumulative_delta = 0.0
    
    for idx in reversed(roll_indices):
        # Datos del contrato nuevo (apertura)
        p_new_open = df_continuous.loc[idx, 'open']
        u_ticker_new = df_continuous.loc[idx, 'Unique_Symbol']
        ts_event = df_continuous.loc[idx, 'Timestamp_NY']
        
        # Datos del contrato viejo (cierre previo)
        p_old_close = df_continuous.loc[idx-1, 'close']
        u_ticker_old = df_continuous.loc[idx-1, 'Unique_Symbol']
        
        # 1. Market Gap (Salto real de mercado entre sesiones)
        market_gaps[idx] = p_new_open - p_old_close
        
        # 2. Rollover Gap (Diferencia técnica New vs Old en el mismo instante)
        # Buscamos en df_raw el precio del contrato viejo en el timestamp actual
        old_at_new_ts = df_raw[(df_raw['Timestamp_NY'] == ts_event) & (df_raw['Unique_Symbol'] == u_ticker_old)]
        
        if not old_at_new_ts.empty:
            p_old_at_ts = old_at_new_ts.iloc[0]['close']
            roll_delta = df_continuous.loc[idx, 'close'] - p_old_at_ts
        else:
            # Fallback si no hay vela simultánea
            roll_delta = (df_continuous.loc[idx, 'close'] - df_continuous.loc[idx, 'open']) - (df_continuous.loc[idx-1, 'close'] - df_continuous.loc[idx-1, 'open'])
            roll_delta = 0 # En GC líquido suele haber datos

        cumulative_delta -= roll_delta
        factors[:idx] = cumulative_delta

    df_continuous['Adj_Factor'] = factors
    df_continuous['Market_Gap'] = market_gaps
    
    for col in ['open', 'high', 'low', 'close']:
        df_continuous[f'{col.capitalize()}_adj'] = (df_continuous[col.lower()] + df_continuous['Adj_Factor']).round(2)

    # 8. VALIDACIÓN FINAL (KPIs DE CALIDAD)
    print("\n✅ INICIANDO VALIDACIÓN DE KPIs:")
    
    # KPI 1: Precios Positivos
    kpi_pos = df_continuous['close'].min() > 0
    print(f"   1. Precios Positivos: {'PASSED' if kpi_pos else 'FAILED'} (Min: {df_continuous['close'].min()})")
    
    # KPI 2: Unicidad
    kpi_unique = df_continuous['Timestamp_NY'].is_unique
    print(f"   2. Unicidad Temporal: {'PASSED' if kpi_unique else 'FAILED'}")
    
    # KPI 3: Continuidad Técnica
    # Revisamos el salto en Close_adj quitando el Market_Gap
    check_idx = roll_indices[0] if roll_indices else None
    if check_idx:
        jump = abs((df_continuous.loc[check_idx, 'Close_adj'] - df_continuous.loc[check_idx-1, 'Close_adj']) - df_continuous.loc[check_idx, 'Market_Gap'])
        kpi_cont = jump < 0.20 # 2 ticks de GC son 0.20
        print(f"   3. Continuidad Técnica: {'PASSED' if kpi_cont else 'FAILED'} (Diff residual: {jump:.4f})")
    
    # KPI 4: Densidad Mantenimiento
    maint_check = ((df_continuous['Timestamp_NY'].dt.hour == 17) & (df_continuous['Timestamp_NY'].dt.minute > 0)).sum()
    print(f"   4. Densidad Mantenimiento: {'PASSED' if maint_check == 0 else 'FAILED'} ({maint_check} velas filtradas)")

    # 9. GUARDADO
    df_final = df_continuous[[
        'Timestamp_NY', 'Timestamp_GMT', 'NY_Offset', 'open', 'high', 'low', 'close', 'volume',
        'symbol', 'instrument_id', 'Is_Rollover_Event', 'Market_Gap', 'Adj_Factor',
        'Open_adj', 'High_adj', 'Low_adj', 'Close_adj'
    ]].copy()
    
    df_final.rename(columns={
        'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 
        'volume': 'Volume', 'symbol': 'Ticker', 'instrument_id': 'Contract_ID'
    }, inplace=True)
    
    df_final.to_parquet(OUTPUT_PATH, compression='snappy')
    print(f"\n💾 Dataset final guardado en: {OUTPUT_PATH}")
    return df_final

if __name__ == "__main__":
    try:
        df_processed = process_gold_data()
    except Exception as e:
        print(f"❌ ERROR CRÍTICO: {e}")

🚀 Iniciando procesamiento maestro: 2026-02-17 21:18:27.220924
📦 Cargando y filtrando instrumentos (REQ-01)...
💎 Filtrando Major Months (G, J, M, Q, V, Z) (REQ-02)...
🕒 Normalizando zonas horarias y offsets (REQ-05)...
🆔 Resolviendo ambigüedad decenal (REQ-03)...
🏆 Determinando contrato líder progresivo...
🧹 Ejecutando limpieza de mantenimiento (17:01-17:59 NY) (REQ-04)...
🔧 Calculando Ajustes y Market Gaps (REQ-08)...

✅ INICIANDO VALIDACIÓN DE KPIs:
   1. Precios Positivos: PASSED (Min: 1045.6)
   2. Unicidad Temporal: PASSED
   3. Continuidad Técnica: FAILED (Diff residual: 2.3000)
   4. Densidad Mantenimiento: PASSED (0 velas filtradas)

💾 Dataset final guardado en: /home/quant/data/processed/gc_1m_raw_continuous.parquet
